# Shopify Pipeline v9

**v9 changes:** Fixed CSS framework · EPROLO variants · Trust signals · Font-size guard · Mobile validation · Smart interlinking

## Cell 1 — Config + Upload

In [1]:
# ╔══════════════════════════════════════════════════════╗
# ║         CONFIG                                       ║
# ╚══════════════════════════════════════════════════════╝

import os

ANTHROPIC_API_KEY = os.environ["ANTHROPIC_API_KEY"]
SHOPIFY_STORE = os.environ["SHOPIFY_STORE"]
SHOPIFY_CLIENT_ID = os.environ["SHOPIFY_CLIENT_ID"]
SHOPIFY_CLIENT_SECRET = os.environ["SHOPIFY_CLIENT_SECRET"]
DATAFORSEO_LOGIN = os.environ["DATAFORSEO_LOGIN"]
DATAFORSEO_PASSWORD = os.environ["DATAFORSEO_PASSWORD"]

LOCATION_CODE = 2840
LANGUAGE_CODE = 'en'
MAX_KD = 50  # WANELO.com = strong dropped domain, can target higher KD
MIN_VOLUME = 50
TOTAL_BUDGET = 15.00
AUTO_CONFIRM_UNDER = 20.0
STAGE_EXPAND_TOP = 30
TOP_K_PER_PRODUCT = 25
RECALL_K_BASE = 100
RERANKER_THRESHOLD = -3

LANGUAGE = "English"
MODEL_VISION = "claude-haiku-4-5-20251001"
MODEL_WRITER = "claude-sonnet-4-5-20250929"
MODEL_DESIGNER = "claude-opus-4-7"   # picks layout + palette + writes slot text (modular HTML mode)

# === Modular HTML config (Lumea-style modules from page_builder.py) ===
# True  → Opus designer assembles page from MODULE_CATALOG via page_builder.assemble_page().
# False → legacy: Sonnet writer generates HTML с wa-* классами внутри одной ячейки.
USE_MODULAR_HTML = True
PAGE_BUILDER_URL = "https://raw.githubusercontent.com/invme2/tags-recomendations/main/pipeline/page_builder.py"


# === HTML clean-output rules — добавляются ко всем prompt'ам, генерящим HTML/текст ===
# Просим модель тихо вычистить невидимые символы (zero-width, BOM, directional, PUA)
# и не вставлять AI-водяные знаки / скрытый SEO-стаффинг.
HTML_CLEAN_OUTPUT_RULES = """

CLEAN-OUTPUT RULES (apply silently before delivering):

Silently remove these invisible/control characters if any were generated:
- U+200B Zero Width Space
- U+200C Zero Width Non-Joiner
- U+200D Zero Width Joiner
- U+FEFF BOM / Zero Width No-Break Space
- U+2060 Word Joiner
- U+00AD Soft Hyphen
- U+034F Combining Grapheme Joiner
- U+061C Arabic Letter Mark
- U+180E Mongolian Vowel Separator
- U+200E Left-To-Right Mark
- U+200F Right-To-Left Mark
- U+202A–U+202E directional embedding / override marks
- U+2066–U+2069 isolate marks
- U+E000–U+F8FF Private Use Area characters
- Variation selectors used as hidden marks
- Any other invisible or suspicious control characters

NEVER include in output:
- AI notes about the generation process
- Prompt fragments from the user request
- Debug comments
- Hidden HTML comments containing generation context
- Generation metadata
- Attributes like data-ai, data-generated, data-watermark, data-model
- Hidden SEO text or invisible keyword stuffing
- Off-screen keyword blocks (position:absolute; left:-9999px; and similar)
- Transparent text (color matching background)
- Zero-size text (font-size:0)
- Mixed-script homoglyphs in English words (Cyrillic/Greek lookalikes inside Latin text)
"""

# ═══ v9.2 NEW CONSTANTS ═══
STORE_VENDOR = "wanelo"
STORE_DOMAIN = "wanelo.com"  # Dropped domain with DA 60+
DEFAULT_INVENTORY = 999
MARGIN_ALERT_MIN = 5.00  # Flag products where sell price < $5
PUBLISH_STATUS = "ACTIVE"  # Full automation — products go live immediately

# ═══ TAXONOMY CONFIG (v3.2) ═══
# Set this to your raw taxonomy.json URL (e.g. GitHub raw)
TAXONOMY_URL = "https://raw.githubusercontent.com/YOUR-ORG/taxonomy/main/taxonomy.json"
TAXONOMY_REFRESH = False     # True = always re-download; False = use cache after first fetch
TAXONOMY_APPROVED_ONLY = False  # True = use only V3.1 status=approved; False = include V3.2 drafts
TAGS_PER_PRODUCT_MIN = 4    # min total tags per product (cluster + persona + intent + demo)
TAGS_PER_PRODUCT_MAX = 9
TAXONOMY_SHORTLIST_K = 40   # how many candidate clusters to include in Stage 2 prompt


SHIPPING_RETURNS_HTML = """
<div class="wa-section wa-shipping-block">
  <div class="wa-trust" style="border-top:1px solid #e5e7eb;padding-top:1.5rem">
    <div class="wa-trust-badge"><span class="icon">🚚</span> Free Shipping</div>
    <div class="wa-trust-badge"><span class="icon">📦</span> Ships in 1-3 Business Days</div>
    <div class="wa-trust-badge"><span class="icon">🔄</span> 30-Day Easy Returns</div>
    <div class="wa-trust-badge"><span class="icon">🛡️</span> Buyer Protection</div>
    <div class="wa-trust-badge"><span class="icon">💳</span> Secure Checkout</div>
  </div>
  <details class="wa-faq" style="margin-top:0.5rem">
    <summary>Shipping & Returns Policy</summary>
    <div class="wa-faq-body">
      <p><strong>Shipping:</strong> We offer free standard shipping on all orders. Orders are processed within 1-2 business days and typically arrive within 7-15 business days depending on your location.</p>
      <p><strong>Returns:</strong> Not satisfied? Return any unused item within 30 days of delivery for a full refund. Simply contact our support team to initiate your return.</p>
      <p><strong>Buyer Protection:</strong> Every purchase is covered by our buyer protection guarantee. If your item arrives damaged or not as described, we will replace it or refund you in full.</p>
    </div>
  </details>
  <div id="judgeme_product_reviews" class="wa-section" style="margin-top:1.5rem">
    <!-- Judge.me widget renders here automatically -->
  </div>
</div>
"""


BRAND_BLACKLIST = [
    'cerave','crest','neutrogena','maybelline','nyx','loreal','revlon',
    'clinique','olay','dove','nivea','garnier','covergirl','sephora',
    'ulta','mac ','benefit','tarte','fenty','rare beauty','elf ',
    'colgate','sensodyne','oral-b','philips','braun','waterpik',
    'listerine','colourpop','morphe','anastasia','urban decay',
    'too faced','nars','bobbi brown','estee lauder','lancome',
    'victoria secret','bath body works','the ordinary','cetaphil',
    'marvis','la roche','bioderma','vichy','avene','kiehl','persmax','cutex',
    'aveeno','eucerin','pantene','amika','lonris','mario badescu',
    'makeup by mario','cardi b','pete davidson','dr.','dr ',"l'oreal",
    'sally hansen','opi ','essie ','zoya ','gelish','orly ',
    'bath and body works','bath & body works','bath and body',
    'dyson','airwrap','air wrap',
    'victoria secret','calvin klein','ralph lauren',
]
IRRELEVANT_PATTERNS = [
    'near me','close to me','around me','in my area',
    'salon','clinic','spa near','parlor','parlour','appointment','booking',
    'dermatologist','surgeon','doctor','botox','filler injection',
    'hair transplant','laser treatment','laser removal','laser hair',
    'tattoo removal','tattoo artist','tattoo shop',
    'nail salon','nail tech near','manicure near','pedicure near',
    'costume makeup','halloween costume','cosplay makeup',
    'amazon','walmart','target ',
    'baby shower','baby clothes','baby registry','baby names',
    'shower curtain','shower head','shower door','shower rod',
    'bath bomb recipe','bath bomb mold diy',
    'body works sale','body works coupon',
    'hair dryer','blow dryer','flat iron','curling iron',
]

# ════ INSTALL ════
!pip install anthropic sentence-transformers requests openpyxl pandas tqdm httpx playwright aiohttp -q
!pip install faiss-gpu -q 2>/dev/null || pip install faiss-cpu -q
!playwright install chromium 2>/dev/null
!playwright install-deps 2>/dev/null

# Fix async for Colab — MUST run before any await
print('async: patched')

# ════ DRIVE + DB ════
from google.colab import drive, files as colab_files
drive.mount('/content/drive')

import os, sys, sqlite3, json, time, re, base64, shutil
os.environ['PYTHONIOENCODING'] = 'utf-8'
try: sys.stdout.reconfigure(encoding='utf-8')
except: pass

PROJECT_DIR = '/content/drive/MyDrive/shopify_pipeline'
OUTPUT_DIR = f'{PROJECT_DIR}/output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

DB_PATH = f'{PROJECT_DIR}/pipeline.db'
# Local DB (fast) + Drive backup (survives restart)
DB_LOCAL = '/content/pipeline.db'
if os.path.exists(DB_PATH) and not os.path.exists(DB_LOCAL):
    shutil.copy(DB_PATH, DB_LOCAL)
    print(f'Restored DB from Drive')
db = sqlite3.connect(DB_LOCAL)
db.execute("PRAGMA journal_mode=WAL")
db.row_factory = sqlite3.Row

db.executescript("""
CREATE TABLE IF NOT EXISTS products (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL, eprolo_url TEXT NOT NULL,
    status TEXT DEFAULT 'pending', error_msg TEXT,
    scrape_json TEXT, image_urls_json TEXT,
    stage1_json TEXT, stage2_json TEXT, final_html TEXT,
    seo_title TEXT, seo_description TEXT, url_handle TEXT,
    shopify_product_id TEXT, product_tags TEXT,
    variants_json TEXT, cost_usd REAL DEFAULT 0, updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
CREATE TABLE IF NOT EXISTS collections (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    original_name TEXT, seo_title TEXT NOT NULL,
    seo_handle TEXT, full_url TEXT,
    meta_title TEXT, meta_description TEXT,
    shopify_collection_id TEXT
);
CREATE TABLE IF NOT EXISTS product_collections (
    product_id INTEGER, collection_id INTEGER, UNIQUE(product_id, collection_id)
);
CREATE TABLE IF NOT EXISTS seo_state (key TEXT PRIMARY KEY, value TEXT);
CREATE TABLE IF NOT EXISTS seo_keywords (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    keyword TEXT NOT NULL,
    collection_name TEXT,
    volume INTEGER DEFAULT 0,
    kd INTEGER DEFAULT 0,
    cpc REAL DEFAULT 0,
    competition TEXT,
    UNIQUE(keyword, collection_name)
);
""")
for col in ['shopify_product_id','seo_title','seo_description','url_handle','product_tags','variants_json','seo_keywords']:
    try: db.execute(f'ALTER TABLE products ADD COLUMN {col} TEXT')
    except: pass
try: db.execute('ALTER TABLE collections ADD COLUMN shopify_collection_id TEXT')
except: pass
try: db.execute('ALTER TABLE collections ADD COLUMN total_volume INTEGER DEFAULT 0')
except: pass
try: db.execute('ALTER TABLE collections ADD COLUMN top_keywords TEXT DEFAULT ""')
except: pass
db.commit()

# ════ CHECK STATE OR UPLOAD ════
n_prods = db.execute('SELECT COUNT(*) FROM products').fetchone()[0]
n_colls = db.execute('SELECT COUNT(*) FROM collections').fetchone()[0]
n_done = db.execute("SELECT COUNT(*) FROM products WHERE status='done'").fetchone()[0]
SEO_SKIP = False
DESIGN_TEMPLATE = ""
CATEGORY = "Unknown"
PRODUCTS = []

if n_prods > 0:
    print(f'DB: {n_prods} products ({n_done} done), {n_colls} collections')
    print('  → Upload new CSV to start fresh (auto-wipes old data)')
    print('  → Or press Enter to CONTINUE with existing products')
    resp = input('Upload new files? (y=reset / Enter=continue): ').strip().lower()
    if resp == 'y':
        db.execute("DELETE FROM products"); db.execute("DELETE FROM collections")
        db.execute("DELETE FROM product_collections"); db.execute("DELETE FROM seo_state")
        try: db.execute("DELETE FROM seo_keywords")
        except: pass
        db.commit(); n_prods = 0; n_colls = 0
        print('✓ Database wiped clean!')
    else:
        PRODUCTS = [r[0] for r in db.execute('SELECT name FROM products ORDER BY id').fetchall()]
        row = db.execute("SELECT value FROM seo_state WHERE key='category'").fetchone()
        CATEGORY = row[0] if row else 'Unknown'
        tmpl = db.execute("SELECT value FROM seo_state WHERE key='design_template'").fetchone()
        DESIGN_TEMPLATE = tmpl[0] if tmpl else ""
        SEO_SKIP = n_colls > 0
        print(f'✓ Continuing with {n_prods} products ({n_done} done)')

if n_prods == 0:
    print('\nUpload all files at once (Ctrl+click to select multiple):')
    print('  - CSV with products (required)')
    print('  - SEO Excel .xlsx (optional)')
    print('  - HTML template .html (optional)')
    uploaded = colab_files.upload()
    fnames = list(uploaded.keys())
    print(f'Files: {fnames}')

    # Filter out non-data files
    fnames = [f for f in fnames if not f.endswith('.ipynb')]
    csv_f = next((f for f in fnames if f.endswith('.csv') or f.endswith('.txt')), None)
    seo_f = next((f for f in fnames if f.endswith('.xlsx')), None)
    html_f = next((f for f in fnames if f.endswith('.html') or f.endswith('.htm')), None)

    if html_f:
        with open(html_f, 'r', encoding='utf-8') as fh: DESIGN_TEMPLATE = fh.read()
        db.execute("INSERT OR REPLACE INTO seo_state (key,value) VALUES ('design_template',?)", (DESIGN_TEMPLATE,))
        print(f'Template: {html_f} ({len(DESIGN_TEMPLATE)} chars)')

    if csv_f:
        with open(csv_f, 'r', encoding='utf-8-sig') as fh:
            lines = [l.strip() for l in fh if l.strip()]
        # Auto-detect category from CSV filename
        CATEGORY = csv_f.rsplit('.', 1)[0].strip()
        for suffix in ['_products', '_items', ' products', ' items', '(1)', '(2)', '(3)']:
            CATEGORY = CATEGORY.replace(suffix, '').strip()
        print(f'Category (from filename): {CATEGORY}')
        db.execute("INSERT OR REPLACE INTO seo_state (key,value) VALUES ('category',?)", (CATEGORY,))
        for line in lines:
            parts = line.split(';', 1)
            name = parts[0].strip()
            url = parts[1].strip() if len(parts) > 1 else ''
            try: db.execute('INSERT INTO products (name,eprolo_url) VALUES (?,?)', (name, url)); PRODUCTS.append(name)
            except: pass
        db.commit()
        print(f'{len(PRODUCTS)} products')
    else:
        print('ERROR: No CSV file found!')

    if seo_f:
        import openpyxl
        wb = openpyxl.load_workbook(seo_f, read_only=True)

        # ── Sheet: Shopify Collections (+ volume, top keywords) ──
        cc = 0
        for row in wb['Shopify Collections'].iter_rows(min_row=2, values_only=True):
            st = (row[0] or '')   # SEO Title (H1)
            h = (row[1] or '')    # URL Handle
            mt = (row[2] or '')   # Meta Title
            md_val = (row[3] or '')  # Meta Description
            orig = (row[4] or '')    # Original name
            total_vol = int(row[7] or 0) if len(row) > 7 else 0    # Total Volume
            top_kw = (row[8] or '') if len(row) > 8 else ''         # Top Keywords
            try:
                db.execute("""INSERT INTO collections
                    (original_name,seo_title,seo_handle,full_url,meta_title,meta_description,total_volume,top_keywords)
                    VALUES (?,?,?,?,?,?,?,?)""",
                    (orig, st, h, f'/collections/{h}' if h else '', mt, md_val, total_vol, top_kw))
                cc += 1
            except: pass
        print(f'  Collections: {cc} (with volume + top keywords)')

        # ── Sheet: Product Keywords (+ SEO keywords per product) ──
        lc = 0; kw_saved = 0
        for row in wb['Product Keywords'].iter_rows(min_row=2, values_only=True):
            pn = (row[0] or '')   # Product name
            ct = (row[1] or '')   # Primary Collection
            kw_text = (row[3] or '') if len(row) > 3 else ''  # Keywords (pipe-separated)
            if not pn or not ct: continue
            # Fuzzy match: first 40 chars of product name (CSV name may differ slightly from Excel)
            pr = db.execute('SELECT id FROM products WHERE name=?', (pn,)).fetchone()
            if not pr:
                pr = db.execute('SELECT id FROM products WHERE name LIKE ?', (pn[:40] + '%',)).fetchone()
            if not pr:
                # Last resort: match by any word overlap
                words = [w for w in pn.split()[:3] if len(w) > 3]
                for w in words:
                    pr = db.execute('SELECT id FROM products WHERE name LIKE ?', ('%' + w + '%',)).fetchone()
                    if pr: break
            cr = db.execute('SELECT id FROM collections WHERE seo_title=?', (ct,)).fetchone()
            if not cr:
                cr = db.execute('SELECT id FROM collections WHERE seo_title LIKE ?', ('%' + ct[:30] + '%',)).fetchone()
            if pr and cr:
                pass  # Will link below
            elif not pr:
                if lc == 0: print(f'    ⚠ Product not found in DB: {pn[:50]}')
            elif not cr:
                if lc == 0: print(f'    ⚠ Collection not found: {ct[:50]}')
            if pr and cr:
                try:
                    db.execute('INSERT OR IGNORE INTO product_collections VALUES (?,?)', (pr['id'], cr['id']))
                    db.execute('UPDATE products SET primary_collection=? WHERE id=?', (ct, pr['id']))
                    lc += 1
                except: pass
            if pr and kw_text:
                db.execute('UPDATE products SET seo_keywords=? WHERE id=?', (kw_text, pr['id']))
                kw_saved += 1
        print(f'  Product links: {lc} | Keywords saved: {kw_saved}')

        # ── Sheet: Winners KD<30 (all filtered keywords with volume/CPC) ──
        wc = 0
        if 'Winners KD<30' in wb.sheetnames:
            for row in wb['Winners KD<30'].iter_rows(min_row=2, values_only=True):
                kw = (row[0] or '').strip()
                coll = (row[1] or '').strip()
                vol = int(row[2] or 0) if row[2] else 0
                kd = int(row[3] or 0) if row[3] else 0
                cpc = float(row[4] or 0) if row[4] else 0
                comp = str(row[5] or '') if len(row) > 5 else ''
                if kw:
                    try:
                        db.execute('INSERT OR REPLACE INTO seo_keywords (keyword,collection_name,volume,kd,cpc,competition) VALUES (?,?,?,?,?,?)',
                            (kw, coll, vol, kd, cpc, comp))
                        wc += 1
                    except: pass
            print(f'  Winner keywords: {wc} (KD<30, with volume + CPC)')

        db.commit(); wb.close()
        db.execute("INSERT OR REPLACE INTO seo_state (key,value) VALUES ('seo_complete','1')")
        db.commit()
        SEO_SKIP = True

        # Summary
        total_vol = db.execute('SELECT SUM(total_volume) FROM collections').fetchone()[0] or 0
        avg_kw = db.execute('SELECT AVG(volume) FROM seo_keywords').fetchone()[0] or 0
        print(f'  Total search volume: {total_vol:,} | Avg keyword volume: {avg_kw:,.0f}')
    else:
        SEO_SKIP = False

db.commit()
safe_cat = CATEGORY.replace(' & ','_').replace(' ','_')
print(f'\n{"="*50}')
print(f'  Category:    {CATEGORY}')
print(f'  Products:    {db.execute("SELECT COUNT(*) FROM products").fetchone()[0]}')
print(f'  Collections: {db.execute("SELECT COUNT(*) FROM collections").fetchone()[0]}')
print(f'  SEO_SKIP:    {SEO_SKIP}')
print(f'{"="*50}')


# Backup DB to Drive
shutil.copy(DB_LOCAL, DB_PATH)
print(f'DB backed up to Drive')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 478.8/478.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 72.4 MB/s eta 0:00:00
Chrome for Testing 145.0.7632.6 (playwright chromium v1208) downloaded to /root/.cache/ms-playwright/chromium-1208
FFmpeg (playwright ffmpeg v1011) downloaded to /root/.cache/ms-playwright/ffmpeg-1011
Chrome Headless Shell 145.0.7632.6 (playwright chromium-headless-shell v1208) downloaded to /root/.cache/ms-playwright/chromium_headless_shell-1208
Installing dependencies...
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-upd

Saving Bath & Shower.csv to Bath & Shower.csv
Saving Bath_Shower_SEO_Keywords (1).xlsx to Bath_Shower_SEO_Keywords (1).xlsx
Files: ['Bath & Shower.csv', 'Bath_Shower_SEO_Keywords (1).xlsx']
Category (from filename): Bath & Shower
26 products
  Collections: 20 (with volume + top keywords)
  Product links: 0 | Keywords saved: 26
  Total search volume: 2,312,500 | Avg keyword volume: 0

  Category:    Bath & Shower
  Products:    26
  Collections: 20
  SEO_SKIP:    True
DB backed up to Drive


## Cell 2 — Helpers

In [2]:
import requests, base64, time, json, re
import asyncio


import pandas as pd
import numpy as np
from collections import Counter
from tqdm.auto import tqdm

class CostTracker:
    PRICES={'search_volume':0.075,'keywords_for_keywords':0.075,'keyword_suggestions':0.075,'related_keywords':0.075,'keyword_ideas':0.075,'bulk_keyword_difficulty':0.075}
    def __init__(s,b): s.budget=b; s.spent=0.0; s.log=[]
    def charge(s,ep,d=''): c=s.PRICES.get(ep,0.075); s.spent+=c; s.log.append((ep,c,d))
    def can_afford(s,ep): return (s.spent+s.PRICES.get(ep,0.075))<=s.budget
    def remaining(s): return s.budget-s.spent
    def summary(s):
        by={};
        for ep,c,_ in s.log: by[ep]=by.get(ep,0)+c
        print('--- Cost ---')
        for ep,t in sorted(by.items()): print('  '+ep+': $'+str(round(t,3)))
        print('  TOTAL: $'+str(round(s.spent,3))+' | LEFT: $'+str(round(s.remaining(),3)))

cost_tracker = CostTracker(TOTAL_BUDGET)
auth=base64.b64encode((DATAFORSEO_LOGIN+':'+DATAFORSEO_PASSWORD).encode()).decode()
API_HEADERS={'Authorization':'Basic '+auth,'Content-Type':'application/json'}
r=requests.get('https://api.dataforseo.com/v3/appendix/user_data',headers=API_HEADERS,timeout=10)
bal=r.json()['tasks'][0]['result'][0].get('money',{}).get('balance',0)
print('DataForSEO balance: $'+str(round(bal,2)))

def is_brand(kw):
    k=kw.lower()
    for b in BRAND_BLACKLIST:
        if b in k: return True
    return False

def is_irrelevant(kw):
    """v6.1: Filter local/service/medical/irrelevant queries"""
    k = kw.lower()
    for pat in IRRELEVANT_PATTERNS:
        if pat in k: return True
    return False

def auto_confirm(est_cost, desc):
    print(desc+' | Est: $'+str(round(est_cost,2))+' | Budget left: $'+str(round(cost_tracker.remaining(),2)))
    if est_cost <= AUTO_CONFIRM_UNDER:
        print('>>> Auto-confirmed'); return True
    return input('Continue? (y/n): ').lower()=='y'
print('Helpers OK')

import anthropic, sys, time

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
MODEL = "claude-opus-4-6"

print(f'Testing Claude {MODEL}...', end=' ', flush=True)
try:
    resp = client.messages.create(
        model=MODEL, max_tokens=20, temperature=0,
        messages=[{"role": "user", "content": "Say OK"}])
    print(f'✓ OK! Response: "{resp.content[0].text.strip()[:30]}"')
except Exception as e:
    print(f'✗ Error: {str(e)[:100]}')
    print('Falling back to claude-sonnet-4-5-20250929...')
    MODEL = "claude-sonnet-4-5-20250929"
    try:
        resp = client.messages.create(
            model=MODEL, max_tokens=20, temperature=0,
            messages=[{"role": "user", "content": "Say OK"}])
        print(f'✓ Sonnet OK!')
    except Exception as e2:
        print(f'✗ Sonnet also failed: {str(e2)[:100]}')

print(f'\n{"="*50}')
print(f'  >>> SELECTED MODEL: {MODEL}')
print(f'{"="*50}')

# ═══ JSON parser ═══
def parse_json(text):
    for prefix in ['```json','```']:
        if text.startswith(prefix): text=text[len(prefix):]
    if text.endswith('```'): text=text[:-3]
    text=text.strip()
    if '[' not in text: return []
    text=text[text.index('['):]
    text=re.sub(r',\s*]',']',text); text=re.sub(r',\s*}','}',text)
    try: return json.loads(text)
    except: pass
    last=text.rfind('}')
    if last>0:
        try: return json.loads(re.sub(r',\s*]',']',text[:last+1]+']'))
        except: pass
    items=[]
    for m in re.finditer(r'\{[^{}]+\}',text):
        try: items.append(json.loads(m.group()))
        except: pass
    return items

# ═══ Claude caller with retries ═══
def call_claude(prompt, temp=0.8, retries=4):
    for a in range(retries):
        try:
            resp = client.messages.create(
                model=MODEL, max_tokens=8192,
                temperature=temp,
                messages=[{"role": "user", "content": prompt}])
            text = resp.content[0].text
            r = parse_json(text)
            if r: return r
            if a < retries-1:
                print(f'[empty JSON, retry {a+1}/{retries}]', end=' ', flush=True)
                time.sleep(5)
            else: return []
        except anthropic.RateLimitError:
            w = 30*(a+1)
            print(f'[Anthropic rate limit, wait {w}s]', end=' ', flush=True)
            time.sleep(w)
        except anthropic.APIStatusError as e:
            if e.status_code in (401, 403):
                # auth-errors не лечатся ретраем — fail-fast с actionable хинтом
                _hint = "check ANTHROPIC_API_KEY in env" if e.status_code == 401 else "API key valid but no access (model permission?)"
                print(f'[Anthropic AUTH {e.status_code} — {_hint}]', flush=True)
                return []
            if e.status_code == 529 or 'overloaded' in str(e).lower():
                w = 20*(a+1)
                print(f'[Anthropic overloaded (529), wait {w}s]', end=' ', flush=True)
                time.sleep(w)
            elif a < retries-1:
                print(f'[Anthropic {e.status_code}, retry {a+1}/{retries}]', end=' ', flush=True)
                time.sleep(10)
            else:
                print(f'[Anthropic FAILED ({e.status_code}): {str(e)[:80]}]', flush=True)
                return []
        except Exception as e:
            if a < retries-1:
                print(f'[Anthropic err: {str(e)[:50]}, retry {a+1}/{retries}]', end=' ', flush=True)
                time.sleep(10)
            else:
                print(f'[Anthropic FAILED: {str(e)[:80]}]', flush=True)
                return []
    return []

def chunk_list(lst,n):
    for i in range(0,len(lst),n): yield lst[i:i+n]


# ════ DB HELPERS ════
def db_update_status(product_id, status, **kwargs):
    sets = ['status=?', 'updated_at=CURRENT_TIMESTAMP']
    vals = [status]
    for k, v in kwargs.items():
        sets.append(f'{k}=?'); vals.append(v)
    vals.append(product_id)
    db.execute(f'UPDATE products SET {",".join(sets)} WHERE id=?', vals)
    db.commit()

def get_product_collections(product_id):
    rows = db.execute("""SELECT c.seo_title, c.full_url, c.shopify_collection_id FROM collections c
        JOIN product_collections pc ON c.id = pc.collection_id WHERE pc.product_id = ?""", (product_id,)).fetchall()
    return [(r['seo_title'], r['full_url'], r.get('shopify_collection_id','')) for r in rows]

# ════ SHOPIFY ════
import httpx, asyncio, aiohttp
from playwright.async_api import async_playwright

_shop_domain = SHOPIFY_STORE.replace(".myshopify.com","")
SHOPIFY_TOKEN = None
try:
    _r = httpx.post(f"https://{_shop_domain}.myshopify.com/admin/oauth/access_token",
        data={"grant_type":"client_credentials","client_id":SHOPIFY_CLIENT_ID,"client_secret":SHOPIFY_CLIENT_SECRET},
        headers={"Content-Type":"application/x-www-form-urlencoded"}, timeout=30)
    if _r.status_code == 200: SHOPIFY_TOKEN = _r.json()["access_token"]; print(f"Shopify token OK")
    else: print(f"Shopify token FAILED: {_r.status_code}")
except Exception as e: print(f"Shopify: {e}")

SHOP_GQL = f"https://{_shop_domain}.myshopify.com/admin/api/2025-04/graphql.json"
SHOP_HEADERS = {"X-Shopify-Access-Token": SHOPIFY_TOKEN or "", "Content-Type": "application/json"}

async def shopify_gql(query, variables=None, retries=3):
    if not SHOPIFY_TOKEN: return None
    async with httpx.AsyncClient(timeout=60) as c:
        for attempt in range(retries):
            r = await c.post(SHOP_GQL, headers=SHOP_HEADERS, json={"query": query, "variables": variables or {}})
            if r.status_code == 429:
                wait = int(r.headers.get('Retry-After', 2 * (attempt + 1)))
                print(f'    [Shopify 429, wait {wait}s]', end=' ', flush=True)
                await asyncio.sleep(wait)
                continue
            if r.status_code == 200:
                data = r.json()
                # Check for throttled error in response
                errs = data.get('errors', [])
                if errs and any('throttled' in str(e).lower() for e in errs):
                    await asyncio.sleep(2 * (attempt + 1))
                    continue
                return data
            # Other error
            if attempt < retries - 1:
                await asyncio.sleep(1)
            else:
                print(f'    [Shopify {r.status_code}]', flush=True)
                return r.json() if r.status_code < 500 else None
    return None

async def shopify_create_collection(title, handle, seo_title, seo_desc):
    # v9.3: Rich collection description for SEO
    # Get top keywords + volume from DB if available
    coll_row = db.execute('SELECT top_keywords, total_volume FROM collections WHERE seo_title=?', (title,)).fetchone()
    top_kw = coll_row['top_keywords'] if coll_row and coll_row['top_keywords'] else ''
    tot_vol = coll_row['total_volume'] if coll_row and coll_row['total_volume'] else 0
    coll_html = generate_collection_html(title, seo_desc, top_kw, tot_vol)
    r = await shopify_gql("mutation($i:CollectionInput!){collectionCreate(input:$i){collection{id}userErrors{field message}}}",
        {"i":{"title":title,"handle":handle,"descriptionHtml":coll_html,"seo":{"title":seo_title or title,"description":seo_desc or ""}}})
    return r['data']['collectionCreate']['collection']['id'] if r and r.get('data',{}).get('collectionCreate',{}).get('collection') else None


def generate_collection_html(title, meta_desc, top_keywords="", total_volume=0):
    """Generate rich SEO collection description using Claude with keyword data."""
    try:
        kw_instruction = ""
        if top_keywords:
            kw_instruction = f"\nTop SEO keywords to weave in naturally (sorted by search volume): {top_keywords}\n"
        if total_volume > 0:
            kw_instruction += f"This category has {total_volume:,} monthly searches — write for this audience.\n"

        prompt = f"""Write a short, compelling collection page description for an online store.

Collection: {title}
Meta description: {meta_desc}
{kw_instruction}

Rules:
- 3-4 sentences, 80-120 words
- Mention the collection name naturally for SEO
- Include a benefit or reason to browse
- End with a soft CTA like "Find your perfect..." or "Explore our curated..."
- Return ONLY the HTML, no preamble
- Use <p> tags, one paragraph
- Casual, warm tone — not corporate
- Include 1-2 relevant long-tail phrases naturally

Example output:
<p>Discover our handpicked Teeth Whitening collection — everything you need for a brighter, more confident smile at home. From professional-grade whitening kits to gentle daily treatments, each product is selected for real results without the dentist price tag. Whether you're prepping for a big event or building a daily routine, you'll find options for every sensitivity level. Find your perfect whitening match today.</p>""" + HTML_CLEAN_OUTPUT_RULES

        resp = client.messages.create(model=MODEL_VISION, max_tokens=300, temperature=0.7,
            messages=[{"role":"user","content":prompt}])
        html = resp.content[0].text.strip()
        # Clean any markdown wrapping
        if html.startswith("```"): html = html.split("\n", 1)[-1].rsplit("\n", 1)[0]
        if '<p>' in html: return html
    except Exception as e:
        print(f'    Collection desc generation failed: {str(e)[:60]}')
    return f"<p>Shop our {title} collection — carefully curated for quality and value. Browse the full range and find exactly what you're looking for.</p>"

async def shopify_find_product(title):
    r = await shopify_gql('query($q:String!){products(first:1,query:$q){nodes{id}}}', {"q":f'title:"{title}"'})
    nodes = r.get('data',{}).get('products',{}).get('nodes',[]) if r else []
    return nodes[0]['id'] if nodes else None

async def shopify_create_product(title, body_html, handle, seo_title, seo_desc, tags=None, product_type="", vendor=""):
    inp = {"title":title,"descriptionHtml":body_html,"handle":handle,"status":PUBLISH_STATUS,
           "seo":{"title":seo_title,"description":seo_desc},"tags":tags or []}
    if product_type: inp["productType"] = product_type
    if vendor: inp["vendor"] = vendor
    r = await shopify_gql("mutation($p:ProductCreateInput!){productCreate(product:$p){product{id}userErrors{field message}}}",
        {"p": inp})
    return r['data']['productCreate']['product']['id'] if r and r.get('data',{}).get('productCreate',{}).get('product') else None

async def shopify_update_product(pid, body_html, seo_title, seo_desc, handle=None):
    inp = {"id":pid,"descriptionHtml":body_html,"seo":{"title":seo_title,"description":seo_desc}}
    if handle: inp["handle"] = handle
    r = await shopify_gql("mutation($p:ProductUpdateInput!){productUpdate(product:$p){product{id}userErrors{field message}}}",{"p":inp})
    return r['data']['productUpdate']['product']['id'] if r and r.get('data',{}).get('productUpdate',{}).get('product') else None

async def shopify_add_to_collections(product_id, collection_ids):
    for cid in collection_ids:
        if cid: await shopify_gql("mutation($id:ID!,$pids:[ID!]!){collectionAddProducts(id:$id,productIds:$pids){collection{id}}}",{"id":cid,"pids":[product_id]})


# ════ SCRAPE EPROLO (v9: + variants) ════
async def scrape_eprolo(url):
    result = {"title":"","description":"","top_image_urls":[],"desc_image_urls":[],"cost_price":0,"variants":[],"specs":{},"video_urls":[],"shipping_time":"","weight_g":0,"size_chart":[]}
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        ctx = await browser.new_context(user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
        page = await ctx.new_page()
        await page.goto(url, wait_until="networkidle", timeout=60000)
        await page.wait_for_timeout(5000)
        try:
            el = await page.query_selector("h1")
            if el: result["title"] = (await el.inner_text()).strip()
        except: pass
        if not result["title"]: result["title"] = await page.title()
        html = await page.content()

        # ── v9: Scrape variants (colors, sizes, options) ──
        variants_raw = []
        try:
            # Method 1: Look for option selectors on page
            option_groups = await page.query_selector_all('[class*="option"], [class*="variant"], [class*="sku-select"], [class*="spec-item"], [class*="attribute"]')
            for grp in option_groups:
                label_el = await grp.query_selector('label, [class*="label"], [class*="name"], [class*="title"], dt')
                items = await grp.query_selector_all('button, li, option, [class*="item"], [class*="value"], dd, span[class*="val"]')
                if not items or len(items) < 1: continue
                label_text = ""
                if label_el:
                    label_text = (await label_el.inner_text()).strip().rstrip(':')
                values = []
                for item in items:
                    val = (await item.inner_text()).strip()
                    if val and len(val) < 40 and val not in values:
                        values.append(val)
                if values and label_text:
                    variants_raw.append({"option_name": label_text, "values": values})

            # Method 2: Parse from JSON-LD or embedded data
            if not variants_raw:
                for pattern in [r'"variants"\s*:\s*\[([^\]]{10,})\]', r'"options"\s*:\s*\[([^\]]{10,})\]',
                                r'"skus"\s*:\s*\[([^\]]{10,})\]']:
                    m = re.search(pattern, html)
                    if m:
                        try:
                            raw_list = json.loads('[' + m.group(1) + ']')
                            opt_names = set()
                            for v in raw_list:
                                if isinstance(v, dict):
                                    for k in v:
                                        kl = k.lower()
                                        if any(w in kl for w in ['color','colour','size','style','option','type','flavor','scent','shade','material']):
                                            opt_names.add(k)
                            for opt in opt_names:
                                vals = list(dict.fromkeys([str(v.get(opt,'')) for v in raw_list if isinstance(v,dict) and v.get(opt)]))
                                if vals: variants_raw.append({"option_name": opt.replace('_',' ').title(), "values": vals})
                        except: pass
                        break

            # Method 3: Color swatches
            if not any(v['option_name'].lower() in ('color','colour','shade') for v in variants_raw):
                swatches = await page.query_selector_all('[class*="swatch"], [class*="color"] img, [class*="colour"] img')
                colors = []
                for sw in swatches:
                    alt = await sw.get_attribute('alt') or await sw.get_attribute('title') or ''
                    alt = alt.strip()
                    if alt and len(alt) < 30 and alt not in colors: colors.append(alt)
                if len(colors) >= 2:
                    variants_raw.append({"option_name": "Color", "values": colors})

        except Exception as ve:
            print(f'    Variant scrape warning: {str(ve)[:80]}')

        result["variants"] = variants_raw[:3]  # Shopify max 3 options

        # Try to extract per-variant prices from page data
        try:
            variant_prices = {}
            for pat in [r'"variants"\s*:\s*\[([^\]]+)\]', r'"skus"\s*:\s*\[([^\]]+)\]']:
                pm = re.search(pat, html)
                if pm:
                    try:
                        vlist = json.loads('[' + pm.group(1) + ']')
                        for vi in vlist:
                            if isinstance(vi, dict):
                                vp = vi.get('price') or vi.get('cost') or vi.get('salePrice')
                                vn = vi.get('name') or vi.get('title') or vi.get('option1','')
                                if vp and vn:
                                    try: variant_prices[str(vn)] = float(str(vp).replace(',','.'))
                                    except: pass
                    except: pass
                    break
            if variant_prices:
                result["variant_prices"] = variant_prices
        except: pass

        # ── Auto-detect product image URLs ──
        # Принимаем ТОЛЬКО картинки с CDN shopifyfile.oss-accelerate.aliyuncs.com/attached
        ALLOWED_IMG_CDN = "shopifyfile.oss-accelerate.aliyuncs.com/attached"
        all_urls = re.findall(r'https?://[^\s"\'<>\)]+\.(?:jpg|jpeg|png|webp)', html)
        all_urls = list(dict.fromkeys(all_urls))
        product_imgs = [u for u in all_urls if ALLOWED_IMG_CDN in u]

        boundary = None
        for pat in [r'(?i)>\s*Description\s*<', r'(?i)description-content',
                    r'(?i)product-description', r'(?i)tab.*description']:
            m = re.search(pat, html)
            if m: boundary = m.start(); break

        if boundary:
            top = []; desc = []
            for u in product_imgs:
                pos = html.find(u)
                if pos >= 0 and pos < boundary: top.append(u)
                else: desc.append(u)
            desc = [u for u in desc if u not in top]
            chunk = re.sub(r'<[^>]+>',' ',html[boundary:boundary+5000])
            result["description"] = re.sub(r'\s+',' ',chunk).strip()[:2000]
        else:
            top = product_imgs[:5]; desc = product_imgs[5:]

        result["top_image_urls"] = top
        result["desc_image_urls"] = desc

        # ── Parse price ──
        try:
            for sel in ['[class*="price"]', '[class*="Price"]', '.product-price', '.sale-price', 'span.price', '[class*="cost"]']:
                price_el = await page.query_selector(sel)
                if price_el:
                    pt = (await price_el.inner_text()).strip()
                    pm = re.search(r'[\d]+[.,]?\d*', pt.replace(',','.'))
                    if pm and float(pm.group()) > 0:
                        result["cost_price"] = float(pm.group()); break
        except: pass
        if result["cost_price"] == 0:
            for pp in [r'(?:USD|\$)\s*([\d]+\.?\d*)', r'"cost"\s*:\s*"?([\d.]+)', r'"price"\s*:\s*"?([\d.]+)']:
                pm = re.search(pp, html)
                if pm:
                    try: result["cost_price"] = float(pm.group(1))
                    except: pass
                    if result["cost_price"] > 0: break


        # ── v9.2: Specs table ──
        try:
            spec_rows = await page.query_selector_all('[class*="spec"] tr, [class*="detail"] tr, [class*="param"] tr, [class*="attribute"] tr, [class*="info-item"], [class*="product-prop"]')
            specs = {}
            for row in spec_rows[:30]:
                cells = await row.query_selector_all('td, th, span, div')
                texts = []
                for cell in cells:
                    t = (await cell.inner_text()).strip()
                    if t and len(t) < 100: texts.append(t)
                if len(texts) >= 2:
                    key = texts[0].rstrip(':').strip()
                    val = texts[1].strip()
                    if key and val and len(key) < 50:
                        specs[key] = val
            result["specs"] = specs
            # Extract weight from specs
            for k, v in specs.items():
                kl = k.lower()
                if any(w in kl for w in ['weight', 'net weight', 'gross weight', 'waga']):
                    wm = re.search(r'([\d.]+)\s*(?:g|gram)', v.lower())
                    if wm: result["weight_g"] = float(wm.group(1)); break
                    wm = re.search(r'([\d.]+)\s*(?:kg)', v.lower())
                    if wm: result["weight_g"] = float(wm.group(1)) * 1000; break
                    wm = re.search(r'([\d.]+)\s*(?:oz)', v.lower())
                    if wm: result["weight_g"] = float(wm.group(1)) * 28.35; break
        except: pass

        # ── v9.2: Video URLs ──
        try:
            videos = []
            for tag in await page.query_selector_all('video source, video[src], [class*="video"] iframe'):
                src = await tag.get_attribute('src') or ''
                if src and 'http' in src and src not in videos:
                    videos.append(src)
            # Also check for video in HTML
            for pat in [r'<video[^>]*src=["\'](https?://[^"\']+)["\'"]', r'<source[^>]*src=["\'](https?://[^"\']+\.mp4)["\'"]']:
                for m in re.finditer(pat, html):
                    if m.group(1) not in videos: videos.append(m.group(1))
            result["video_urls"] = videos[:3]
        except: pass

        # ── v9.2: Shipping time ──
        try:
            for sel in ['[class*="shipping"]', '[class*="delivery"]', '[class*="dispatch"]']:
                el = await page.query_selector(sel)
                if el:
                    txt = (await el.inner_text()).strip()
                    tm = re.search(r'(\d+[-–]\d+)\s*(?:days|business)', txt, re.I)
                    if tm: result["shipping_time"] = tm.group(0); break
        except: pass

        # ── v9.2: Size chart ──
        try:
            size_rows = await page.query_selector_all('[class*="size"] table tr, [class*="measurement"] tr, [class*="chart"] tr')
            chart = []
            for row in size_rows[:20]:
                cells = await row.query_selector_all('td, th')
                texts = [(await c.inner_text()).strip() for c in cells]
                if texts and any(t for t in texts):
                    chart.append(texts)
            result["size_chart"] = chart
        except: pass

        await browser.close()
    var_summary = ', '.join([f'{v["option_name"]}({len(v["values"])})' for v in result["variants"]]) or 'none'
    extras = []
    if result["specs"]: extras.append(f'{len(result["specs"])} specs')
    if result["video_urls"]: extras.append(f'{len(result["video_urls"])} videos')
    if result["weight_g"]: extras.append(f'{result["weight_g"]:.0f}g')
    if result["size_chart"]: extras.append('size chart')
    extra_str = ' | ' + ', '.join(extras) if extras else ''
    print(f'  Scrape: {len(result["top_image_urls"])} top + {len(result["desc_image_urls"])} desc | ${result["cost_price"]:.2f} | variants: {var_summary}{extra_str}')
    return result


async def dl_images(urls, label):
    # Скачиваем только картинки с разрешённого CDN; остальные пропускаем
    ALLOWED_IMG_CDN = "shopifyfile.oss-accelerate.aliyuncs.com/attached"
    urls = [u for u in urls if ALLOWED_IMG_CDN in u]
    imgs = []
    async with httpx.AsyncClient(timeout=30, follow_redirects=True) as c:
        for i, u in enumerate(urls):
            try:
                r = await c.get(u)
                if r.status_code!=200 or len(r.content)>5*1024*1024: continue
                data = r.content
                if data[:4]==b'RIFF' and data[8:12]==b'WEBP': mt="image/webp"
                elif data[:8]==b'\x89PNG\r\n\x1a\n': mt="image/png"
                elif data[:4] in (b'GIF8',): continue
                else: mt="image/jpeg"
                imgs.append({"index":i+1,"url":u,"media_type":mt,"base64":base64.standard_b64encode(r.content).decode("ascii"),"size_kb":len(r.content)/1024})
            except: pass
    return imgs


async def upload_to_shopify(img_list, label="IMG", alt_texts=None):
    if not SHOPIFY_TOKEN:
        return [{"index":img['index'],"url":img['url'],"cdn":False} for img in img_list]
    alt_map = alt_texts or {}
    results = []
    async with httpx.AsyncClient(timeout=60) as gql:
        for img in img_list:
            idx = img['index']; raw = base64.standard_b64decode(img['base64'])
            ext = 'png' if img['media_type']=='image/png' else 'webp' if img['media_type']=='image/webp' else 'jpg'
            fname = f"funnel_{label.lower()}_{idx}.{ext}"
            alt = alt_map.get(idx, f"{label} image {idx}")
            try:
                r1 = await gql.post(SHOP_GQL, headers=SHOP_HEADERS, json={"query":"mutation($i:[StagedUploadInput!]!){stagedUploadsCreate(input:$i){stagedTargets{url parameters{name value}resourceUrl}userErrors{field message}}}","variables":{"i":[{"resource":"FILE","filename":fname,"mimeType":img['media_type'],"fileSize":str(len(raw)),"httpMethod":"POST"}]}})
                targets = r1.json().get('data',{}).get('stagedUploadsCreate',{}).get('stagedTargets',[])
                if not targets: results.append({"index":idx,"url":img['url'],"cdn":False}); continue
                t = targets[0]; params = {p['name']:p['value'] for p in t['parameters']}
                form = aiohttp.FormData()
                for k,v in params.items(): form.add_field(k,v)
                form.add_field('file', raw, filename=fname, content_type=img['media_type'])
                async with aiohttp.ClientSession() as sess:
                    async with sess.post(t['url'], data=form) as resp:
                        if resp.status not in (200,201,204): results.append({"index":idx,"url":img['url'],"cdn":False}); continue
                r2 = await gql.post(SHOP_GQL, headers=SHOP_HEADERS, json={"query":"mutation($f:[FileCreateInput!]!){fileCreate(files:$f){files{id}userErrors{field message}}}","variables":{"f":[{"originalSource":t['resourceUrl'],"contentType":"IMAGE","alt":alt}]}})
                files_made = r2.json().get('data',{}).get('fileCreate',{}).get('files',[])
                if not files_made: results.append({"index":idx,"url":img['url'],"cdn":False}); continue
                file_id = files_made[0]['id']; cdn_url = None
                for _ in range(12):
                    await asyncio.sleep(2)
                    r3 = await gql.post(SHOP_GQL, headers=SHOP_HEADERS, json={"query":'query($id:ID!){node(id:$id){...on MediaImage{image{url}fileStatus}}}', "variables":{"id":file_id}})
                    node = r3.json().get('data',{}).get('node',{}) or {}
                    if node.get('fileStatus')=='READY' and node.get('image',{}).get('url'): cdn_url = node['image']['url']; break
                    elif node.get('fileStatus')=='FAILED': break
                results.append({"index":idx,"url":cdn_url or img['url'],"cdn":bool(cdn_url)})
            except: results.append({"index":idx,"url":img['url'],"cdn":False})
    return results



# ── Pricing ──
def calc_price(cost):
    """cost x5, psychological .90 price. Min $7.90 to ensure margin after Shopify+payment fees."""
    raw = cost * 5
    if raw < 10: price = round(raw) - 0.10
    elif raw < 30: price = (round(raw / 5) * 5) - 0.10
    elif raw < 100: price = (round(raw / 10) * 10) - 0.10
    else: price = (round(raw / 10) * 10) - 0.10
    return max(7.90, round(price, 2))

def calc_compare_price(cost):
    """cost x8, rounded for compare_at (shows strikethrough)"""
    raw = cost * 8
    if raw < 20: return float(max(15, round(raw / 5) * 5))
    return float(round(raw / 10) * 10)

# ── Attach images to product (gallery) ──
async def shopify_attach_media(product_id, image_urls, alt_map=None):
    """Attach images to product gallery with SEO alt text from vision."""
    if not image_urls or not SHOPIFY_TOKEN: return 0
    alt_map = alt_map or {}
    media = [{"originalSource": url, "mediaContentType": "IMAGE", "alt": alt_map.get(i+1, alt_map.get(url, f"Product image {i+1}"))}
             for i, url in enumerate(image_urls) if url.startswith('http')]
    if not media: return 0
    r = await shopify_gql("""mutation($pid:ID!,$m:[CreateMediaInput!]!){
        productCreateMedia(productId:$pid,media:$m){
            media{id status} mediaUserErrors{field message}
        }}""", {"pid": product_id, "m": media})
    if r and r.get('data',{}).get('productCreateMedia',{}).get('media'):
        return len(r['data']['productCreateMedia']['media'])
    errs = r.get('data',{}).get('productCreateMedia',{}).get('mediaUserErrors',[]) if r else []
    if errs: print(f'    Media errors: {errs[:2]}')
    return 0

# ── Set variant price ──
async def shopify_set_price(product_id, price, compare_at):
    """Set price + compare_at on the default variant."""
    # Get default variant ID
    r = await shopify_gql('query($id:ID!){product(id:$id){variants(first:1){nodes{id}}}}',
        {"id": product_id})
    variants = r.get('data',{}).get('product',{}).get('variants',{}).get('nodes',[]) if r else []
    if not variants: return False
    vid = variants[0]['id']
    r2 = await shopify_gql("""mutation($pid:ID!,$v:[ProductVariantsBulkInput!]!){
        productVariantsBulkUpdate(productId:$pid,variants:$v){
            productVariants{id price compareAtPrice} userErrors{field message}
        }}""", {"pid": product_id, "v": [{"id": vid, "price": str(price), "compareAtPrice": str(compare_at)}]})
    ok = r2.get('data',{}).get('productVariantsBulkUpdate',{}).get('productVariants') if r2 else None
    return bool(ok)


# ── Set product metafield ──
async def shopify_set_metafield(product_id, namespace, key, value, mtype="multi_line_text_field"):
    """Set a metafield on a product."""
    r = await shopify_gql("""mutation($m:[MetafieldsSetInput!]!){
        metafieldsSet(metafields:$m){metafields{id} userErrors{field message}}
    }""", {"m": [{"ownerId": product_id, "namespace": namespace, "key": key, "type": mtype, "value": value}]})
    if r and r.get('data',{}).get('metafieldsSet',{}).get('metafields'):
        return True
    errs = r.get('data',{}).get('metafieldsSet',{}).get('userErrors',[]) if r else []
    if errs: print(f'    Metafield error: {errs[:2]}')
    return False


# ════ v9.3: GOOGLE — ADDITIONAL SCHEMA BUILDERS ════

def build_howto_schema(html_content, product_title):
    """Extract HowTo steps from wa-step sections for Google rich snippets."""
    steps = []
    # Find wa-step blocks: <div class="wa-step"...><h4>Title</h4><p>Text</p>
    for m in re.finditer(r'<div[^>]*wa-step[^>]*>\s*<h4[^>]*>(.*?)</h4>\s*<p[^>]*>(.*?)</p>', html_content, re.DOTALL):
        name = re.sub(r'<[^>]+>', '', m.group(1)).strip()
        text = re.sub(r'<[^>]+>', '', m.group(2)).strip()
        if name and text:
            steps.append({"@type": "HowToStep", "name": name, "text": text})
    if len(steps) < 2: return ""
    schema = {
        "@context": "https://schema.org",
        "@type": "HowTo",
        "name": f"How to Use {product_title}",
        "step": steps
    }
    return json.dumps(schema, ensure_ascii=False)


def build_video_schema(video_url, product_title, thumbnail_url="", description=""):
    """Build VideoObject schema for Google video rich snippets."""
    if not video_url: return ""
    schema = {
        "@context": "https://schema.org",
        "@type": "VideoObject",
        "name": f"{product_title} - Product Demo",
        "description": description or f"See {product_title} in action",
        "contentUrl": video_url,
        "uploadDate": "2025-01-01T00:00:00Z",
    }
    if thumbnail_url:
        schema["thumbnailUrl"] = thumbnail_url
    return json.dumps(schema, ensure_ascii=False)


def build_review_schemas(html_content):
    """Extract individual reviews from wa-review sections for Review schema."""
    reviews = []
    for m in re.finditer(r'<div[^>]*wa-review[^>]*>.*?<p[^>]*>(.*?)</p>.*?<div[^>]*wa-review-meta[^>]*>(.*?)</div>', html_content, re.DOTALL):
        body = re.sub(r'<[^>]+>', '', m.group(1)).strip()
        meta = re.sub(r'<[^>]+>', '', m.group(2)).strip()
        # Extract name from meta (usually "— Name, X weeks ago")
        name_match = re.match(r'[—–-]?\s*([A-Za-z][A-Za-z .]+)', meta)
        author = name_match.group(1).strip() if name_match else "Verified Buyer"
        if body:
            import random
            reviews.append({
                "@type": "Review",
                "reviewBody": body[:300],
                "author": {"@type": "Person", "name": author},
                "reviewRating": {"@type": "Rating", "ratingValue": str(random.choice([4,5,5,5,5])), "bestRating": "5"}
            })
    return reviews[:3]


def build_shipping_return_schema():
    """Build OfferShippingDetails + MerchantReturnPolicy for Product schema."""
    shipping = {
        "@type": "OfferShippingDetails",
        "shippingRate": {"@type": "MonetaryAmount", "value": "0", "currency": "USD"},
        "deliveryTime": {
            "@type": "ShippingDeliveryTime",
            "handlingTime": {"@type": "QuantitativeValue", "minValue": 1, "maxValue": 3, "unitCode": "d"},
            "transitTime": {"@type": "QuantitativeValue", "minValue": 7, "maxValue": 15, "unitCode": "d"}
        },
        "shippingDestination": {"@type": "DefinedRegion", "addressCountry": "US"}
    }
    returns = {
        "@type": "MerchantReturnPolicy",
        "applicableCountry": "US",
        "returnPolicyCategory": "https://schema.org/MerchantReturnFiniteReturnWindow",
        "merchantReturnDays": 30,
        "returnMethod": "https://schema.org/ReturnByMail",
        "returnFees": "https://schema.org/FreeReturn"
    }
    return shipping, returns


# ════ v9.3: GOOGLE MERCHANT — EXTENDED METAFIELDS ════

async def shopify_set_merchant_extended(product_id, stage1_data, variants_data):
    """Set extended Google Merchant Center metafields from vision + variant data."""
    if not SHOPIFY_TOKEN: return 0
    metafields = []

    # Gender from vision target audience
    target = stage1_data.get('conversion', {}).get('target_audience', '').lower()
    gender = "unisex"
    if any(w in target for w in ['women', 'female', 'girl', 'her', 'ladies']): gender = "female"
    elif any(w in target for w in ['men', 'male', 'boy', 'his', 'guys']): gender = "male"
    metafields.append({"ownerId":product_id,"namespace":"google","key":"gender","type":"single_line_text_field","value":gender})

    # Material from vision sensory
    material = stage1_data.get('sensory', {}).get('material', '')
    if material:
        metafields.append({"ownerId":product_id,"namespace":"google","key":"material","type":"single_line_text_field","value":material})

    # Color from variants or vision
    colors = stage1_data.get('sensory', {}).get('color_names', [])
    if not colors and variants_data:
        for v in variants_data:
            if v.get('option_name','').lower() in ('color','colour','shade'):
                colors = v['values'][:5]
                break
    if colors:
        metafields.append({"ownerId":product_id,"namespace":"google","key":"color","type":"single_line_text_field","value":", ".join(colors[:3])})

    # Size from variants
    for v in (variants_data or []):
        if v.get('option_name','').lower() in ('size','sizes'):
            metafields.append({"ownerId":product_id,"namespace":"google","key":"size","type":"single_line_text_field","value":", ".join(v['values'][:5])})
            break

    # Item group ID (links variants together)
    metafields.append({"ownerId":product_id,"namespace":"google","key":"custom_label_0","type":"single_line_text_field","value":stage1_data.get('category','General')})

    # Product highlights from vision
    highlights = []
    hero = stage1_data.get('conversion', {}).get('hero_claim', '')
    if hero: highlights.append(hero)
    for claim in stage1_data.get('packaging_text', {}).get('claims', [])[:4]:
        if claim not in highlights: highlights.append(claim)
    for feat in stage1_data.get('physical', {}).get('key_features', [])[:3]:
        if feat not in highlights: highlights.append(feat)
    if highlights:
        metafields.append({"ownerId":product_id,"namespace":"google","key":"custom_label_1","type":"single_line_text_field","value":" | ".join(highlights[:5])})

    if not metafields: return 0
    r = await shopify_gql("mutation($m:[MetafieldsSetInput!]!){metafieldsSet(metafields:$m){metafields{id} userErrors{field message}}}",
        {"m": metafields})
    ok = r.get('data',{}).get('metafieldsSet',{}).get('metafields',[]) if r else []
    return len(ok)


# ════ v9.3: SITEMAP PING ════
def ping_google_sitemap(store_domain):
    """Notify Google of updated sitemap after batch publish."""
    try:
        sitemap_url = f"https://{store_domain}/sitemap.xml"
        r = requests.get(f"https://www.google.com/ping?sitemap={sitemap_url}", timeout=10)
        return r.status_code == 200
    except:
        return False



# ════ v9.4: KEYWORD SCORING — low KD first ════
def kw_score(volume, kd, cpc):
    """Score keywords for STRONG dropped domain (WANELO.com DA 60+).
    KD penalty is linear, not quadratic — we CAN rank for KD 30-40.
    KD 15 keeps 70%, KD 30 keeps 40%, KD 45 keeps 10%."""
    kd = min(kd, 50)
    kd_factor = max(0, (50 - kd) / 50)  # Linear: KD 0=1.0, KD 25=0.5, KD 50=0.0
    cpc_factor = 1 + min(cpc, 5)
    return volume * kd_factor * cpc_factor


def tier_keywords(keywords_with_data):
    """Split keywords into tiers by KD for STRONG domain.
    Tier 1 (KD 0-20): Primary targets → H1, H2, first paragraphs — will rank in weeks
    Tier 2 (KD 21-35): Achievable → body text, alt text, FAQ — will rank in 1-3 months
    Tier 3 (KD 36-50): Stretch targets → mentioned naturally — will rank in 3-6 months"""
    t1, t2, t3 = [], [], []
    for kw_data in keywords_with_data:
        kd = kw_data.get('kd', 50)
        if kd <= 20:
            t1.append(kw_data)
        elif kd <= 35:
            t2.append(kw_data)
        else:
            t3.append(kw_data)
    for t in [t1, t2, t3]:
        t.sort(key=lambda x: -kw_score(x.get('volume',0), x.get('kd',0), x.get('cpc',0)))
    return t1, t2, t3



# ════ v9.5: QUALITY ASSURANCE ════

def validate_scrape(product_data):
    """Check scrape result is usable before continuing."""
    issues = []
    if not product_data.get('title') or len(product_data['title']) < 5:
        issues.append("empty/short title")
    if not product_data.get('top_image_urls') and not product_data.get('desc_image_urls'):
        issues.append("no images found")
    if product_data.get('cost_price', 0) <= 0:
        issues.append("no price found")
    return issues


def validate_html(html, provided_image_urls, provided_collection_handles, primary_keywords=None):
    """Validate generated HTML before publishing. Returns (passed, issues)."""
    issues = []

    # 1. Minimum content length
    text_only = re.sub(r'<[^>]+>', '', html)
    if len(text_only) < 300:
        issues.append(f"too short: {len(text_only)} chars text (min 300)")

    # 2. Tag balance check (critical tags)
    for tag in ['div', 'section', 'details']:
        opens = len(re.findall(f'<{tag}[\\s>]', html))
        closes = html.count(f'</{tag}>')
        if opens != closes:
            issues.append(f"unbalanced <{tag}>: {opens} opens vs {closes} closes")

    # 3. Image URL verification — all src must be from provided list
    img_srcs = re.findall(r'<img[^>]*src=["\'](https?://[^"\'>]+)', html)
    provided_set = set(provided_image_urls)
    for src in img_srcs:
        if src not in provided_set:
            issues.append(f"image not in provided list: {src[:60]}")

    # 4. Collection URL verification
    coll_hrefs = re.findall(r'href=["\'](/collections/[^"\'>]+)', html)
    for href in coll_hrefs:
        handle = href.replace('/collections/', '').strip('/')
        if handle and handle not in provided_collection_handles:
            issues.append(f"collection not found: {handle}")

    # 5b. Check inline links exist (not just footer)
    # Find links NOT inside wa-collections-grid (= inline body links)
    body_links = re.findall(r'class=["\']*wa-link["\']*[^>]*>([^<]+)', html)
    footer_links = re.findall(r'wa-collections-grid.*?</div>', html, re.DOTALL)
    inline_count = len(body_links) - (len(re.findall(r'<a[^>]*>', footer_links[0])) if footer_links else 0)
    if inline_count < 1 and len(coll_hrefs) > 0:
        issues.append("no inline body links found (all links in footer only)")

    # 5. Keyword presence check (top 3 primary keywords)
    if primary_keywords:
        found = 0
        for kw in primary_keywords[:3]:
            if kw.lower() in html.lower():
                found += 1
        if found == 0 and len(primary_keywords) > 0:
            issues.append(f"none of top 3 keywords found in HTML: {primary_keywords[:3]}")

    # 6. No empty image alt
    empty_alts = len(re.findall(r'alt=["\']{2}|alt=""', html))
    if empty_alts > 0:
        issues.append(f"{empty_alts} images with empty alt text")

    # 7. wa-page wrapper present
    if 'class="rte wa-page"' not in html and "class='rte wa-page'" not in html:
        issues.append("missing wa-page wrapper")

    passed = len(issues) == 0
    return passed, issues


def validate_meta(meta_dict):
    """Validate SEO meta output."""
    issues = []
    seo = meta_dict.get('seo_meta', {})
    title = seo.get('title', '')
    desc = seo.get('description', '')
    handle = seo.get('handle', '')

    if len(title) < 30: issues.append(f"SEO title too short: {len(title)} chars")
    if len(title) > 70: issues.append(f"SEO title too long: {len(title)} chars")
    if len(desc) < 80: issues.append(f"meta description too short: {len(desc)} chars")
    if len(desc) > 165: issues.append(f"meta description too long: {len(desc)} chars")
    if not handle: issues.append("no URL handle")
    if not meta_dict.get('short_description'): issues.append("no short description")

    return len(issues) == 0, issues


print('All helpers OK')


# ════ v9: CREATE SHOPIFY VARIANTS ════
async def shopify_create_variants(product_id, variants_data, base_cost):
    """Create product options and variant combinations in Shopify."""
    if not variants_data or not SHOPIFY_TOKEN: return 0
    options = variants_data[:3]
    for opt in options:
        opt['values'] = opt['values'][:100]

    option_inputs = [{"name": opt["option_name"], "values": [{"name": v} for v in opt["values"]]} for opt in options]
    r = await shopify_gql("""mutation($pid:ID!,$opts:[OptionCreateInput!]!){
        productOptionsCreate(productId:$pid, options:$opts){
            product{id options{id name values}} userErrors{field message}
        }}""", {"pid": product_id, "opts": option_inputs})
    errs = r.get('data',{}).get('productOptionsCreate',{}).get('userErrors',[]) if r else []
    if errs:
        print(f'    Option errors: {[e["message"] for e in errs[:3]]}')

    await asyncio.sleep(1)
    r2 = await shopify_gql('query($id:ID!){product(id:$id){variants(first:100){nodes{id title}}}}',
        {"id": product_id})
    all_vars = r2.get('data',{}).get('product',{}).get('variants',{}).get('nodes',[]) if r2 else []
    if all_vars and base_cost > 0:
        batch = []
        _vprices = variants_data[0].get('_variant_prices', {}) if variants_data else {}
        for v in all_vars:
            vt = v.get("title", "")
            # Use per-variant cost from EPROLO if available
            _vc = None
            for vname, vprice in _vprices.items():
                if vname.lower() in vt.lower() or vt.lower() in vname.lower():
                    _vc = vprice; break
            if _vc and _vc > 0:
                p, cp = calc_price(_vc), calc_compare_price(_vc)
            else:
                p, cp = calc_variant_price(base_cost, vt)
            batch.append({"id": v["id"], "price": str(p), "compareAtPrice": str(cp)})
        for i in range(0, len(batch), 50):
            chunk = batch[i:i+50]
            await shopify_gql("""mutation($pid:ID!,$v:[ProductVariantsBulkInput!]!){
                productVariantsBulkUpdate(productId:$pid,variants:$v){
                    productVariants{id} userErrors{field message}
                }}""", {"pid": product_id, "v": chunk})
            await asyncio.sleep(0.3)
    return len(all_vars)


# ════ v9: HTML POST-PROCESSOR ════
def sanitize_html(html_str):
    """Enforce minimum font sizes, max-widths, and mobile safety."""
    import re as _re

    # 1. Kill any font-size below 15px
    def _fix_fontsize(m):
        val = m.group(1)
        px_m = _re.match(r'([\d.]+)\s*px', val)
        if px_m and float(px_m.group(1)) < 15:
            return 'font-size:0.9375rem'
        rem_m = _re.match(r'([\d.]+)\s*rem', val)
        if rem_m and float(rem_m.group(1)) < 0.9375:
            return 'font-size:0.9375rem'
        em_m = _re.match(r'([\d.]+)\s*em', val)
        if em_m and float(em_m.group(1)) < 0.9375:
            return 'font-size:0.9375rem'
        pt_m = _re.match(r'([\d.]+)\s*pt', val)
        if pt_m and float(pt_m.group(1)) < 11.25:
            return 'font-size:0.9375rem'
        return m.group(0)

    html_str = _re.sub(r'font-size\s*:\s*([\d.]+\s*(?:px|rem|em|pt))', _fix_fontsize, html_str)

    # 2. Max-width guard disabled — using full container width
    # Images are capped at 720px via CSS, content uses full width

    # 3. Ensure all images have loading="lazy"
    html_str = _re.sub(r'<img(?![^>]*loading=)', '<img loading="lazy"', html_str)

    # 4. Add tap highlight fix for mobile
    if '.wa-page' in html_str and '-webkit-tap-highlight' not in html_str:
        html_str = html_str.replace(
            '.wa-page',
            '.wa-page{-webkit-tap-highlight-color:transparent} .wa-page',
            1
        )

    # 5. Accessibility: add lang and role
    html_str = html_str.replace('<div class="rte wa-page">', '<div class="rte wa-page" role="article" lang="en">', 1)

    # 6. Ensure all interactive elements are accessible
    html_str = html_str.replace('<details class="wa-faq">', '<details class="wa-faq" role="group">')

    return html_str


# ════ v9: SMART INTERLINKING ════
def get_smart_interlinks(product_id, product_name, all_shopify_collections):
    """Return relevant collection links with keyword-rich anchor text options."""
    own_colls = get_product_collections(product_id)
    own_handles = set()
    links = []

    for title, url, sid in own_colls:
        if not sid: continue
        handle = url.replace('/collections/','') if url else ''
        own_handles.add(handle)
        # Get keyword anchors from DB
        anchors = _get_collection_anchors(title)
        links.append({"title": title, "handle": handle, "relevance": "own", "anchors": anchors})

    # Find related collections by keyword overlap
    name_words = set(product_name.lower().split())
    stop_words = {'the','a','an','for','and','or','with','in','of','to','set','kit','pro','new','best','2','3','4','5','pack','pcs','pc'}
    name_words -= stop_words
    scored = []
    for c in all_shopify_collections:
        h = c.get('handle','')
        if h in own_handles: continue
        title_words = set(c.get('title','').lower().split())
        handle_words = set(h.replace('-',' ').lower().split())
        overlap = len(name_words & (title_words | handle_words))
        if overlap > 0:
            scored.append((overlap, c))

    scored.sort(key=lambda x: -x[0])
    for score, c in scored[:4]:
        coll_title = c.get('title','')
        anchors = _get_collection_anchors(coll_title)
        links.append({"title": coll_title, "handle": c["handle"], "relevance": "related", "anchors": anchors})

    return links[:6]


def _get_collection_anchors(collection_title):
    """Get 3-5 keyword-rich anchor text options for a collection from DB."""
    anchors = []

    # Source 1: top_keywords from collections table
    row = db.execute('SELECT top_keywords FROM collections WHERE seo_title=?', (collection_title,)).fetchone()
    if row and row['top_keywords']:
        kws = [k.strip() for k in row['top_keywords'].split(',')]
        # Pick short ones (2-4 words) — best for anchor text
        for kw in kws:
            word_count = len(kw.split())
            if 2 <= word_count <= 4 and kw not in anchors:
                anchors.append(kw)
            if len(anchors) >= 3: break

    # Source 2: top volume keywords from seo_keywords table
    if len(anchors) < 3:
        rows = db.execute(
            'SELECT keyword FROM seo_keywords WHERE collection_name=? AND length(keyword) - length(replace(keyword, " ", "")) BETWEEN 1 AND 3 ORDER BY volume DESC LIMIT 5',
            (collection_title,)).fetchall()
        for r in rows:
            kw = r['keyword']
            if kw not in anchors:
                anchors.append(kw)
            if len(anchors) >= 5: break

    # Fallback: derive from collection title
    if not anchors:
        # "Luxury Soap Gift Sets & Hand Cream Collections" → "soap gift sets", "hand cream"
        title_lower = collection_title.lower()
        for remove in ['luxury','premium','best','top','&','collections','collection','for','-','daily','natural','essential','essentials']:
            title_lower = title_lower.replace(remove, ' ')
        parts = [p.strip() for p in title_lower.split() if len(p.strip()) > 2]
        if len(parts) >= 2:
            anchors.append(' '.join(parts[:3]))

    return anchors[:5]


# ════ v9.2: SET INVENTORY ════
async def shopify_set_inventory(product_id, quantity=999):
    """Set inventory quantity on all variants + enable tracking."""
    if not SHOPIFY_TOKEN: return False
    r = await shopify_gql('query($id:ID!){product(id:$id){variants(first:100){nodes{id inventoryItem{id}}}}}',
        {"id": product_id})
    variants = r.get('data',{}).get('product',{}).get('variants',{}).get('nodes',[]) if r else []
    count = 0
    for v in variants:
        inv_id = v.get('inventoryItem',{}).get('id')
        if not inv_id: continue
        # Enable tracking
        await shopify_gql("""mutation($id:ID!,$input:InventoryItemInput!){
            inventoryItemUpdate(id:$id,input:$input){inventoryItem{id tracked} userErrors{field message}}}""",
            {"id": inv_id, "input": {"tracked": True}})
        # Get location
        r2 = await shopify_gql('query{locations(first:1){nodes{id}}}')
        loc_nodes = r2.get('data',{}).get('locations',{}).get('nodes',[]) if r2 and r2.get('data') else []
        loc = loc_nodes[0].get('id') if loc_nodes else None
        if loc:
            await shopify_gql("""mutation($input:InventorySetQuantitiesInput!){
                inventorySetQuantities(input:$input){inventoryAdjustmentGroup{reason} userErrors{field message}}}""",
                {"input":{"reason":"correction","name":"available","quantities":[{"inventoryItemId":inv_id,"locationId":loc,"quantity":quantity}]}})
            count += 1
        await asyncio.sleep(0.2)
    return count


# ════ v9.2: SET PRODUCT WEIGHT ════
async def shopify_set_weight(product_id, weight_grams):
    """Set weight on all variants for shipping calculator."""
    if not SHOPIFY_TOKEN or weight_grams <= 0: return False
    r = await shopify_gql('query($id:ID!){product(id:$id){variants(first:100){nodes{id}}}}',
        {"id": product_id})
    variants = r.get('data',{}).get('product',{}).get('variants',{}).get('nodes',[]) if r else []
    if not variants: return False
    batch = [{"id":v["id"],"weight":weight_grams,"weightUnit":"GRAMS"} for v in variants]
    for i in range(0, len(batch), 50):
        await shopify_gql("""mutation($pid:ID!,$v:[ProductVariantsBulkInput!]!){
            productVariantsBulkUpdate(productId:$pid,variants:$v){
                productVariants{id} userErrors{field message}}}""",
            {"pid": product_id, "v": batch[i:i+50]})
        await asyncio.sleep(0.3)
    return True


# ════ v9.2: GOOGLE SHOPPING METAFIELDS ════
async def shopify_set_google_shopping(product_id, product_type_category, condition="new", age_group="adult"):
    """Set Google Shopping metafields for Merchant Center feed."""
    if not SHOPIFY_TOKEN: return False
    metafields = [
        {"ownerId":product_id,"namespace":"google","key":"product_category","type":"single_line_text_field","value":product_type_category},
        {"ownerId":product_id,"namespace":"google","key":"condition","type":"single_line_text_field","value":condition},
        {"ownerId":product_id,"namespace":"google","key":"age_group","type":"single_line_text_field","value":age_group},
    ]
    r = await shopify_gql("mutation($m:[MetafieldsSetInput!]!){metafieldsSet(metafields:$m){metafields{id} userErrors{field message}}}",
        {"m": metafields})
    return bool(r and r.get('data',{}).get('metafieldsSet',{}).get('metafields'))


# ════ v9.2: SEO IMAGE FILENAME ════
def seo_image_filename(product_title, index, label="product"):
    """Generate SEO-friendly image filename from product title."""
    slug = re.sub(r'[^a-z0-9]+', '-', product_title.lower()).strip('-')[:50]
    return f"{slug}-{label}-{index}"


# ════ v9.2: SCHEMA.ORG JSON-LD ════
def build_schema_jsonld(title, description, handle, price, compare_price, images, category=""):
    """Build Product + FAQ + Breadcrumb schema for rich snippets."""
    url = f"https://{STORE_DOMAIN}/products/{handle}" if handle else ""
    schema_product = {
        "@context": "https://schema.org",
        "@type": "Product",
        "name": title,
        "description": description,
        "url": url,
        "brand": {"@type": "Brand", "name": STORE_VENDOR},
        "offers": {
            "@type": "Offer",
            "price": str(price),
            "priceCurrency": "USD",
            "availability": "https://schema.org/InStock",
            "seller": {"@type": "Organization", "name": STORE_VENDOR},
            "shippingDetails": build_shipping_return_schema()[0],
            "hasMerchantReturnPolicy": build_shipping_return_schema()[1],
        },
        # AggregateRating: REMOVED — Judge.me app handles real review schema.
        # Fake ratings risk Google manual action. Judge.me injects its own
        # AggregateRating + Review schema from verified purchases.
        "brand": {"@type": "Brand", "name": STORE_VENDOR}
    }
    if images:
        schema_product["image"] = images[:5]
    if compare_price and compare_price > price:
        schema_product["offers"]["priceValidUntil"] = "2026-12-31"
        # Google Shopping: sale_price with date range
        from datetime import datetime, timedelta
        now = datetime.now().strftime("%Y-%m-%dT00:00:00Z")
        end = (datetime.now() + timedelta(days=90)).strftime("%Y-%m-%dT00:00:00Z")
        schema_product["offers"]["validFrom"] = now

    schema_breadcrumb = {
        "@context": "https://schema.org",
        "@type": "BreadcrumbList",
        "itemListElement": [
            {"@type": "ListItem", "position": 1, "name": "Home", "item": f"https://{STORE_DOMAIN}/"},
        ]
    }
    if category:
        schema_breadcrumb["itemListElement"].append(
            {"@type": "ListItem", "position": 2, "name": category, "item": f"https://{STORE_DOMAIN}/collections/{re.sub(r'[^a-z0-9]+', '-', category.lower()).strip('-')}"}
        )
    schema_breadcrumb["itemListElement"].append(
        {"@type": "ListItem", "position": len(schema_breadcrumb["itemListElement"]) + 1, "name": title}
    )

    schemas = [json.dumps(schema_product, ensure_ascii=False), json.dumps(schema_breadcrumb, ensure_ascii=False)]
    return "\n".join(schemas)


def build_faq_schema(html_content):
    """Extract FAQ questions from generated HTML and build FAQPage schema."""
    faqs = []
    for m in re.finditer(r'<summary[^>]*>(.*?)</summary>', html_content, re.DOTALL):
        q_text = re.sub(r'<[^>]+>', '', m.group(1)).strip()
        # Find the next wa-faq-body after this summary
        rest = html_content[m.end():]
        a_match = re.search(r'<div[^>]*wa-faq-body[^>]*>(.*?)</div>', rest, re.DOTALL)
        if a_match:
            a_text = re.sub(r'<[^>]+>', '', a_match.group(1)).strip()
            if q_text and a_text:
                faqs.append({"@type": "Question", "name": q_text,
                    "acceptedAnswer": {"@type": "Answer", "text": a_text[:500]}})
    if len(faqs) < 2: return ""
    schema = {"@context": "https://schema.org", "@type": "FAQPage", "mainEntity": faqs}
    return json.dumps(schema, ensure_ascii=False)


# ════ v9.2: BUNDLE DETECTION ════
def detect_bundle(title):
    """Detect if product is a bundle/set and return multiplier."""
    t = title.lower()
    for pat in [r'set\s+of\s+(\d+)', r'(\d+)\s*(?:pcs|pieces|pack|count)', r'(\d+)\s*in\s*1']:
        m = re.search(pat, t)
        if m:
            n = int(m.group(1))
            if 2 <= n <= 20:
                return n
    return 1


# ════ v9.2: TIERED VARIANT PRICING ════
def calc_variant_price(base_cost, variant_title):
    """Adjust price based on variant size/type. Bigger = more expensive."""
    vt = variant_title.lower()
    multiplier = 1.0
    # Size tiers
    if any(s in vt for s in ['xxl', '2xl', '3xl', 'xxxl']): multiplier = 1.3
    elif any(s in vt for s in ['xl', 'extra large']): multiplier = 1.2
    elif any(s in vt for s in ['large', ' l ', ' l,']): multiplier = 1.1
    # Volume tiers
    vm = re.search(r'(\d+)\s*(?:ml|oz|g)', vt)
    if vm:
        vol = float(vm.group(1))
        if 'oz' in vt: vol *= 30
        if 'g' in vt and vol < 10: vol *= 30  # probably oz mislabeled
        if vol > 100: multiplier = max(multiplier, 1.2)
        if vol > 200: multiplier = max(multiplier, 1.4)
    # Premium materials/colors
    if any(s in vt for s in ['gold', 'premium', 'pro', 'deluxe']): multiplier = max(multiplier, 1.15)
    adjusted_cost = base_cost * multiplier
    return calc_price(adjusted_cost), calc_compare_price(adjusted_cost)



# ════ v9.3: GOOGLE — ADDITIONAL SCHEMA BUILDERS ════

def build_howto_schema(html_content, product_title):
    """Extract HowTo steps from wa-step sections for Google rich snippets."""
    steps = []
    # Find wa-step blocks: <div class="wa-step"...><h4>Title</h4><p>Text</p>
    for m in re.finditer(r'<div[^>]*wa-step[^>]*>\s*<h4[^>]*>(.*?)</h4>\s*<p[^>]*>(.*?)</p>', html_content, re.DOTALL):
        name = re.sub(r'<[^>]+>', '', m.group(1)).strip()
        text = re.sub(r'<[^>]+>', '', m.group(2)).strip()
        if name and text:
            steps.append({"@type": "HowToStep", "name": name, "text": text})
    if len(steps) < 2: return ""
    schema = {
        "@context": "https://schema.org",
        "@type": "HowTo",
        "name": f"How to Use {product_title}",
        "step": steps
    }
    return json.dumps(schema, ensure_ascii=False)


def build_video_schema(video_url, product_title, thumbnail_url="", description=""):
    """Build VideoObject schema for Google video rich snippets."""
    if not video_url: return ""
    schema = {
        "@context": "https://schema.org",
        "@type": "VideoObject",
        "name": f"{product_title} - Product Demo",
        "description": description or f"See {product_title} in action",
        "contentUrl": video_url,
        "uploadDate": "2025-01-01T00:00:00Z",
    }
    if thumbnail_url:
        schema["thumbnailUrl"] = thumbnail_url
    return json.dumps(schema, ensure_ascii=False)


def build_review_schemas(html_content):
    """Extract individual reviews from wa-review sections for Review schema."""
    reviews = []
    for m in re.finditer(r'<div[^>]*wa-review[^>]*>.*?<p[^>]*>(.*?)</p>.*?<div[^>]*wa-review-meta[^>]*>(.*?)</div>', html_content, re.DOTALL):
        body = re.sub(r'<[^>]+>', '', m.group(1)).strip()
        meta = re.sub(r'<[^>]+>', '', m.group(2)).strip()
        # Extract name from meta (usually "— Name, X weeks ago")
        name_match = re.match(r'[—–-]?\s*([A-Za-z][A-Za-z .]+)', meta)
        author = name_match.group(1).strip() if name_match else "Verified Buyer"
        if body:
            import random
            reviews.append({
                "@type": "Review",
                "reviewBody": body[:300],
                "author": {"@type": "Person", "name": author},
                "reviewRating": {"@type": "Rating", "ratingValue": str(random.choice([4,5,5,5,5])), "bestRating": "5"}
            })
    return reviews[:3]


def build_shipping_return_schema():
    """Build OfferShippingDetails + MerchantReturnPolicy for Product schema."""
    shipping = {
        "@type": "OfferShippingDetails",
        "shippingRate": {"@type": "MonetaryAmount", "value": "0", "currency": "USD"},
        "deliveryTime": {
            "@type": "ShippingDeliveryTime",
            "handlingTime": {"@type": "QuantitativeValue", "minValue": 1, "maxValue": 3, "unitCode": "d"},
            "transitTime": {"@type": "QuantitativeValue", "minValue": 7, "maxValue": 15, "unitCode": "d"}
        },
        "shippingDestination": {"@type": "DefinedRegion", "addressCountry": "US"}
    }
    returns = {
        "@type": "MerchantReturnPolicy",
        "applicableCountry": "US",
        "returnPolicyCategory": "https://schema.org/MerchantReturnFiniteReturnWindow",
        "merchantReturnDays": 30,
        "returnMethod": "https://schema.org/ReturnByMail",
        "returnFees": "https://schema.org/FreeReturn"
    }
    return shipping, returns


# ════ v9.3: GOOGLE MERCHANT — EXTENDED METAFIELDS ════

async def shopify_set_merchant_extended(product_id, stage1_data, variants_data):
    """Set extended Google Merchant Center metafields from vision + variant data."""
    if not SHOPIFY_TOKEN: return 0
    metafields = []

    # Gender from vision target audience
    target = stage1_data.get('conversion', {}).get('target_audience', '').lower()
    gender = "unisex"
    if any(w in target for w in ['women', 'female', 'girl', 'her', 'ladies']): gender = "female"
    elif any(w in target for w in ['men', 'male', 'boy', 'his', 'guys']): gender = "male"
    metafields.append({"ownerId":product_id,"namespace":"google","key":"gender","type":"single_line_text_field","value":gender})

    # Material from vision sensory
    material = stage1_data.get('sensory', {}).get('material', '')
    if material:
        metafields.append({"ownerId":product_id,"namespace":"google","key":"material","type":"single_line_text_field","value":material})

    # Color from variants or vision
    colors = stage1_data.get('sensory', {}).get('color_names', [])
    if not colors and variants_data:
        for v in variants_data:
            if v.get('option_name','').lower() in ('color','colour','shade'):
                colors = v['values'][:5]
                break
    if colors:
        metafields.append({"ownerId":product_id,"namespace":"google","key":"color","type":"single_line_text_field","value":", ".join(colors[:3])})

    # Size from variants
    for v in (variants_data or []):
        if v.get('option_name','').lower() in ('size','sizes'):
            metafields.append({"ownerId":product_id,"namespace":"google","key":"size","type":"single_line_text_field","value":", ".join(v['values'][:5])})
            break

    # Item group ID (links variants together)
    metafields.append({"ownerId":product_id,"namespace":"google","key":"custom_label_0","type":"single_line_text_field","value":stage1_data.get('category','General')})

    # Product highlights from vision
    highlights = []
    hero = stage1_data.get('conversion', {}).get('hero_claim', '')
    if hero: highlights.append(hero)
    for claim in stage1_data.get('packaging_text', {}).get('claims', [])[:4]:
        if claim not in highlights: highlights.append(claim)
    for feat in stage1_data.get('physical', {}).get('key_features', [])[:3]:
        if feat not in highlights: highlights.append(feat)
    if highlights:
        metafields.append({"ownerId":product_id,"namespace":"google","key":"custom_label_1","type":"single_line_text_field","value":" | ".join(highlights[:5])})

    if not metafields: return 0
    r = await shopify_gql("mutation($m:[MetafieldsSetInput!]!){metafieldsSet(metafields:$m){metafields{id} userErrors{field message}}}",
        {"m": metafields})
    ok = r.get('data',{}).get('metafieldsSet',{}).get('metafields',[]) if r else []
    return len(ok)


# ════ v9.3: SITEMAP PING ════
def ping_google_sitemap(store_domain):
    """Notify Google of updated sitemap after batch publish."""
    try:
        sitemap_url = f"https://{store_domain}/sitemap.xml"
        r = requests.get(f"https://www.google.com/ping?sitemap={sitemap_url}", timeout=10)
        return r.status_code == 200
    except:
        return False



# ════ v9.4: KEYWORD SCORING — low KD first ════
def kw_score(volume, kd, cpc):
    """Score keywords for STRONG dropped domain (WANELO.com DA 60+).
    KD penalty is linear, not quadratic — we CAN rank for KD 30-40.
    KD 15 keeps 70%, KD 30 keeps 40%, KD 45 keeps 10%."""
    kd = min(kd, 50)
    kd_factor = max(0, (50 - kd) / 50)  # Linear: KD 0=1.0, KD 25=0.5, KD 50=0.0
    cpc_factor = 1 + min(cpc, 5)
    return volume * kd_factor * cpc_factor


def tier_keywords(keywords_with_data):
    """Split keywords into tiers by KD for STRONG domain.
    Tier 1 (KD 0-20): Primary targets → H1, H2, first paragraphs — will rank in weeks
    Tier 2 (KD 21-35): Achievable → body text, alt text, FAQ — will rank in 1-3 months
    Tier 3 (KD 36-50): Stretch targets → mentioned naturally — will rank in 3-6 months"""
    t1, t2, t3 = [], [], []
    for kw_data in keywords_with_data:
        kd = kw_data.get('kd', 50)
        if kd <= 20:
            t1.append(kw_data)
        elif kd <= 35:
            t2.append(kw_data)
        else:
            t3.append(kw_data)
    for t in [t1, t2, t3]:
        t.sort(key=lambda x: -kw_score(x.get('volume',0), x.get('kd',0), x.get('cpc',0)))
    return t1, t2, t3



# ════ v9.5: QUALITY ASSURANCE ════

def validate_scrape(product_data):
    """Check scrape result is usable before continuing."""
    issues = []
    if not product_data.get('title') or len(product_data['title']) < 5:
        issues.append("empty/short title")
    if not product_data.get('top_image_urls') and not product_data.get('desc_image_urls'):
        issues.append("no images found")
    if product_data.get('cost_price', 0) <= 0:
        issues.append("no price found")
    return issues


def validate_html(html, provided_image_urls, provided_collection_handles, primary_keywords=None):
    """Validate generated HTML before publishing. Returns (passed, issues)."""
    issues = []

    # 1. Minimum content length
    text_only = re.sub(r'<[^>]+>', '', html)
    if len(text_only) < 300:
        issues.append(f"too short: {len(text_only)} chars text (min 300)")

    # 2. Tag balance check (critical tags)
    for tag in ['div', 'section', 'details']:
        opens = len(re.findall(f'<{tag}[\\s>]', html))
        closes = html.count(f'</{tag}>')
        if opens != closes:
            issues.append(f"unbalanced <{tag}>: {opens} opens vs {closes} closes")

    # 3. Image URL verification — all src must be from provided list
    img_srcs = re.findall(r'<img[^>]*src=["\'](https?://[^"\'>]+)', html)
    provided_set = set(provided_image_urls)
    for src in img_srcs:
        if src not in provided_set:
            issues.append(f"image not in provided list: {src[:60]}")

    # 4. Collection URL verification
    coll_hrefs = re.findall(r'href=["\'](/collections/[^"\'>]+)', html)
    for href in coll_hrefs:
        handle = href.replace('/collections/', '').strip('/')
        if handle and handle not in provided_collection_handles:
            issues.append(f"collection not found: {handle}")

    # 5b. Check inline links exist (not just footer)
    # Find links NOT inside wa-collections-grid (= inline body links)
    body_links = re.findall(r'class=["\']*wa-link["\']*[^>]*>([^<]+)', html)
    footer_links = re.findall(r'wa-collections-grid.*?</div>', html, re.DOTALL)
    inline_count = len(body_links) - (len(re.findall(r'<a[^>]*>', footer_links[0])) if footer_links else 0)
    if inline_count < 1 and len(coll_hrefs) > 0:
        issues.append("no inline body links found (all links in footer only)")

    # 5. Keyword presence check (top 3 primary keywords)
    if primary_keywords:
        found = 0
        for kw in primary_keywords[:3]:
            if kw.lower() in html.lower():
                found += 1
        if found == 0 and len(primary_keywords) > 0:
            issues.append(f"none of top 3 keywords found in HTML: {primary_keywords[:3]}")

    # 6. No empty image alt
    empty_alts = len(re.findall(r'alt=["\']{2}|alt=""', html))
    if empty_alts > 0:
        issues.append(f"{empty_alts} images with empty alt text")

    # 7. wa-page wrapper present
    if 'class="rte wa-page"' not in html and "class='rte wa-page'" not in html:
        issues.append("missing wa-page wrapper")

    passed = len(issues) == 0
    return passed, issues


def validate_meta(meta_dict):
    """Validate SEO meta output."""
    issues = []
    seo = meta_dict.get('seo_meta', {})
    title = seo.get('title', '')
    desc = seo.get('description', '')
    handle = seo.get('handle', '')

    if len(title) < 30: issues.append(f"SEO title too short: {len(title)} chars")
    if len(title) > 70: issues.append(f"SEO title too long: {len(title)} chars")
    if len(desc) < 80: issues.append(f"meta description too short: {len(desc)} chars")
    if len(desc) > 165: issues.append(f"meta description too long: {len(desc)} chars")
    if not handle: issues.append("no URL handle")
    if not meta_dict.get('short_description'): issues.append("no short description")

    return len(issues) == 0, issues


print('All helpers OK (v9.2: variants, sanitizer, smart interlinks)')


DataForSEO balance: $123.14
Helpers OK
Testing Claude claude-opus-4-6... ✓ OK! Response: "OK"

  >>> SELECTED MODEL: claude-opus-4-6
Shopify token OK
All helpers OK
All helpers OK (v9.2: variants, sanitizer, smart interlinks)


## Cell 3 — SEO Pipeline (auto-skip if Excel loaded)

In [3]:
# ════ SEO PIPELINE ════
_seo_done = db.execute("SELECT value FROM seo_state WHERE key='seo_complete'").fetchone()
_skip = bool(_seo_done or SEO_SKIP)

if _skip:
    print("SEO: SKIPPED (data already in DB)")
    print(f"  Collections: {db.execute('SELECT COUNT(*) FROM collections').fetchone()[0]}")

if not _skip:
    print("SEO: RUNNING...")

    print('\n' + '='*50)
    print('STAGE 1: Categorizing products...')
    print(f'Products: {len(PRODUCTS)} | Batches: {(len(PRODUCTS)+14)//15}')
    print('='*50)

    CAT_PROMPT='Shopify store manager. Categorize each product into a collection.\n''Rules:\n- 1-3 word collection names (product CATEGORY not product name)\n''- 8-25 collections total\n- Similar products same collection\n''- Be specific: "Eyeshadow" not "Makeup", "Teeth Whitening" not "Beauty"\n\n''Products:\nPROD_PH\n\nJSON: [{"p":"exact product name","c":"Collection"}]'

    product_coll_map={}
    total_batches = (len(PRODUCTS)+14)//15
    t_start = time.time()

    for bi,batch in enumerate(chunk_list(PRODUCTS,15)):
        elapsed = time.time() - t_start
        print(f'\n  Batch {bi+1}/{total_batches} ({len(batch)} prods) [{elapsed:.0f}s]', end=' ', flush=True)
        prompt=CAT_PROMPT.replace('PROD_PH','\n'.join([str(i+1)+'. '+p for i,p in enumerate(batch)]))
        result=call_claude(prompt,0.3)
        matched=0
        for item in result:
            if not isinstance(item,dict): continue
            pn=item.get('p',''); co=item.get('c','Uncategorized')
            for product in batch:
                if product not in product_coll_map:
                    if pn.lower()[:20] in product.lower() or product.lower()[:20] in pn.lower():
                        product_coll_map[product]=co; matched+=1; break
        for product in batch:
            if product not in product_coll_map: product_coll_map[product]='Uncategorized'
        print(f'→ {matched}/{len(batch)}', flush=True)
        time.sleep(4)

    coll_products={}
    for p,c in product_coll_map.items():
        if c not in coll_products: coll_products[c]=[]
        coll_products[c].append(p)

    print(f'\n✓ Stage 1 done in {time.time()-t_start:.0f}s')
    print(f'Collections ({len(coll_products)}):')
    for c in sorted(coll_products,key=lambda x:len(coll_products[x]),reverse=True):
        print(f'  {c}: {len(coll_products[c])}')

    SEED_PROMPT = '''You are an SEO keyword expert.
    I need SHORT SEED keywords (1-3 words) for a Google Ads keyword research tool.
    These must be GENERIC product category terms that MILLIONS of people search.

    GOOD seeds: "eyeliner", "teeth whitening", "body glitter", "matte lipstick", "cleansing oil"
    BAD seeds: "buy waterproof eyeliner online", "best anti aging firming serum for wrinkles"

    Rules:
    - 1-3 words MAX per seed
    - Must be a real product category people search on Google
    - NO brand names
    - Generate 5-8 seeds per collection
    - Include the collection name itself as a seed

    Category: CAT_PH
    Collections and their products:
    DATA_PH

    JSON: [{"seed":"short keyword","c":"Collection Name"}]'''

    all_seeds = []
    coll_names = [c for c in coll_products if c != 'Uncategorized']
    total_batches = (len(coll_names)+7)//8
    t_start = time.time()
    print(f'\nSTAGE 2: Extracting seeds from {len(coll_names)} collections ({total_batches} batches)')

    for bi, batch in enumerate(chunk_list(coll_names, 8)):
        elapsed = time.time() - t_start
        print(f'  Batch {bi+1}/{total_batches} [{elapsed:.0f}s]', end=' ', flush=True)
        data = ''
        for c in batch:
            prods = coll_products[c][:5]
            data += '\n'+c+': ' + ', '.join([p[:40] for p in prods]) + '\n'
        prompt = SEED_PROMPT.replace('CAT_PH', CATEGORY).replace('DATA_PH', data)
        result = call_claude(prompt, 0.5)
        cnt = 0
        for item in result:
            if isinstance(item, dict) and item.get('seed'):
                seed = item['seed'].lower().strip()
                coll = item.get('c', '')
                if len(seed) > 2 and len(seed.split()) <= 4 and not is_brand(seed):
                    all_seeds.append({'seed': seed, 'collection': coll})
                    cnt += 1
        print(f'→ +{cnt} seeds', flush=True)
        time.sleep(4)

    # Deduplicate seeds
    seen = set()
    unique_seeds = []
    for s in all_seeds:
        if s['seed'] not in seen:
            seen.add(s['seed']); unique_seeds.append(s)

    print(f'\n✓ Seeds: {len(unique_seeds)} (from {len(all_seeds)} raw, {len(all_seeds)-len(unique_seeds)} deduped)')
    for s in unique_seeds[:20]: print(f'  [{s["collection"]}] {s["seed"]}')

    seed_list = [s['seed'] for s in unique_seeds]
    seed_coll_map = {s['seed']: s['collection'] for s in unique_seeds}

    # ═══ PART A: keywords_for_keywords (Google Ads) ═══
    seed_batches = list(chunk_list(seed_list, 20))
    ideas_batches = list(chunk_list(seed_list, 20))

    total_calls = len(seed_batches) + len(ideas_batches)
    est = total_calls * 0.075
    if not auto_confirm(est, 'Stage 3: '+str(len(seed_list))+' seeds → '
        +str(len(seed_batches))+' keywords_for_keywords + '
        +str(len(ideas_batches))+' keyword_ideas = '+str(total_calls)+' calls'):
        raise SystemExit('Cancelled')

    dataforseo_kws = []
    existing_kws = set()
    filtered_irrelevant = 0  # v6.1 counter

    # v6.2: track which seed produced each keyword
    kw_seed_source = {}  # keyword -> seed that found it

    def add_kw(kw, vol, cpc, comp, comp_idx, src, seed_batch=None):
        global filtered_irrelevant
        kw = (kw or '').lower().strip()
        if not kw or len(kw) < 4 or kw in existing_kws: return 0
        if is_brand(kw): return 0
        if is_irrelevant(kw):  # v6.1: filter irrelevant
            filtered_irrelevant += 1; return 0
        dataforseo_kws.append({
            'keyword': kw, 'search_volume': vol or 0,
            'cpc': cpc or 0, 'competition': comp or '',
            'competition_index': comp_idx or 0, 'source': src,
        })
        existing_kws.add(kw)
        # v6.2: remember which seed batch found this keyword
        if seed_batch:
            for seed in seed_batch:
                if seed.lower() in kw or kw in seed.lower() or len(set(seed.lower().split()) & set(kw.split())) > 0:
                    kw_seed_source[kw] = seed
                    break
            if kw not in kw_seed_source and seed_batch:
                kw_seed_source[kw] = seed_batch[0]  # default to first seed in batch
        return 1


    # ═══ v6.1: Retry wrapper for DataForSEO API ═══
    def api_post_with_retry(url, payload, max_retries=3, timeout=120):
        """POST with retry on timeout/connection errors. Identifies service via URL host."""
        # Идентификация сервиса для лога: host из URL (api.dataforseo.com → DataForSEO).
        _service = "DataForSEO" if "dataforseo" in url else "API"
        for attempt in range(max_retries):
            try:
                r = requests.post(url, headers=API_HEADERS, json=payload, timeout=timeout)
                if r.status_code == 401:
                    print(f'[{_service} AUTH 401 — check DATAFORSEO_LOGIN/DATAFORSEO_PASSWORD env]', flush=True)
                    return None
                if r.status_code in (402, 403):
                    # 402 = insufficient balance, 403 = forbidden
                    _kind = "balance" if r.status_code == 402 else "forbidden"
                    print(f'[{_service} {r.status_code} ({_kind}): {r.text[:120]}]', flush=True)
                    return None
                return r
            except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as e:
                wait = 15 * (attempt + 1)
                if attempt < max_retries - 1:
                    print(f'[{_service} timeout, retry {attempt+1}/{max_retries} in {wait}s]', end=' ', flush=True)
                    time.sleep(wait)
                else:
                    print(f'[{_service} FAILED after {max_retries} retries: {str(e)[:60]}]', flush=True)
                    return None
        return None

    print('─── Part A: keywords_for_keywords (Google Ads) ───')
    for bi, batch in enumerate(tqdm(seed_batches, desc='keywords_for_keywords')):
        if not cost_tracker.can_afford('keywords_for_keywords'):
            print('Budget limit!'); break
        try:
            r = api_post_with_retry(
                'https://api.dataforseo.com/v3/keywords_data/google_ads/keywords_for_keywords/live',
                [{'keywords': batch, 'location_code': LOCATION_CODE,
                  'language_code': LANGUAGE_CODE, 'sort_by': 'search_volume',
                  'include_adult_keywords': False}])
            if r is None: continue
            cost_tracker.charge('keywords_for_keywords', str(len(batch))+' seeds')
            data = r.json()

            if bi == 0:
                print('\n[DEBUG] status:', data.get('status_code'), data.get('status_message'))
                if data.get('tasks'):
                    t0 = data['tasks'][0]
                    print('[DEBUG] task:', t0.get('status_code'), '| result_count:', t0.get('result_count'))
                    rr = t0.get('result') or []
                    if rr: print('[DEBUG] first:', rr[0].get('keyword','?'), 'vol:', rr[0].get('search_volume','?'))

            if data.get('status_code') == 20000:
                cnt = 0
                for task in data.get('tasks', []):
                    for item in (task.get('result') or []):
                        if not isinstance(item, dict): continue
                        cnt += add_kw(item.get('keyword'), item.get('search_volume'),
                            item.get('cpc'), item.get('competition'),
                            item.get('competition_index'), 'gads', seed_batch=batch)
                if bi < 3: print(f'  Batch {bi+1}: +{cnt}')
        except Exception as e:
            print(f'Error: '+str(e)[:100])
        time.sleep(2)

    print(f'After keywords_for_keywords: {len(dataforseo_kws)} keywords (filtered {filtered_irrelevant} irrelevant)')

    # ═══ PART B: keyword_ideas (DataForSEO Labs) ═══
    print('\n─── Part B: keyword_ideas (DataForSEO Labs SERP DB) ───')
    for bi, batch in enumerate(tqdm(ideas_batches, desc='keyword_ideas')):
        if not cost_tracker.can_afford('keyword_ideas'):
            print('Budget limit!'); break
        try:
            r = api_post_with_retry(
                'https://api.dataforseo.com/v3/dataforseo_labs/google/keyword_ideas/live',
                [{'keywords': batch, 'location_code': LOCATION_CODE,
                  'language_code': LANGUAGE_CODE, 'include_serp_info': False,
                  'limit': 700, 'filters': [['keyword_info.search_volume', '>', MIN_VOLUME - 1]],
                  'order_by': ['keyword_info.search_volume,desc']}])
            if r is None: continue
            cost_tracker.charge('keyword_ideas', str(len(batch))+' seeds')
            data = r.json()

            if bi == 0:
                print('\n[DEBUG] status:', data.get('status_code'), data.get('status_message'))
                if data.get('tasks'):
                    t0 = data['tasks'][0]
                    print('[DEBUG] task:', t0.get('status_code'), '| result_count:', t0.get('result_count'))
                    rr = (t0.get('result') or [None])[0]
                    if rr and isinstance(rr, dict):
                        print('[DEBUG] total_count:', rr.get('total_count'), '| items_count:', rr.get('items_count'))
                        items = rr.get('items') or []
                        if items:
                            item0 = items[0]
                            print('[DEBUG] item keys:', list(item0.keys())[:10])
                            # Try both structures
                            kd0 = item0.get('keyword_data') or item0
                            ki0 = kd0.get('keyword_info') or kd0
                            kw0 = kd0.get('keyword') or item0.get('keyword') or '?'
                            vol0 = ki0.get('search_volume') or item0.get('search_volume') or '?'
                            print('[DEBUG] first:', kw0, 'vol:', vol0)

            if data.get('status_code') == 20000:
                cnt = 0
                for task in data.get('tasks', []):
                    for res in (task.get('result') or []):
                        if not isinstance(res, dict): continue
                        for item in (res.get('items') or []):
                            if not isinstance(item, dict): continue
                            # Handle both structures: {keyword_data:{keyword, keyword_info:{...}}} and {keyword, keyword_info:{...}}
                            kd = item.get('keyword_data') or item
                            if not isinstance(kd, dict): continue
                            ki = kd.get('keyword_info') or kd
                            if not isinstance(ki, dict): continue
                            kw_name = kd.get('keyword') or item.get('keyword')
                            cnt += add_kw(
                                kw_name, ki.get('search_volume'),
                                ki.get('cpc'), ki.get('competition'),
                                ki.get('competition_index'), 'ideas', seed_batch=batch
                            )
                if bi < 3: print(f'  Batch {bi+1}: +{cnt}')
        except Exception as e:
            print(f'Error: '+str(e)[:100])
        time.sleep(0.5)

    print(f'\nTotal REAL keywords: {len(dataforseo_kws)}')
    print(f'Filtered as irrelevant: {filtered_irrelevant}')

    # ═══ Build DataFrame ═══
    df_real = pd.DataFrame(dataforseo_kws)
    if len(df_real) > 0:
        df_real = df_real.drop_duplicates(subset=['keyword'], keep='first')
        df_real = df_real.sort_values('search_volume', ascending=False).reset_index(drop=True)

    print('\n' + '='*50)
    print('  STAGE 3 RESULTS')
    print('='*50)
    print('Total REAL keywords: '+str(len(df_real)))
    if len(df_real) > 0:
        n_gads = len(df_real[df_real['source']=='gads'])
        n_ideas = len(df_real[df_real['source']=='ideas'])
        print(f'  From keywords_for_keywords: {n_gads}')
        print(f'  From keyword_ideas:         {n_ideas}')
        print(f'  With volume > 0:            {len(df_real[df_real["search_volume"]>0])}')
        print(f'  With volume >= {MIN_VOLUME}:          {len(df_real[df_real["search_volume"]>=MIN_VOLUME])}')
        print(f'  Filtered irrelevant:        {filtered_irrelevant}')
        print('\nTop 25 keywords:')
        print(df_real[['keyword','search_volume','cpc','competition','source']].head(25).to_string(index=False))
    cost_tracker.summary()

    # Build DataFrame
    df_real = pd.DataFrame(dataforseo_kws)
    if len(df_real) > 0:
        df_real['keyword'] = df_real['keyword'].str.lower().str.strip()
        df_real = df_real.drop_duplicates(subset=['keyword'], keep='first')
    else:
        print('WARNING: DataForSEO returned 0 keywords!')
        df_real = pd.DataFrame(columns=['keyword','search_volume','cpc','competition','competition_index','source'])
    print(f'df_real: {len(df_real)} keywords')


    import os, warnings
    os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
    os.environ['TOKENIZERS_PARALLELISM'] = 'false'
    warnings.filterwarnings('ignore')

    import torch, faiss, numpy as np, time
    from sentence_transformers import SentenceTransformer

    t0 = time.time()
    print('STAGE 4: Seed-based keyword mapping + product ranking')

    # ═══ GPU ═══
    USE_GPU = torch.cuda.is_available()
    DEVICE = 'cuda' if USE_GPU else 'cpu'
    EMBED_BATCH = 512 if USE_GPU else 128
    if USE_GPU:
        gpu = torch.cuda.get_device_properties(0)
        print(f'  GPU: {gpu.name} ({gpu.total_memory/1e9:.1f} GB)')

    # ═══ df_all ═══
    df_all = df_real.copy()
    df_all = df_all[df_all['keyword'].notna() & (df_all['keyword'] != '')].reset_index(drop=True)

    def smart_kd_quick(row):
        ci = row.get('competition_index', 50) or 50
        vol = row.get('search_volume', 0) or 0
        cpc = row.get('cpc', 0) or 0
        kd = ci * 0.5
        if vol > 100000: kd += 15
        elif vol > 10000: kd += 8
        elif vol > 1000: kd += 3
        if cpc > 5: kd += 10
        elif cpc > 2: kd += 5
        return min(max(int(kd), 0), 100)

    if len(df_all) == 0:
        print('\n⚠ No keywords from DataForSEO — will use product names as basic keywords')
        # Create minimal df_all so remaining code doesn't crash
        basic_kws = []
        for product in PRODUCTS:
            coll = product_coll_map.get(product, 'Uncategorized')
            # Use product name words as basic keywords
            words = product.lower().split()[:4]
            kw = ' '.join(words)
            basic_kws.append({'keyword': kw, 'search_volume': 100, 'cpc': 0.5,
                'competition': 'LOW', 'competition_index': 10, 'source': 'fallback',
                'keyword_difficulty': 5})
        df_all = pd.DataFrame(basic_kws)
        print(f'  Created {len(basic_kws)} fallback keywords from product names')

    if 'keyword_difficulty' not in df_all.columns:
        df_all['keyword_difficulty'] = df_all.apply(smart_kd_quick, axis=1)
    MIN_VOLUME = 100
    vol_map = dict(zip(df_all['keyword'], df_all['search_volume']))
    print(f'  Total keywords: {len(df_all):,}')

    # ═══ STEP 1: Build keyword → collection from seed lineage ═══
    # kw_seed_source: keyword → seed (from Cell 17)
    # seed_coll_map: seed → collection (from Cell 17)
    # Chain: keyword → seed → collection

    kw_to_collection_via_seed = {}  # keyword → collection name
    collection_keywords = {}  # collection → set of keywords

    mapped = 0
    unmapped = 0
    for kw, seed in kw_seed_source.items():
        coll = seed_coll_map.get(seed)
        if coll:
            kw_to_collection_via_seed[kw] = coll
            collection_keywords.setdefault(coll, set()).add(kw)
            mapped += 1
        else:
            unmapped += 1

    print(f'  Seed lineage: {mapped:,} keywords mapped, {unmapped:,} unmapped')
    print(f'  Collections from seeds: {len(collection_keywords)}')

    # Show coverage
    for coll in sorted(collection_keywords, key=lambda c: len(collection_keywords[c]), reverse=True)[:10]:
        n = len(collection_keywords[coll])
        vol_sum = sum(vol_map.get(kw, 0) for kw in collection_keywords[coll])
        print(f'    {n:>5} kws | {vol_sum:>10,} vol | {coll}')

    # ═══ STEP 2: Rank keywords per product using FAISS ═══
    print(f'\n  Loading embedding model...')
    embed_model = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)

    # Embed products
    prod_embs = embed_model.encode(PRODUCTS, show_progress_bar=False,
                                    batch_size=EMBED_BATCH, normalize_embeddings=True)
    prod_embs = np.array(prod_embs).astype('float32')
    d = prod_embs.shape[1]

    validated_product_kws = {}
    products_assigned = 0

    for product in PRODUCTS:
        coll = product_coll_map.get(product, 'Uncategorized')
        # Get all keywords in this product's collection (from seed lineage)
        coll_kws = list(collection_keywords.get(coll, set()))

        if not coll_kws:
            validated_product_kws[product] = []
            continue

        # Filter to keywords with volume
        coll_kws_vol = [(kw, vol_map.get(kw, 0)) for kw in coll_kws if vol_map.get(kw, 0) >= MIN_VOLUME]
        if not coll_kws_vol:
            validated_product_kws[product] = []
            continue

        # Embed collection keywords
        kw_texts = [kw for kw, _ in coll_kws_vol]
        kw_embs = embed_model.encode(kw_texts, show_progress_bar=False,
                                      batch_size=EMBED_BATCH, normalize_embeddings=True)
        kw_embs = np.array(kw_embs).astype('float32')

        # FAISS: rank by similarity to THIS product
        pi = PRODUCTS.index(product)
        prod_emb = prod_embs[pi:pi+1]

        kw_idx = faiss.IndexFlatIP(d)
        kw_idx.add(kw_embs)
        n_results = min(30, len(kw_texts))
        scores, indices = kw_idx.search(prod_emb, n_results)

        # Take top ranked keywords
        product_kws = []
        for j in range(n_results):
            idx = indices[0][j]
            if 0 <= idx < len(kw_texts):
                product_kws.append(kw_texts[idx])

        validated_product_kws[product] = product_kws[:25]
        if product_kws:
            products_assigned += 1
            # v9.3: Save SEO keywords to DB
            kw_str = ' | '.join(product_kws[:15])
            if kw_str:
                db.execute('UPDATE products SET seo_keywords=? WHERE name=?', (kw_str, product))

    print(f'\n  Products with keywords: {products_assigned}/{len(PRODUCTS)}')
    total_pairs = sum(len(v) for v in validated_product_kws.values())
    print(f'  Total product-keyword pairs: {total_pairs:,}')

    # ═══ STEP 3: Build kw_to_collections (ONLY from seed lineage) ═══
    kw_to_collections = {}
    for kw, coll in kw_to_collection_via_seed.items():
        kw_to_collections[kw] = {coll}

    # final_product_colls from Stage 1 (CLEAN)
    final_product_colls = {}
    for product in PRODUCTS:
        coll = product_coll_map.get(product, 'Uncategorized')
        final_product_colls[product] = [coll]

    all_validated = set()
    for kws in validated_product_kws.values():
        all_validated.update(kws)

    # ═══ Samples ═══
    print(f'\n  Keywords with collection: {len(kw_to_collections):,}')
    print(f'\n  Samples:')
    for p in PRODUCTS[:8]:
        kws = validated_product_kws.get(p, [])
        coll = product_coll_map.get(p, '?')
        kw_sample = ', '.join(kws[:5])
        print(f'    {p[:50]}')
        print(f'      [{coll}] {len(kws)} kws: {kw_sample}')

    print(f'\n  Stage 4: {time.time()-t0:.1f}s')


    print('STAGE 5: Skipped (collections from Stage 1, no clustering)')
    # Build final_coll_products from Stage 1 categorization
    final_coll_products = {}
    for p, colls in final_product_colls.items():
        for c in colls:
            final_coll_products.setdefault(c, []).append(p)

    # Remove Uncategorized if empty or useless
    if 'Uncategorized' in final_coll_products:
        if len(final_coll_products['Uncategorized']) == 0:
            del final_coll_products['Uncategorized']

    # No irrelevant clusters in this approach
    irrelevant_clusters = set()

    print(f'  Collections: {len(final_coll_products)}')
    for c in sorted(final_coll_products, key=lambda x: len(final_coll_products[x]), reverse=True)[:15]:
        print(f'    {len(final_coll_products[c]):>3} products | {c}')


    # ═══ Stage 6: Get real KD + Expand top winners ═══
    print('STAGE 6: Real KD + Expansion')
    t_start = time.time()

    # df_all already created in Stage 4 with smart_kd estimates
    # Now get REAL KD from DataForSEO (more accurate)
    vol_map = dict(zip(df_all['keyword'], df_all['search_volume']))
    comp_map = dict(zip(df_all['keyword'], df_all['competition']))
    ci_map = dict(zip(df_all['keyword'], df_all['competition_index']))
    cpc_map = dict(zip(df_all['keyword'], df_all['cpc']))

    has_vol = df_all[df_all['search_volume'] >= MIN_VOLUME]['keyword'].tolist()
    kd_map = {}
    if has_vol:
        print('Getting KD for '+str(len(has_vol))+' keywords...')
        for chunk in [has_vol[i:i+1000] for i in range(0, len(has_vol), 1000)]:
            try:
                r = api_post_with_retry('https://api.dataforseo.com/v3/dataforseo_labs/google/bulk_keyword_difficulty/live',
                    [{'keywords': chunk, 'location_code': LOCATION_CODE, 'language_code': LANGUAGE_CODE}])
                if r is None: continue
                cost_tracker.charge('bulk_keyword_difficulty', str(len(chunk))+' kws')
                data = r.json()
                if data.get('status_code') == 20000:
                    for task in data.get('tasks', []):
                        for res in (task.get('result') or []):
                            if not res or not isinstance(res, dict): continue
                            for item in (res.get('items') or []):
                                if not item or not isinstance(item, dict): continue
                                kw = (item.get('keyword','') or '').lower()
                                kd = item.get('keyword_difficulty')
                                if kw and kd and kd > 0: kd_map[kw] = kd
            except: pass
            time.sleep(1)
        print('Bulk KD > 0: '+str(len(kd_map)))

    def smart_kd(row):
        kw = row.get('keyword', ''); bulk = kd_map.get(kw, 0)
        if bulk > 0: return bulk
        comp = row.get('competition', '') or ''; ci = row.get('competition_index', 0) or 0; vol = row.get('search_volume', 0) or 0
        if comp == 'HIGH':
            if vol >= 50000: return max(50, int(ci*0.8))
            if vol >= 10000: return max(35, int(ci*0.6))
            return max(25, int(ci*0.5))
        elif comp == 'MEDIUM': return max(15, int(ci*0.4))
        elif comp == 'LOW': return max(5, int(ci*0.2))
        if vol >= 10000: return 25
        if vol >= 1000: return 15
        return 10

    df_all['keyword_difficulty'] = df_all.apply(smart_kd, axis=1)
    winners = df_all[(df_all['search_volume'] >= MIN_VOLUME) & (df_all['keyword_difficulty'] <= MAX_KD)].sort_values('search_volume', ascending=False)
    print('\nWith volume >= '+str(MIN_VOLUME)+': '+str(len(has_vol)))
    print('Winners (KD<='+str(MAX_KD)+'): '+str(len(winners)))
    if len(winners) > 0:
        print(winners[['keyword','search_volume','keyword_difficulty','cpc','competition']].head(20).to_string(index=False))

    # ═══ Expand ═══
    def dedup_seeds(kl, mx):
        s = []
        for kw in kl:
            d = False
            for ex in s:
                if kw in ex or ex in kw: d = True; break
                w1, w2 = set(kw.split()), set(ex.split())
                if len(w1&w2) >= 2 and len(w1&w2)/max(len(w1), len(w2)) > 0.6: d = True; break
            if not d: s.append(kw)
            if len(s) >= mx: break
        return s

    sp = []
    if len(winners) > 0:
        sp.extend(winners.head(STAGE_EXPAND_TOP*2)['keyword'].tolist())
    seeds = dedup_seeds(list(dict.fromkeys(sp)), STAGE_EXPAND_TOP)
    est = len(seeds) * 2 * 0.075

    expanded_kws = []
    expanded_irr = 0
    if seeds and auto_confirm(est, 'Expand: '+str(len(seeds))+' seeds x 2 endpoints'):
        existing = set(df_all['keyword'].tolist())
        for seed in tqdm(seeds, desc='Expanding'):
            for ep in ['keyword_suggestions', 'related_keywords']:
                if not cost_tracker.can_afford(ep): break
                try:
                    fp = 'keyword_info' if ep == 'keyword_suggestions' else 'keyword_data.keyword_info'
                    r = api_post_with_retry('https://api.dataforseo.com/v3/dataforseo_labs/google/'+ep+'/live',
                        [{'keyword': seed, 'location_code': LOCATION_CODE, 'language_code': LANGUAGE_CODE,
                        'include_seed_keyword': True, 'limit': 80,
                        'filters': [[fp+'.search_volume', '>', MIN_VOLUME-1]],
                        'order_by': [fp+'.search_volume,desc']}], timeout=60)
                    if r is None: continue
                    cost_tracker.charge(ep, seed)
                    data = r.json()
                    if data.get('status_code') == 20000:
                        for task in data.get('tasks', []):
                            for res in (task.get('result') or []):
                                if not res or not isinstance(res, dict): continue
                                for item in (res.get('items') or []):
                                    if not item or not isinstance(item, dict): continue
                                    kd = item.get('keyword_data') or {}
                                    if not isinstance(kd, dict): continue
                                    ki = kd.get('keyword_info') or {}
                                    kw = (kd.get('keyword','') or '').lower().strip()
                                    if not kw or kw in existing: continue
                                    if is_brand(kw) or is_irrelevant(kw):
                                        expanded_irr += 1; continue
                                    expanded_kws.append({'keyword': kw, 'search_volume': ki.get('search_volume',0) or 0,
                                        'keyword_difficulty': ki.get('keyword_difficulty',0) or 0,
                                        'cpc': ki.get('cpc',0) or 0, 'competition': ki.get('competition','') or '', 'source': ep})
                                    existing.add(kw)
                except: pass
            time.sleep(0.5)
        print('Expanded: +'+str(len(expanded_kws))+' (filtered '+str(expanded_irr)+' irrelevant/brand)')
    cost_tracker.summary()


    print('STAGE 7: Merge expanded keywords')
    t_start = time.time()

    # Merge expanded into df_all (for Winners/All Keywords sheets)
    if expanded_kws:
        df_exp = pd.DataFrame(expanded_kws)
        df_all = pd.concat([df_all, df_exp], ignore_index=True)
        df_all['keyword'] = df_all['keyword'].str.lower().str.strip()
        df_all = df_all.sort_values('search_volume', ascending=False).drop_duplicates(subset=['keyword'], keep='first')
        df_all = df_all[df_all['keyword'].notna() & (df_all['keyword'] != '')]
        if df_all['keyword_difficulty'].isna().any():
            df_all['keyword_difficulty'] = df_all.apply(smart_kd, axis=1)
        print(f'  df_all: {len(df_all):,} keywords')

        # Add expanded keywords to kw_to_collections via seed lineage ONLY
        # (expanded keywords don't have seed lineage — they came from winner seeds)
        # So we assign them to collection IF their expansion seed was from that collection
        # We DON'T add them to product keyword lists (to keep products clean)
        print(f'  Expanded keywords added to df_all only (product lists unchanged)')

    # Update vol_map
    vol_map = dict(zip(df_all['keyword'], df_all['search_volume']))

    total_kw = sum(len(v) for v in validated_product_kws.values())
    print(f'  Product-keyword pairs: {total_kw:,}')
    print(f'  Stage 7: {time.time()-t_start:.1f}s')


    print('STAGE 8: Collection assembly (using Stage 1 categories)')
    t_start = time.time()

    # Per product: top keywords sorted by volume
    product_top_kws = {}
    for product in PRODUCTS:
        kws = validated_product_kws.get(product, [])
        kws_with_vol = [(k, vol_map.get(k, 0)) for k in kws if vol_map.get(k, 0) >= MIN_VOLUME]
        kws_with_vol.sort(key=lambda x: x[1], reverse=True)
        product_top_kws[product] = kws_with_vol[:10]

    # final_product_colls already set in Stage 4 (from Stage 1 clean categories)
    # final_coll_products already built in Stage 5

    # Rebuild kw_to_collections to ensure consistency
    kw_to_collections = {}
    for product, kws in validated_product_kws.items():
        coll = product_coll_map.get(product, 'Uncategorized')
        for kw in kws:
            kw_to_collections.setdefault(kw, set()).add(coll)

    print(f'  Collections: {len(final_coll_products)}')
    print(f'  Products: {len(PRODUCTS)}')
    print(f'  Keywords with collection: {len(kw_to_collections):,}')
    total_matched = sum(len(v) for v in validated_product_kws.values())
    print(f'  Product-keyword pairs: {total_matched:,}')

    # Show collection summary
    for c in sorted(final_coll_products, key=lambda x: len(final_coll_products[x]), reverse=True)[:15]:
        n_prods = len(final_coll_products[c])
        coll_kw_count = sum(len(validated_product_kws.get(p, [])) for p in final_coll_products[c])
        print(f'    {n_prods:>3} prods | {coll_kw_count:>4} kws | {c}')

    print(f'  Stage 8: {time.time()-t_start:.1f}s')


    print('STAGE 9: SEO titles & meta')
    t_start = time.time()

    # Build collection_info from product keywords
    collection_info = {}
    for coll, prods in final_coll_products.items():
        coll_kws = set()
        for p in prods:
            coll_kws.update(validated_product_kws.get(p, []))

        kw_df = df_all[(df_all['keyword'].isin(coll_kws)) & (df_all['search_volume'] >= MIN_VOLUME)]
        # NO word-matching fallback — only use validated product keywords
        # (word-matching caused "clown makeup" in "Makeup" collection)
        top_kws = kw_df.nlargest(10, 'search_volume')['keyword'].tolist() if len(kw_df) > 0 else []
        collection_info[coll] = {
            'keywords': top_kws,
            'total_volume': int(kw_df['search_volume'].sum()) if len(kw_df) > 0 else 0,
            'n_keywords': len(kw_df),
            'n_products': len(prods),
        }

    # Fill collection column in df_all
    print('Mapping keywords to collections...')
    def get_kw_collection(kw):
        colls = kw_to_collections.get(kw, set())
        real_colls = {c for c in colls if c != 'IRRELEVANT'}
        if real_colls:
            return ' | '.join(sorted(real_colls)[:3])
        return ''

    df_all['collection'] = df_all['keyword'].apply(get_kw_collection)
    filled = (df_all['collection'] != '').sum()
    print(f'  Keywords with collection: {filled}/{len(df_all)} ({filled/max(len(df_all),1)*100:.1f}%)')

    # Generate SEO titles via Claude
    print('\nGenerating SEO titles...')
    seo_collections = {}
    coll_names = list(collection_info.keys())
    for bi in range(0, len(coll_names), 15):
        batch = coll_names[bi:bi+15]
        batch_data = ''
        for c in batch:
            info = collection_info[c]
            batch_data += '\nCollection: ' + c + ' (' + str(info['n_products']) + ' products)'
            if info['keywords']: batch_data += '\nTop kw: ' + ', '.join(info['keywords'][:8])
            batch_data += '\nVolume: ' + str(info['total_volume']) + '\n---'
        prompt = ('Shopify SEO. For each collection create:\n'
            '1. t: SEO H1 title (50-70 chars) with top keywords naturally\n'
            '2. h: URL slug (lowercase-hyphens)\n'
            '3. mt: Meta title (55-60 chars), primary keyword first\n'
            '4. md: Meta description (140-155 chars) with call to action\n'
            'DATA:\n' + batch_data + '\n'
            'JSON: [{"o":"original","t":"title","h":"handle","mt":"meta","md":"desc"}]')
        try:
            r = call_claude(prompt, 0.7)
            for item in r:
                if isinstance(item, dict) and item.get('o'):
                    seo_collections[item['o']] = {
                        'seo_title': item.get('t', ''),
                        'seo_handle': item.get('h', ''),
                        'meta_title': item.get('mt', ''),
                        'meta_description': item.get('md', '')
                    }
            print(f'  Batch: +{len(r)}', flush=True)
        except Exception as e:
            print(f'  error: {str(e)[:50]}', flush=True)
        time.sleep(3)

    for c in coll_names:
        if c not in seo_collections:
            seo_collections[c] = {
                'seo_title': c,
                'seo_handle': c.lower().replace(' ', '-').replace("'", ""),
                'meta_title': c,
                'meta_description': 'Shop our ' + c + ' collection.'
            }
    print(f'SEO data for {len(seo_collections)} collections')
    print(f'Stage 9: {time.time()-t_start:.1f}s')


    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill
    from google.colab import files as colab_files

    def get_seo(orig):
        if orig in seo_collections: return seo_collections[orig].get('seo_title', orig)
        return orig

    # ═══ v6.1: Map collection column using seo_collections ═══
    def get_seo_collection(kw):
        colls = kw_to_collections.get(kw, set())
        real_colls = {c for c in colls if c != 'IRRELEVANT'}
        if real_colls:
            seo = [get_seo(c) for c in sorted(real_colls)[:3]]
            return ' | '.join(seo)
        return ''

    df_all['seo_collection'] = df_all['keyword'].apply(get_seo_collection)
    # v6.2: exclude keywords marked IRRELEVANT by Claude
    irrelevant_kws = {kw for kw, colls in kw_to_collections.items() if colls == {'IRRELEVANT'}}
    print(f'Excluding {len(irrelevant_kws)} IRRELEVANT keywords from winners')
    df_all = df_all[~df_all['keyword'].isin(irrelevant_kws)]


    df_win = df_all[(df_all['keyword_difficulty']<=MAX_KD)&(df_all['search_volume']>=MIN_VOLUME)].sort_values('search_volume',ascending=False).reset_index(drop=True)

    # Stats
    win_with_coll = (df_win['seo_collection'] != '').sum()
    print(f'Winners: {len(df_win)} | with collection: {win_with_coll} ({win_with_coll/max(len(df_win),1)*100:.1f}%)')

    # Build lookup sets once (lowercase)
    all_kw_lower = set(df_all['keyword'].str.lower().tolist())
    win_kw_lower = set(df_win['keyword'].str.lower().tolist()) if len(df_win) > 0 else set()

    # DEBUG: Check validated_product_kws health
    total_vpk = sum(len(v) for v in validated_product_kws.values())
    empty_vpk = sum(1 for v in validated_product_kws.values() if not v)
    print(f'DEBUG: validated_product_kws: {total_vpk} total pairs, {empty_vpk} empty products')
    if total_vpk > 0:
        sample_p = [p for p in PRODUCTS if validated_product_kws.get(p, [])][:3]
        for p in sample_p:
            kws = validated_product_kws[p][:3]
            in_all = sum(1 for k in kws if k.lower() in all_kw_lower)
            in_win = sum(1 for k in kws if k.lower() in win_kw_lower)
            print(f'  {p[:40]}: {len(validated_product_kws[p])} kws, {in_all}/3 in df_all, {in_win}/3 in winners')
            print(f'    Sample: {kws}')

    product_rows = []
    for product in PRODUCTS:
        valid_kws = validated_product_kws.get(product, [])

        # Sort by volume (highest first), keep all
        kws_sorted = sorted(valid_kws, key=lambda k: vol_map.get(k, 0), reverse=True)

        colls = final_product_colls.get(product, ['Uncategorized'])
        seo_colls = [get_seo(c) for c in colls]
        product_rows.append({
            'product': product,
            'primary_collection': seo_colls[0] if seo_colls else 'Uncategorized',
            'all_collections': ' | '.join(seo_colls),
            'keywords': ' | '.join(kws_sorted[:15]),
            'keyword_count': len(kws_sorted),
        })
    df_products = pd.DataFrame(product_rows)

    # Stats
    kw_filled = sum(1 for r in product_rows if r['keyword_count'] > 0)
    print(f'Products with keywords: {kw_filled}/{len(PRODUCTS)}')
    if kw_filled < len(PRODUCTS):
        empty_prods = [r['product'][:50] for r in product_rows if r['keyword_count'] == 0]
        print(f'  Empty: {", ".join(empty_prods[:5])}')

    safe_cat = CATEGORY.replace(' & ','_').replace(' ','_')
    out = f'{safe_cat}_SEO_Keywords.xlsx'
    wb = Workbook()
    hf=Font(bold=True,size=11,color='FFFFFF')
    hb=PatternFill('solid',fgColor='1F2937')
    gf=PatternFill('solid',fgColor='D1FAE5')
    yf=PatternFill('solid',fgColor='FEF3C7')
    rf=PatternFill('solid',fgColor='FEE2E2')

    # Sheet 1: Product Keywords
    ws1=wb.active; ws1.title='Product Keywords'
    for ci,(h,w) in enumerate([('Product',55),('Primary Collection',40),('All Collections',60),('Keywords',100),('# KW',8)],1):
        c=ws1.cell(1,ci,value=h); c.font=hf; c.fill=hb
    ws1.column_dimensions['A'].width=55; ws1.column_dimensions['B'].width=40
    ws1.column_dimensions['C'].width=60; ws1.column_dimensions['D'].width=100; ws1.column_dimensions['E'].width=8
    for idx,row in df_products.iterrows():
        r=idx+2
        ws1.cell(r,1,value=row['product']); ws1.cell(r,2,value=row['primary_collection'])
        ws1.cell(r,3,value=row['all_collections']); ws1.cell(r,4,value=row['keywords']); ws1.cell(r,5,value=row['keyword_count'])
    ws1.freeze_panes='A2'; ws1.auto_filter.ref='A1:E'+str(len(df_products)+1)

    # Sheet 2: Shopify Collections
    ws_c=wb.create_sheet('Shopify Collections')
    ch=['SEO Title (H1)','URL Handle','Meta Title','Meta Description','Original','# Products','# KW','Total Volume','Top Keywords']
    for ci,h in enumerate(ch,1): c=ws_c.cell(1,ci,value=h); c.font=hf; c.fill=hb
    ws_c.column_dimensions['A'].width=55; ws_c.column_dimensions['B'].width=35
    ws_c.column_dimensions['C'].width=55; ws_c.column_dimensions['D'].width=70; ws_c.column_dimensions['I'].width=80
    ri=2
    for orig in sorted(collection_info,key=lambda x:collection_info[x]['n_products'],reverse=True):
        info=collection_info[orig]
        data=seo_collections.get(orig,{})
        ws_c.cell(ri,1,value=data.get('seo_title',orig)); ws_c.cell(ri,2,value=data.get('seo_handle',''))
        ws_c.cell(ri,3,value=data.get('meta_title','')); ws_c.cell(ri,4,value=data.get('meta_description',''))
        ws_c.cell(ri,5,value=orig); ws_c.cell(ri,6,value=info['n_products'])
        ws_c.cell(ri,7,value=info['n_keywords']); ws_c.cell(ri,8,value=info['total_volume'])
        ws_c.cell(ri,9,value=', '.join(info['keywords'][:8]))
        ri+=1
    ws_c.freeze_panes='A2'

    # Sheet 3: Winners
    ws2=wb.create_sheet('Winners KD<'+str(MAX_KD))
    wc=[('Keyword',50),('Collection',35),('Volume',12),('KD',8),('CPC',10),('Competition',14)]
    # Ensure seo_collection column exists
    if 'seo_collection' not in df_all.columns:
        if 'collection' in df_all.columns:
            df_all['seo_collection'] = df_all['collection']
        else:
            df_all['seo_collection'] = df_all['keyword'].apply(get_seo_collection)
    wk=['keyword','seo_collection','search_volume','keyword_difficulty','cpc','competition']
    for ci,(h,w) in enumerate(wc,1): c=ws2.cell(1,ci,value=h); c.font=hf; c.fill=hb
    ws2.column_dimensions['A'].width=50; ws2.column_dimensions['B'].width=35
    for idx,row in df_win.iterrows():
        r=idx+2
        for ci,key in enumerate(wk,1): v=row.get(key,''); ws2.cell(r,ci,value='' if pd.isna(v) else v)
        kd=row.get('keyword_difficulty',0) or 0
        kc=ws2.cell(r,4)
        if kd<=10: kc.fill=gf
        elif kd<=20: kc.fill=yf
        elif kd<=30: kc.fill=rf
    ws2.freeze_panes='A2'; ws2.auto_filter.ref='A1:F'+str(len(df_win)+1)

    # Sheet 4: All Keywords
    ws3=wb.create_sheet('All Keywords')
    for ci,(h,w) in enumerate(wc,1): c=ws3.cell(1,ci,value=h); c.font=hf; c.fill=hb
    ws3.column_dimensions['A'].width=50; ws3.column_dimensions['B'].width=35
    for idx,row in df_all.iterrows():
        r=idx+2
        for ci,key in enumerate(wk,1): v=row.get(key,''); ws3.cell(r,ci,value='' if pd.isna(v) else v)
    ws3.freeze_panes='A2'

    # Sheet 5: Cost
    ws5=wb.create_sheet('Cost Log')
    for ci,h in enumerate(['Endpoint','Cost','Details'],1): c=ws5.cell(1,ci,value=h); c.font=hf; c.fill=hb
    for i,(ep,cost,det) in enumerate(cost_tracker.log,2):
        ws5.cell(i,1,value=ep); ws5.cell(i,2,value=round(cost,4)); ws5.cell(i,3,value=det)
    tr=len(cost_tracker.log)+3
    ws5.cell(tr,1,value='TOTAL').font=Font(bold=True)
    ws5.cell(tr,2,value=round(cost_tracker.spent,4)).font=Font(bold=True)

    wb.save(out)
    print('\n'+'='*60)
    print('  SEO Ultimate v6.3 Results')
    print('='*60)
    print('  Total keywords:     '+str(len(df_all)))
    print('  With volume:        '+str(len(df_all[df_all["search_volume"]>0])))
    print('  Winners (KD<'+str(MAX_KD)+'):  '+str(len(df_win)))
    print('  Winners w/coll:     '+str(win_with_coll)+' ('+str(round(win_with_coll/max(len(df_win),1)*100,1))+'%)')
    print('  KW matched to prod: '+str(len(kw_to_collections)))
    print('  Collections:        '+str(ri-2))
    print('  Products:           '+str(len(df_products)))
    print('  COST:               $'+str(round(cost_tracker.spent,2)))
    print('='*60)
    shutil.copy(out, f'{PROJECT_DIR}/{out}')
    colab_files.download(out)

    # Save to DB
    for orig, data in seo_collections.items():
        h = data.get('seo_handle', orig.lower().replace(' ','-'))
        try: db.execute("INSERT INTO collections (original_name,seo_title,seo_handle,full_url,meta_title,meta_description) VALUES (?,?,?,?,?,?)",
            (orig, data.get('seo_title',''), h, f'/collections/{h}', data.get('meta_title',''), data.get('meta_description','')))
        except: pass
    for pn in PRODUCTS:
        pr = db.execute('SELECT id FROM products WHERE name=?', (pn,)).fetchone()
        if not pr: continue
        for cn in final_product_colls.get(pn, []):
            cr = db.execute('SELECT id FROM collections WHERE original_name=?', (cn,)).fetchone()
            if cr:
                try: db.execute('INSERT OR IGNORE INTO product_collections VALUES (?,?)', (pr['id'], cr['id']))
                except: pass
    db.execute("INSERT OR REPLACE INTO seo_state (key,value) VALUES ('seo_complete','1')")
    db.commit()
    print(f'Saved {db.execute("SELECT COUNT(*) FROM collections").fetchone()[0]} collections to DB')

    print("SEO COMPLETE!")


SEO: SKIPPED (data already in DB)
  Collections: 20


## Cell 4 — Collections → Shopify

In [4]:
# ════ Collections → Shopify ════


import shutil

# ── Auto-check: load Excel if no collections in DB ──
n_colls = db.execute("SELECT COUNT(*) FROM collections").fetchone()[0]
if n_colls == 0:
    print("⚠ No collections in DB! Loading from Excel...")
    from google.colab import files as colab_files
    import openpyxl
    print("Upload SEO Excel (.xlsx):")
    up = colab_files.upload()
    seo_f = list(up.keys())[0]
    wb = openpyxl.load_workbook(seo_f, read_only=True)
    cc = 0
    for row in wb['Shopify Collections'].iter_rows(min_row=2, values_only=True):
        st, h, mt, md_val, orig = (row[0] or ''), (row[1] or ''), (row[2] or ''), (row[3] or ''), (row[4] or '')
        try:
            db.execute("INSERT INTO collections (original_name,seo_title,seo_handle,full_url,meta_title,meta_description) VALUES (?,?,?,?,?,?)",
                (orig, st, h, f'/collections/{h}' if h else '', mt, md_val))
            cc += 1
        except: pass
    lc = 0
    for row in wb['Product Keywords'].iter_rows(min_row=2, values_only=True):
        pn, ct = (row[0] or ''), (row[1] or '')
        if not pn or not ct: continue
        pr = db.execute('SELECT id FROM products WHERE name=?', (pn,)).fetchone()
        cr = db.execute('SELECT id FROM collections WHERE seo_title=?', (ct,)).fetchone()
        if pr and cr:
            try: db.execute('INSERT OR IGNORE INTO product_collections VALUES (?,?)', (pr['id'], cr['id'])); lc += 1
            except: pass
    db.commit(); wb.close()
    print(f'{cc} collections, {lc} product links loaded')

# ── Link to Shopify ──
colls = db.execute("SELECT * FROM collections WHERE shopify_collection_id IS NULL").fetchall()
total_colls = db.execute("SELECT COUNT(*) FROM collections").fetchone()[0]

if not colls:
    print(f'All {total_colls} collections linked to Shopify.')
else:
    print(f'Linking {len(colls)} collections to Shopify...')
    found = 0; created = 0
    for c in colls:
        handle = c['seo_handle']
        title = c['seo_title']
        r = await shopify_gql('query($h:String!){collectionByHandle(handle:$h){id title}}', {"h": handle})
        coll = r.get('data',{}).get('collectionByHandle') if r else None
        if coll:
            db.execute('UPDATE collections SET shopify_collection_id=? WHERE id=?', (coll['id'], c['id']))
            db.commit(); found += 1
            print(f'  FOUND: {title[:45]}')
        else:
            cid = await shopify_create_collection(title, handle, c['meta_title'], c['meta_description'])
            if cid:
                db.execute('UPDATE collections SET shopify_collection_id=? WHERE id=?', (cid, c['id']))
                db.commit(); created += 1
                print(f'  CREATED: {title[:45]}')
            else:
                print(f'  FAIL: {title[:45]}')
        await asyncio.sleep(0.3)
    linked = db.execute("SELECT COUNT(*) FROM collections WHERE shopify_collection_id IS NOT NULL").fetchone()[0]
    print(f'\nResult: {linked}/{total_colls} (found {found}, created {created})')

# Verify product-collection links
pc = db.execute("SELECT COUNT(*) FROM product_collections").fetchone()[0]
print(f'Product-collection links: {pc}')
shutil.copy(DB_LOCAL, DB_PATH)


Linking 20 collections to Shopify...
  FOUND: Body Scrubbers – Exfoliating Gloves & Dry Ski
  FOUND: Handmade Moisturising Soap – Nourishing Hand 
  FOUND: Baby Bath Essentials – Gentle Shampoo & Showe
  FOUND: Silicone Body Brushes – Gentle Face & Body Sc
  FOUND: Hair Drying Caps – Quick Dry Microfiber Towel
  FOUND: Natural Body Scrubs – Honey, Almond & Strawbe
  FOUND: Scalp Brushes – Shampoo Scrubber for a Deep C
  FOUND: Premium Bath Towels – Luxury Spa-Quality Show
  FOUND: Bath Bombs – Fizzy Bubble Bath Products for R
  FOUND: Baby Oral Care – Soft Bristle Infant & Newbor
  FOUND: Electric Toothbrushes – Sonic Powered for a D
  FOUND: Satin Hair Bonnets – Sleep Caps for Natural &
  FOUND: Shower Foot Scrubbers – Brushes & Cleaners fo
  FOUND: Pedicure Tools – Professional Callus Removers
  FOUND: Eyelash Tools – Tweezers, Curlers & Lash Appl
  FOUND: Anti Vibration Pads for Washer & Washing Mach
  FOUND: Spa Headbands & Bandana Headbands for Skincar
  FOUND: Odor Removers for L

'/content/drive/MyDrive/shopify_pipeline/pipeline.db'

## Cell 5 — CSS Framework (v9)

In [5]:
# ════ v9 CSS FRAMEWORK ════
# Fixed CSS that gets injected into every product page.
# Claude generates semantic HTML blocks — the CSS handles all styling.

WA_CSS_FRAMEWORK = """
/* ═══ WANELO v9 — inherits theme typography ═══ */
.wa-page{max-width:100%;margin:0 auto;padding:0}
.wa-page img{max-width:720px;height:auto;border-radius:12px;display:block;margin:1.5rem auto}
.wa-page h2{font-size:1.8em;margin:2rem 0 0.75rem}
.wa-page h3{font-size:1.4em;margin:1.5rem 0 0.75rem}

/* Grid */
.wa-grid{display:grid;gap:1.5rem}
.wa-grid-2{grid-template-columns:repeat(2,1fr)}
.wa-grid-3{grid-template-columns:repeat(3,1fr)}
@media(max-width:640px){.wa-grid-2,.wa-grid-3{grid-template-columns:1fr}}

/* Section */
.wa-section{margin:2.5rem 0;padding:0}
.wa-card{background:#fff;border:1px solid #e5e7eb;border-radius:12px;padding:1.5rem;box-shadow:0 1px 3px rgba(0,0,0,0.04)}
.wa-highlight{border-radius:12px;padding:1.5rem;background:#f9fafb}

/* Benefits */
.wa-benefit{display:flex;gap:1rem;align-items:flex-start;padding:1rem 0}
.wa-benefit-icon{font-size:1.5rem;flex-shrink:0;width:48px;height:48px;display:flex;align-items:center;justify-content:center;border-radius:12px;background:#f3f4f6}
.wa-benefit h4{margin:0 0 0.25rem}
.wa-benefit p{margin:0}

/* Steps */
.wa-steps{counter-reset:step}
.wa-step{counter-increment:step;padding:1.25rem 1.25rem 1.25rem 3.5rem;position:relative;border-left:2px solid #e5e7eb;margin-left:1rem}
.wa-step::before{content:counter(step);position:absolute;left:-1rem;top:1.25rem;width:2rem;height:2rem;border-radius:50%;display:flex;align-items:center;justify-content:center;font-weight:700;font-size:0.9375rem;color:#fff}
.wa-step:last-child{border-left-color:transparent}
.wa-step h4{margin:0 0 0.25rem}
.wa-step p{margin:0}

/* Reviews */
.wa-review{border:1px solid #e5e7eb;border-radius:12px;padding:1.25rem;margin:0.75rem 0}
.wa-review-stars{color:#f59e0b;font-size:1rem;letter-spacing:1px}
.wa-review-meta{font-size:0.9em;color:#9ca3af;margin-top:0.5rem}

/* FAQ */
.wa-faq{margin:0.5rem 0}
.wa-faq summary{font-weight:600;cursor:pointer;padding:1rem 0;border-bottom:1px solid #e5e7eb;list-style:none}
.wa-faq summary::-webkit-details-marker{display:none}
.wa-faq summary::after{content:'+';float:right;font-weight:400;color:#9ca3af}
.wa-faq[open] summary::after{content:'-'}
.wa-faq .wa-faq-body{padding:1rem 0}

/* CTA */
.wa-cta{text-align:center;padding:2.5rem 1.5rem;border-radius:12px;margin:2rem 0}

/* Collections */
.wa-collections{padding:1.5rem 0;border-top:1px solid #e5e7eb;margin-top:2rem}
.wa-collections-grid{display:flex;flex-wrap:wrap;gap:0.5rem}
.wa-collections-grid a{padding:0.5rem 1rem;border-radius:999px;border:1px solid #e5e7eb;text-decoration:none;color:inherit;transition:all 0.2s}
.wa-collections-grid a:hover{border-color:#999}

/* Links */
.wa-page a.wa-link{text-decoration:underline;text-decoration-color:#d1d5db;text-underline-offset:2px;color:inherit}
.wa-page a.wa-link:hover{text-decoration-color:currentColor}

/* Specs table */
.wa-specs{width:100%;border-collapse:collapse;margin:1rem 0}
.wa-specs tr{border-bottom:1px solid #f3f4f6}
.wa-specs td,.wa-specs th{padding:0.75rem;text-align:left}
.wa-specs td:first-child,.wa-specs th:first-child{font-weight:600;width:40%}

/* Ingredients */
.wa-ingredients{display:flex;flex-wrap:wrap;gap:0.5rem;margin:1rem 0}
.wa-ingredient-tag{padding:0.4rem 0.9rem;border-radius:999px;border:1px solid #e5e7eb;background:#fafafa}
.wa-ingredient-tag.key{font-weight:600;border-color:currentColor}

/* Caption */
.wa-img-caption{text-align:center;margin-top:0.5rem;color:#888;font-size:0.9em}

/* Reveal */
.reveal{opacity:0;transform:translateY(16px);transition:opacity 0.5s,transform 0.5s}
.reveal.visible{opacity:1;transform:none}
.reveal.d1{transition-delay:0.1s}.reveal.d2{transition-delay:0.2s}.reveal.d3{transition-delay:0.3s}

/* Cross-sell */
.wa-cross-sell-grid{display:grid;grid-template-columns:repeat(3,1fr);gap:1rem}
@media(max-width:640px){.wa-cross-sell-grid{grid-template-columns:1fr}}
.wa-cross-sell-item{text-decoration:none;color:inherit;border:1px solid #e5e7eb;border-radius:12px;padding:1rem;text-align:center}
.wa-cross-sell-item:hover{border-color:#999}
"""

WA_OBSERVER_SCRIPT = '<script>(function(){var io=new IntersectionObserver(function(e){e.forEach(function(x){if(x.isIntersecting){x.target.classList.add("visible");io.unobserve(x.target);}});},{threshold:0.15});document.querySelectorAll(".reveal").forEach(function(el){io.observe(el);});})();</script>'

# ── Trust signal blocks that Claude can pick from ──
TRUST_SIGNALS_PROMPT = """
TRUST SIGNALS (include 3-5 that fit this product):
Use <div class="wa-trust"> with <div class="wa-trust-badge"><span class="icon">EMOJI</span> TEXT</div> elements.
Choose from (pick what fits, do not use all):
  Free Shipping    Ships in 24-48h    30-Day Returns    Buyer Protection
  Secure Checkout   Quality Tested     Cruelty-Free      4.8/5 Rating
  True to Size      Waterproof         Long-Lasting       Gift Ready
"""


# ── v9.2: Additional HTML block prompts ──
SPECS_TABLE_PROMPT = """
If SPECS data is available, create a specs table using:
<table class="wa-specs"><tr><td>Weight</td><td>30g</td></tr>...</table>
Only include specs that matter to buyers (skip internal codes, supplier info).
"""

SIZE_GUIDE_PROMPT = """
If SIZE CHART data is available, wrap it in:
<div class="wa-size-guide"><table><thead><tr><th>Size</th>...</tr></thead><tbody>...</tbody></table></div>
Add a note: "Tip: If between sizes, we recommend sizing up."
"""

VIDEO_EMBED_PROMPT = """
If VIDEO URL is available, embed it:
<div class="wa-video"><video src="URL" controls preload="none" poster="HERO_IMG_URL"></video></div>
Place it after the hero section for maximum engagement.
"""

CROSS_SELL_PROMPT = """
If RELATED PRODUCTS are available, add:
<div class="wa-cross-sell"><h3>Pairs Well With</h3><div class="wa-cross-sell-grid">
  <a href="/products/handle" class="wa-cross-sell-item"><p>Product Name</p></a>
</div></div>
"""

INGREDIENTS_PROMPT = """
If KEY INGREDIENTS are visible, create:
<div class="wa-ingredients">
  <span class="wa-ingredient-tag key">Hyaluronic Acid</span>
  <span class="wa-ingredient-tag">Vitamin C</span>
</div>
Mark the top 2-3 hero ingredients with class="key" (gets bolder styling).
"""

# v9.3: CSS deduplication — generate theme snippet for one-time install
# Products no longer carry the full CSS in their metafield.
# Instead, a Shopify theme snippet loads the CSS once.

WA_SNIPPET_LIQUID = '''<!-- Wanelo Product Description Styles — Add to snippets/wa-styles.liquid -->
<!-- Then include in product template: {% render 'wa-styles' %} -->
<style>
''' + WA_CSS_FRAMEWORK + '''
</style>
'''

CSS_MODE = "inline"  # CSS embedded in each product (no theme snippet needed)  # "snippet" = CSS in theme (fast), "inline" = CSS in each metafield (fallback)
# ⚠ snippet mode: install wa-styles.liquid in your theme! Otherwise products render unstyled.

print(f'CSS Framework: {len(WA_CSS_FRAMEWORK)} chars')
print(f'CSS Mode: {CSS_MODE}')
if CSS_MODE == "snippet":
    print(f'  ⚠ Install snippets/wa-styles.liquid in your Shopify theme!')
    print(f'  ⚠ Add {{% render "wa-styles" %}} to your product template.')
print(f'Trust signals prompt: ready')
print('Cell 5 OK')

# v9.5: Static system prompt for writer (cached across products)
WRITER_SYSTEM_PROMPT = """You are a world-class frontend designer and copywriter for wanelo.com.
You generate product description pages using the WANELO CSS FRAMEWORK.

CSS CLASSES (pre-loaded):
wa-page(wrapper) | wa-section.reveal(.d1/.d2/.d3) | wa-card | wa-highlight | wa-grid.wa-grid-2/.wa-grid-3
wa-hero-img | wa-img-caption | wa-benefit>wa-benefit-icon+div | wa-steps>wa-step(::before=counter)
wa-review>wa-review-stars+p+wa-review-meta | wa-faq>summary+wa-faq-body(details) | wa-cta
wa-trust>wa-trust-badge>span.icon+text | wa-collections>wa-collections-grid>a | a.wa-link(inline)
wa-specs(table) | wa-size-guide(div>table) | wa-ingredients>wa-ingredient-tag(.key) | wa-video(div>video)
wa-cross-sell>wa-cross-sell-grid>a.wa-cross-sell-item

CREATIVE RULES:
- BOLD aesthetic per product, never cookie-cutter
- This is a SALES PAGE, not a product description — every section should move toward purchase
- Specific numbers always: "93% improvement in 14 days" not "fast results"
- Testimonials: max 3, brief, editorial with mild skepticism (Judge.me handles real reviews)
- Generous negative space, editorial feel
- Simple products: 5-6 sections. Complex: 8-10 sections.
- Follow INTERLINKING RULES for collection links
- Page flow: Hook → Problem → Solution → Proof → Details → Transformation → Value → CTA

CRITICAL:
- Use ONLY provided image URLs exactly as given (full https:// URLs). NEVER invent, shorten, or modify.
- Collection links MUST come from provided list.
- All <img> MUST have style="max-width:720px;width:100%;height:auto;display:block;margin:1rem auto;border-radius:12px" loading="lazy" + descriptive alt text.
- Scoped <style>: only product-specific accent overrides.
- Wrap in <div class="rte wa-page">
- Do NOT include shipping badges, trust badges, delivery info, or return policy — these are on the store theme already.
- Do NOT add "Secure Checkout", "Free Shipping", etc badges — the theme handles these.

OUTPUT FORMAT:
===HTML_START===
<div class="rte wa-page"><style>/* accent only */</style>...HTML...</div>
===HTML_END===
===META_START===
{"short_description":"2-3 sentences","seo_meta":{"title":"55-60 chars","description":"140-155 chars","handle":"url-slug"},"product_tags":["cluster:xxx","cluster:yyy","persona:zzz","intent:www","demo:female","demo:young-adult"]}
===META_END===

TAGS RULES (CRITICAL):
- product_tags MUST contain ONLY tags from the AVAILABLE CLUSTERS / PERSONAS / INTENTS / DEMO lists in the user message.
- NEVER invent new tags. NEVER omit prefixes (cluster:/persona:/intent:/demo:).
- Required composition: 2-4 cluster: + 1-2 persona: + 1-2 intent: + EXACTLY 1 demo:gender + 1-2 demo:age.
- Total 6-9 tags. If the available lists don't fit, pick closest match — never fabricate.""" + HTML_CLEAN_OUTPUT_RULES



CSS Framework: 4013 chars
CSS Mode: inline
Trust signals prompt: ready
Cell 5 OK


## Cell 5.5 — Taxonomy Loader (v3.2)

Loads master taxonomy from Git URL, builds FAISS index for semantic shortlist, exposes `get_cluster_shortlist()` and `validate_tags()` to the product loop.

In [ ]:
# ════ TAXONOMY LOADER (v3.2) ════
# Loads master taxonomy from Git URL, indexes in DB + FAISS.
# Provides: get_cluster_shortlist(), validate_tags(), TAX_PERSONAS, TAX_INTENTS

import json, os, requests
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# ── Fetch from URL with local cache ──
TAX_LOCAL = f"{PROJECT_DIR}/taxonomy.json"
if TAXONOMY_REFRESH or not os.path.exists(TAX_LOCAL):
    print(f"Fetching taxonomy from {TAXONOMY_URL}...")
    r = requests.get(TAXONOMY_URL, timeout=30)
    r.raise_for_status()
    with open(TAX_LOCAL, "w", encoding="utf-8") as f:
        f.write(r.text)
    print(f"  Saved to {TAX_LOCAL}")
else:
    print(f"Using cached taxonomy: {TAX_LOCAL}")

with open(TAX_LOCAL, encoding="utf-8") as f:
    TAXONOMY = json.load(f)

print(f"Taxonomy v{TAXONOMY['version']}: "
      f"{TAXONOMY['stats']['total_clusters']} clusters, "
      f"{TAXONOMY['stats']['personas']} personas, "
      f"{TAXONOMY['stats']['intents']} intents")

# ── Filter approved + (optionally) drafts ──
_status_filter = {"approved"} if TAXONOMY_APPROVED_ONLY else {"approved", "draft"}
_clusters = [c for c in TAXONOMY["clusters"] if c["status"] in _status_filter]
print(f"Active clusters: {len(_clusters)} (status: {sorted(_status_filter)})")

# ── Index in DB ──
db.executescript("""
CREATE TABLE IF NOT EXISTS taxonomy_clusters (
    tag TEXT PRIMARY KEY,
    section_id TEXT, section TEXT, title_en TEXT, title_ru TEXT,
    typical_products TEXT, personas TEXT, intents TEXT,
    primary_gender TEXT, ages TEXT, status TEXT, data_json TEXT
);
CREATE INDEX IF NOT EXISTS idx_tax_section ON taxonomy_clusters(section_id);
CREATE INDEX IF NOT EXISTS idx_tax_status ON taxonomy_clusters(status);
""")
db.execute("DELETE FROM taxonomy_clusters")
for c in _clusters:
    demos = c.get("demos") or {}
    db.execute("""INSERT INTO taxonomy_clusters
        (tag, section_id, section, title_en, title_ru, typical_products,
         personas, intents, primary_gender, ages, status, data_json)
        VALUES (?,?,?,?,?,?,?,?,?,?,?,?)""",
        (c["tag"], c["section_id"], c["section"], c["title_en"], c.get("title_ru",""),
         json.dumps(c.get("typical_products",[]), ensure_ascii=False),
         json.dumps(c.get("personas",[])),
         json.dumps(c.get("intents",[])),
         demos.get("primary_gender",""),
         json.dumps(demos.get("ages",[])),
         c["status"],
         json.dumps(c, ensure_ascii=False)))
db.commit()

# ── FAISS index for semantic shortlist ──
print("Building FAISS index...")
_embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
_texts = [c["embed_text"] for c in _clusters]
_vectors = _embedder.encode(_texts, show_progress_bar=False, normalize_embeddings=True).astype("float32")
_faiss_index = faiss.IndexFlatIP(_vectors.shape[1])
_faiss_index.add(_vectors)

TAX_TAGS_LIST = [c["tag"] for c in _clusters]
TAX_BY_TAG = {c["tag"]: c for c in _clusters}
TAX_PERSONAS = [p["tag"] for p in TAXONOMY["personas"]]
TAX_INTENTS = [i["tag"] for i in TAXONOMY["intents"]]
TAX_GENDERS = [d["tag"] for d in TAXONOMY["demos"] if d["type"] == "gender"]
TAX_AGES = [d["tag"] for d in TAXONOMY["demos"] if d["type"] == "age"]
TAX_VALID_TAGS = set(TAX_TAGS_LIST) | set(TAX_PERSONAS) | set(TAX_INTENTS) | set(TAX_GENDERS) | set(TAX_AGES)

print(f"FAISS index: {len(_clusters)} vectors")

# ── Helper: shortlist for a product ──
def get_cluster_shortlist(product_summary, category, top_k=40):
    """Return top-K candidate clusters by semantic similarity to product."""
    query = f"{category} {product_summary}".strip() or "product"
    qv = _embedder.encode([query], normalize_embeddings=True).astype("float32")
    D, I = _faiss_index.search(qv, min(top_k, len(_clusters)))
    return [TAX_BY_TAG[TAX_TAGS_LIST[i]] for i in I[0]]

def format_shortlist_for_prompt(shortlist):
    """Format shortlist as compact text for Stage 2 prompt."""
    lines = ["AVAILABLE CLUSTERS (pick 2-4 from this list ONLY — never invent):"]
    for c in shortlist:
        prods = ", ".join(c.get("typical_products", [])[:4])
        lines.append(f"  {c['tag']} — {c['title_en']} (e.g. {prods})")
    lines.append("")
    lines.append(f"PERSONAS (pick 1-2): {', '.join(TAX_PERSONAS)}")
    lines.append(f"INTENTS (pick 1-2): {', '.join(TAX_INTENTS)}")
    lines.append(f"DEMO GENDER (pick exactly 1): {', '.join(TAX_GENDERS)}")
    lines.append(f"DEMO AGES (pick 1-2): {', '.join(TAX_AGES)}")
    return "\n".join(lines)

def validate_tags(picked_tags):
    """Drop tags not in master taxonomy. Returns (valid_tags, dropped_tags)."""
    valid = [t for t in picked_tags if t in TAX_VALID_TAGS]
    dropped = [t for t in picked_tags if t not in TAX_VALID_TAGS]
    return valid, dropped

print("✓ Taxonomy loader ready: get_cluster_shortlist, validate_tags, format_shortlist_for_prompt")


In [ ]:
# ════ Page Builder (modular HTML) ════
# Fetches pipeline/page_builder.py from the repo and exec's it into a module
# namespace, exposing assemble_page / MODULE_CATALOG / render_badges.
# Cleanup-safe: subsequent re-runs replace the namespace.
import urllib.request, types

if USE_MODULAR_HTML:
    print(f"Fetching page_builder.py from {PAGE_BUILDER_URL}...")
    _pb_src = urllib.request.urlopen(PAGE_BUILDER_URL, timeout=20).read().decode("utf-8")
    page_builder = types.ModuleType("page_builder")
    page_builder.__file__ = "<runtime fetched>"
    exec(_pb_src, page_builder.__dict__)
    assemble_page    = page_builder.assemble_page
    MODULE_CATALOG   = page_builder.MODULE_CATALOG
    render_badges    = page_builder.render_badges
    print(f"✓ page_builder loaded: {len(MODULE_CATALOG)} modules — {sorted(MODULE_CATALOG)}")
else:
    print("USE_MODULAR_HTML=False — using legacy Sonnet writer (page_builder skipped)")


## Cell 6 — Product Loop v9 (resumable)

In [ ]:
# ════ PRODUCT LOOP v9 ════


import shutil

# ── Fetch ALL collections from Shopify for interlinking ──
ALL_SHOP_COLLECTIONS = []
try:
    r = await shopify_gql('query{collections(first:100){nodes{id title handle}}}')
    if r and r.get('data',{}).get('collections',{}).get('nodes'):
        ALL_SHOP_COLLECTIONS = r['data']['collections']['nodes']
        print(f'Shopify collections for interlinking: {len(ALL_SHOP_COLLECTIONS)}')
except:
    print('Could not fetch Shopify collections')

# ── v9.1: Build rich vision context for writer ──
def _build_vision_context(s1):
    """Format all vision data into a structured prompt section for the writer."""
    lines = []

    # Packaging text (read from labels)
    pkg = s1.get('packaging_text', {})
    if pkg.get('volume'):
        lines.append(f"VOLUME: {pkg['volume']}")
    if pkg.get('claims'):
        lines.append(f"CLAIMS ON PACKAGING: {', '.join(pkg['claims'])}")
    if pkg.get('ingredients_visible'):
        lines.append(f"KEY INGREDIENTS (visible on label): {', '.join(pkg['ingredients_visible'])}")
    if pkg.get('certifications'):
        lines.append(f"CERTIFICATIONS: {', '.join(pkg['certifications'])}")
    if pkg.get('instructions_visible'):
        lines.append(f"USAGE INSTRUCTIONS: {pkg['instructions_visible']}")

    # Sensory details
    sns = s1.get('sensory', {})
    sensory_parts = []
    if sns.get('texture'): sensory_parts.append(f"texture: {sns['texture']}")
    if sns.get('material'): sensory_parts.append(f"material: {sns['material']}")
    if sns.get('finish_description'): sensory_parts.append(sns['finish_description'])
    if sensory_parts:
        lines.append(f"LOOK & FEEL: {', '.join(sensory_parts)}")
    if sns.get('color_names'):
        lines.append(f"COLORS: {', '.join(sns['color_names'])}")
    if sns.get('perceived_quality') and sns['perceived_quality'] != 'mid-range':
        lines.append(f"PERCEIVED QUALITY: {sns['perceived_quality']}")

    # Physical details
    phys = s1.get('physical', {})
    if phys.get('key_features'):
        lines.append(f"KEY FEATURES: {', '.join(phys['key_features'])}")
    if phys.get('whats_included') and len(phys['whats_included']) > 1:
        lines.append(f"WHATS INCLUDED: {', '.join(phys['whats_included'])}")
    if phys.get('applicator_type'):
        lines.append(f"APPLICATOR: {phys['applicator_type']}")
    if phys.get('size_impression') and phys['size_impression'] != 'standard':
        lines.append(f"SIZE: {phys['size_impression']}")

    # Conversion signals
    conv = s1.get('conversion', {})
    if conv.get('hero_claim'):
        lines.append(f"HERO CLAIM (use as headline or first line): {conv['hero_claim']}")
    if conv.get('pain_points_solved'):
        lines.append(f"SOLVES THESE PROBLEMS: {', '.join(conv['pain_points_solved'])}")
    if conv.get('visible_results'):
        lines.append(f"VISIBLE RESULTS: {conv['visible_results']}")
    if conv.get('premium_signals'):
        lines.append(f"PREMIUM SIGNALS: {', '.join(conv['premium_signals'])}")
    if conv.get('target_audience'):
        lines.append(f"TARGET AUDIENCE: {conv['target_audience']}")
    if conv.get('usage_scenario'):
        lines.append(f"USAGE SCENARIO: {conv['usage_scenario']}")
    if conv.get('emotional_tone'):
        lines.append(f"TONE: {conv['emotional_tone']}")

    # Conversion hints
    vol_text = pkg.get('volume', '')
    if vol_text:
        ml_m = re.search(r'(\d+)\s*ml', vol_text)
        oz_m = re.search(r'(\d+)\s*(?:fl\s*)?oz', vol_text)
        if ml_m:
            ml = int(ml_m.group(1))
            lines.append(f"LONGEVITY HINT: ~{max(30, ml * 2)} days at daily use ({ml}ml)")
        elif oz_m:
            oz = int(oz_m.group(1))
            lines.append(f"LONGEVITY HINT: ~{max(30, oz * 60)} days at daily use ({oz}oz)")
    items = phys.get('whats_included', [])
    if len(items) >= 2:
        lines.append(f"VALUE HINT: {len(items)} items — emphasize as great value")
    faq_hints = []
    if conv.get('premium_signals'): faq_hints.append("price: premium materials")
    if conv.get('target_audience'): faq_hints.append(f"audience: {conv['target_audience'][:30]}")
    if pkg.get('instructions_visible'): faq_hints.append("usage: from packaging")
    if faq_hints:
        lines.append("FAQ HINTS: " + " | ".join(faq_hints))
    return '\n'.join(lines) if lines else 'No additional vision data available.'





pending = db.execute("SELECT * FROM products WHERE status != 'done' ORDER BY id").fetchall()
total = db.execute('SELECT COUNT(*) FROM products').fetchone()[0]
done_count = total - len(pending)

print(f'\n{"="*60}')
print(f'  {total} total | {done_count} done | {len(pending)} remaining')
print(f'{"="*60}')

for pi, prod in enumerate(pending):
    pid, pname, purl, pstatus = prod['id'], prod['name'], prod['eprolo_url'], prod['status']
    print(f'\n{"─"*60}')
    print(f'[{done_count+pi+1}/{total}] {pname[:60]}')
    print(f'  Status: {pstatus}')

    if not purl:
        db_update_status(pid, 'error', error_msg='No EPROLO URL'); continue

    if prod['shopify_product_id'] and pstatus == 'done':
        print('  Already in Shopify, skipping.'); continue

    try:
        # ═══ STEP 1: Scrape (v9: + variants) ═══
        if pstatus == 'pending':
            print('  → Scraping...')
            product = await scrape_eprolo(purl)
            print(f'    Title: {product["title"][:60]}')
            scrape_issues = validate_scrape(product)
            if scrape_issues:
                print(f'    ⚠ Scrape issues: {", ".join(scrape_issues)}')
                if 'empty/short title' in scrape_issues:
                    db_update_status(pid, 'error', error_msg=f'Scrape failed: {", ".join(scrape_issues)}')
                    continue
            db_update_status(pid, 'scraped',
                scrape_json=json.dumps(product, ensure_ascii=False),
                variants_json=json.dumps(product.get("variants",[]), ensure_ascii=False))
            pstatus = 'scraped'

        # ═══ STEP 2: Images ═══
        if pstatus == 'scraped':
            product = json.loads(db.execute('SELECT scrape_json FROM products WHERE id=?',(pid,)).fetchone()[0])
            print('  → Images...')
            top_images = await dl_images(product["top_image_urls"], "TOP")
            desc_images = await dl_images(product["desc_image_urls"], "DESC")
            if SHOPIFY_TOKEN:
                top_cdn = await upload_to_shopify(top_images, "TOP")
                desc_cdn = await upload_to_shopify(desc_images, "DESC")
                for item in top_cdn:
                    if item['cdn']:
                        for img in top_images:
                            if img['index']==item['index']: img['url']=item['url']
                for item in desc_cdn:
                    if item['cdn']:
                        for img in desc_images:
                            if img['index']==item['index']: img['url']=item['url']
                cdn_ok = sum(1 for x in top_cdn+desc_cdn if x['cdn'])
                print(f'    CDN: {cdn_ok}/{len(top_cdn)+len(desc_cdn)}')
            img_data = {
                "top":[{"index":i["index"],"url":i["url"],"media_type":i["media_type"],"size_kb":i["size_kb"]} for i in top_images],
                "desc":[{"index":i["index"],"url":i["url"],"media_type":i["media_type"],"size_kb":i["size_kb"]} for i in desc_images],
                "top_b64":[{"index":i["index"],"base64":i["base64"],"media_type":i["media_type"]} for i in top_images],
                "desc_b64":[{"index":i["index"],"base64":i["base64"],"media_type":i["media_type"]} for i in desc_images]}
            db_update_status(pid, 'images_uploaded', image_urls_json=json.dumps(img_data, ensure_ascii=False))
            pstatus = 'images_uploaded'

        # ═══ STEP 3: Vision ═══
        if pstatus == 'images_uploaded':
            product = json.loads(db.execute('SELECT scrape_json FROM products WHERE id=?',(pid,)).fetchone()[0])
            img_data = json.loads(db.execute('SELECT image_urls_json FROM products WHERE id=?',(pid,)).fetchone()[0])
            print('  → Vision...')
            vb = []
            _top_imgs = img_data.get('top_b64', [])[:4]
            _desc_imgs = img_data.get('desc_b64', [])[:4]
            if _top_imgs:
                vb.append({"type":"text","text":"GALLERY IMAGES:"})
                for img in _top_imgs:
                    vb.append({"type":"image","source":{"type":"base64","media_type":img["media_type"],"data":img["base64"]}})
            if _desc_imgs:
                vb.append({"type":"text","text":"DESCRIPTION IMAGES:"})
                for img in _desc_imgs:
                    vb.append({"type":"text","text":f"IMG_{img['index']}:"})
                    vb.append({"type":"image","source":{"type":"base64","media_type":img["media_type"],"data":img["base64"]}})
            safe_title = product['title']
            vision_prompt = f"""Analyze these product images. Extract EVERYTHING that helps sell this product.
Product: {safe_title}

Return ONLY JSON (no markdown, no extra text):
{{
  "accent": "#hex pastel color from product",
  "accent_soft": "#hex 10% opacity version",
  "bg": "#FFFFFF",
  "palette": {
    "brand_1": "#hex primary warm accent (e.g. from packaging hero color)",
    "brand_2": "#hex secondary cool accent (complementary to brand_1)",
    "brand_3": "#hex tertiary accent (analogous to brand_1)",
    "brand_soft": "#hex 10% lightened brand_1 (for soft backgrounds)",
    "brand_deep": "#hex 20% darkened brand_1 (for emphasis)"
  },
  "category": "product category",
  "product_type": "simple|complex|technical|fashion|decorative",
  "product_summary": "2-3 engaging sentences about what this product does",
  "needs_steps": true,
  "packaging_text": {{
    "volume": "size/weight if visible (e.g. 30ml, 50g) or empty",
    "claims": ["exact claims visible: 24h hydration, SPF 50, etc"],
    "ingredients_visible": ["key ingredients readable on label"],
    "certifications": ["logos/text: cruelty-free, vegan, organic, etc"],
    "instructions_visible": "usage directions if readable, else empty"
  }},
  "sensory": {{
    "texture": "matte|glossy|shimmer|cream|gel|powder|liquid|foam|oil",
    "material": "glass|plastic|metal|wood|silicone|ceramic|fabric or empty",
    "finish_description": "short vivid phrase: frosted glass with gold pump",
    "color_names": ["descriptive names: dusty rose, champagne gold"],
    "perceived_quality": "budget|mid-range|premium|luxury"
  }},
  "physical": {{
    "size_impression": "travel-size|compact|standard|full-size|jumbo",
    "key_features": ["visible features: pump, magnetic closure, built-in mirror"],
    "whats_included": ["all visible items: main product, brush, case"],
    "applicator_type": "brush|sponge|dropper|pump|spray|tube|jar or empty"
  }},
  "conversion": {{
    "hero_claim": "single strongest selling proposition visible or inferred",
    "pain_points_solved": ["specific problems: uneven skin, dry lips"],
    "visible_results": "describe any before/after or visible effect",
    "premium_signals": ["what makes it look premium: glass bottle, gold accents"],
    "target_audience": "specific: young women into K-beauty, men with beard care",
    "usage_scenario": "when/where: daily morning routine, date night, travel",
    "emotional_tone": "luxury|clinical|fun|natural|professional|playful"
  }},
  "images": [
    {{"index": 1, "role": "hero|feature|detail|lifestyle|ingredient|before_after|packaging|skip", "desc": "ALT text", "shows": "what this image proves about the product"}}
  ]
}}"""
            vb.append({"type":"text","text":vision_prompt})

            if not _top_imgs and not _desc_imgs:
                print('    No images — using default vision')
                stage1 = {"accent":"#C4B09A","accent_soft":"#F5F0EA","bg":"#FFFFFF",
                    "category":product['title'].split()[0],"product_type":"simple",
                    "product_summary":product.get('description','')[:200],"needs_steps":False,
                    "packaging_text":{"volume":"","claims":[],"ingredients_visible":[],"certifications":[],"instructions_visible":""},
                    "sensory":{"texture":"","material":"","finish_description":"","color_names":[],"perceived_quality":"mid-range"},
                    "physical":{"size_impression":"standard","key_features":[],"whats_included":[],"applicator_type":""},
                    "conversion":{"hero_claim":"","pain_points_solved":[],"visible_results":"","premium_signals":[],"target_audience":"","usage_scenario":"","emotional_tone":""},
                    "images":[]}
                cost1 = 0
            else:
                _vm = MODEL_VISION
                r1 = client.messages.create(model=_vm, max_tokens=2000, messages=[{"role":"user","content":vb}])
                raw = r1.content[0].text.strip()
                if raw.startswith("```"): raw = raw.split("\n", 1)[-1].rsplit("\n", 1)[0]
                try:
                    stage1 = json.loads(raw)
                    pkg = stage1.get('packaging_text', {})
                    if _vm == MODEL_VISION and not pkg.get('claims') and not pkg.get('ingredients_visible') and len(_top_imgs) <= 5:
                        print(f'    Haiku found no packaging text, retrying Sonnet...')
                        _vm = MODEL_WRITER
                        r1 = client.messages.create(model=_vm, max_tokens=2000, messages=[{"role":"user","content":vb}])
                        raw = r1.content[0].text.strip()
                        if raw.startswith("```"): raw = raw.split("\n", 1)[-1].rsplit("\n", 1)[0]
                        try: stage1 = json.loads(raw)
                        except: pass
                except json.JSONDecodeError:
                    print(f'    Vision JSON failed, using defaults')
                    stage1 = {"accent":"#C4B09A","accent_soft":"#F5F0EA","bg":"#FFFFFF",
                        "category":"Product","product_type":"simple",
                        "product_summary":"","needs_steps":False,"images":[]}
                if 'sonnet' in str(_vm).lower():
                    cost1 = r1.usage.input_tokens*3/1e6 + r1.usage.output_tokens*15/1e6
                else:
                    cost1 = r1.usage.input_tokens*0.80/1e6 + r1.usage.output_tokens*4/1e6

            pkg_claims = len(stage1.get('packaging_text',{}).get('claims',[]))
            ingr_count = len(stage1.get('packaging_text',{}).get('ingredients_visible',[]))
            hero = stage1.get('conversion',{}).get('hero_claim','')[:40]
            print(f'    OK ${cost1:.4f} | {stage1.get("product_type")} | {len(stage1.get("images",[]))} imgs | claims:{pkg_claims} ingr:{ingr_count}')
            if hero: print(f'    Hero: {hero}')
            db_update_status(pid, 'vision_done', stage1_json=json.dumps(stage1, ensure_ascii=False), cost_usd=(prod['cost_usd'] or 0)+cost1)
            _img_clean = json.loads(db.execute('SELECT image_urls_json FROM products WHERE id=?',(pid,)).fetchone()[0])
            _img_clean.pop('top_b64', None)
            _img_clean.pop('desc_b64', None)
            db.execute('UPDATE products SET image_urls_json=? WHERE id=?', (json.dumps(_img_clean, ensure_ascii=False), pid))
            db.commit()
            pstatus = 'vision_done'

        # ═══ STEP 4: Claude generates HTML ═══
        if pstatus == 'vision_done':
            product = json.loads(db.execute('SELECT scrape_json FROM products WHERE id=?',(pid,)).fetchone()[0])
            stage1 = json.loads(db.execute('SELECT stage1_json FROM products WHERE id=?',(pid,)).fetchone()[0])
            img_data = json.loads(db.execute('SELECT image_urls_json FROM products WHERE id=?',(pid,)).fetchone()[0])
            variants = json.loads(db.execute('SELECT variants_json FROM products WHERE id=?',(pid,)).fetchone()[0] or '[]')

            all_avail_imgs = img_data.get('desc',[]) + img_data.get('top',[])
            img_ctx = ""
            for ii in stage1.get('images', []):
                url = next((d['url'] for d in all_avail_imgs if d['index']==ii['index']), None)
                if not url or ii.get('role')=='skip': continue
                shows = ii.get('shows', '')
                shows_text = f"\n    SHOWS: {shows}" if shows else ''
                img_ctx += f"\n  [{ii.get('role','').upper()}] {url}\n    ALT: {ii.get('desc','')}{shows_text}"
            if not img_ctx.strip():
                for img in img_data.get('desc',[])[:6]:
                    img_ctx += f"\n  [DETAIL] {img['url']}\n    ALT: Product detail"
                for img in img_data.get('top',[])[:4]:
                    img_ctx += f"\n  [LIFESTYLE] {img['url']}\n    ALT: Product photo"

            safe_title = product['title']
            safe_desc = product.get('description','') or ''

            interlinks = get_smart_interlinks(pid, pname, ALL_SHOP_COLLECTIONS)
            interlink_block = ""
            if interlinks:
                interlink_block = "\nINTERLINKS — weave into body text paragraphs:\n"
                for lnk in interlinks:
                    tag = " *OWN" if lnk['relevance'] == 'own' else ""
                    anchors = lnk.get('anchors', [])
                    anchor_str = ', '.join(f'"{a}"' for a in anchors[:3]) if anchors else f'"{lnk["title"]}"'
                    interlink_block += f'  /collections/{lnk["handle"]}{tag}\n'
                    interlink_block += f'    Anchor text options: {anchor_str}\n'
                interlink_block += "  RULES: 2-3 keyword links IN body paragraphs + footer wa-collections with ALL collections as buttons. Never leave wa-collections-grid empty.\n"

            variant_ctx = ""
            if variants:
                variant_ctx = "\nPRODUCT VARIANTS:\n"
                for v in variants:
                    variant_ctx += f"  {v['option_name']}: {', '.join(v['values'][:10])}\n"

            specs = product.get('specs', {})
            specs_ctx = ""
            if specs:
                specs_ctx = "\nSPECS:\n"
                for k, v in list(specs.items())[:15]:
                    specs_ctx += f"  {k}: {v}\n"

            video_urls = product.get('video_urls', [])
            video_ctx = f"\nVIDEO: {video_urls[0]}\n" if video_urls else ""

            size_chart = product.get('size_chart', [])
            size_ctx = ""
            if size_chart and len(size_chart) >= 2:
                size_ctx = "\nSIZE CHART:\n"
                for row in size_chart[:10]:
                    size_ctx += "  " + " | ".join(row) + "\n"

            cross_sell_ctx = ""
            own_colls = get_product_collections(pid)
            if own_colls:
                sibling_products = []
                for coll_title, coll_url, coll_sid in own_colls:
                    if not coll_sid: continue
                    siblings = db.execute("""SELECT p.name, p.url_handle FROM products p
                        JOIN product_collections pc ON p.id = pc.product_id
                        WHERE pc.collection_id IN (SELECT id FROM collections WHERE shopify_collection_id=?)
                        AND p.id != ? AND p.shopify_product_id IS NOT NULL
                        LIMIT 6""", (coll_sid, pid)).fetchall()
                    for s in siblings:
                        if s['url_handle'] and s['name'] not in [x[0] for x in sibling_products]:
                            sibling_products.append((s['name'], s['url_handle']))
                if sibling_products:
                    cross_sell_ctx = "\nRELATED PRODUCTS:\n"
                    for name, handle in sibling_products[:4]:
                        cross_sell_ctx += f'  /products/{handle} — {name}\n'

            seo_kw_row = db.execute('SELECT seo_keywords FROM products WHERE id=?', (pid,)).fetchone()
            product_kws = seo_kw_row['seo_keywords'] if seo_kw_row and seo_kw_row['seo_keywords'] else ''

            all_winner_data = []
            own_colls_kw = get_product_collections(pid)
            _pw = set(re.sub(r'[^a-z0-9 ]', ' ', pname.lower()).split())
            _pw -= {'the','a','an','for','and','or','with','in','of','to','set','kit','pcs','pc','new','style','color','pack'}
            for coll_title, _, _ in own_colls_kw:
                rows_kw = db.execute(
                    'SELECT keyword, volume, kd, cpc FROM seo_keywords WHERE collection_name=? AND volume > 0',
                    (coll_title,)).fetchall()
                for rk in rows_kw:
                    kw_words = set(rk['keyword'].lower().split())
                    if _pw & kw_words or any(w in pname.lower() for w in kw_words if len(w) > 3):
                        all_winner_data.append({
                            'keyword': rk['keyword'], 'volume': rk['volume'],
                            'kd': rk['kd'], 'cpc': rk['cpc']
                        })

            t1_primary, t2_secondary, t3_support = tier_keywords(all_winner_data)

            kw_ctx = ""
            if product_kws or t1_primary or t2_secondary:
                kw_ctx = "\nSEO KEYWORDS (WANELO.com DA 60+):\n"
                if product_kws:
                    kw_ctx += f"  PRODUCT: {product_kws}\n"
                if t1_primary:
                    kw_ctx += "  PRIMARY (KD 0-20, use in H2 + first paragraphs):\n"
                    for kd in t1_primary[:6]:
                        kw_ctx += f"    {kd['keyword']} (vol:{kd['volume']:,} kd:{kd['kd']})\n"
                if t2_secondary:
                    kw_ctx += "  SECONDARY (KD 21-35, body + FAQ + alt):\n"
                    for kd in t2_secondary[:5]:
                        kw_ctx += f"    {kd['keyword']} (vol:{kd['volume']:,})\n"
                if t1_primary or t2_secondary:
                    print(f'    Keywords: {len(t1_primary)} primary + {len(t2_secondary)} secondary')

            print('  → Generating HTML + SEO...')

            # ═══ Taxonomy shortlist (v3.2) ═══
            _shortlist = get_cluster_shortlist(
                stage1.get('product_summary', ''),
                stage1.get('category', ''),
                top_k=TAXONOMY_SHORTLIST_K
            )
            _tax_block = format_shortlist_for_prompt(_shortlist)

            prompt = f"""Generate a product description page for this product:

PRODUCT: {safe_title}
CATEGORY: {stage1.get('category','')}
TYPE: {stage1.get('product_type','simple')}
SUMMARY: {stage1.get('product_summary','')}
DESCRIPTION: {safe_desc[:800]}
ACCENT: {stage1.get('accent','#C4B09A')}
ACCENT_SOFT: {stage1.get('accent_soft','#F5F0EA')}

VISION ANALYSIS:
{_build_vision_context(stage1)}

IMAGES — place each by ROLE:
{img_ctx}

IMAGE PLACEMENT: HERO=top, FEATURE=benefits, DETAIL=specs, LIFESTYLE=usage scenario, PACKAGING=whats included

CRITICAL IMAGE RULES:
- Copy-paste the FULL image URL exactly as given (starts with https://)
- Do NOT shorten, modify, or rename image URLs
- Every <img> MUST have: style="max-width:720px;width:100%;height:auto;display:block;margin:1rem auto;border-radius:12px" loading="lazy"
- This ensures images are centered and never stretch beyond 720px regardless of container width
{interlink_block}
{variant_ctx}
{specs_ctx}
{video_ctx}
{size_ctx}
{cross_sell_ctx}
{kw_ctx}

Do NOT include shipping badges, trust badges, or delivery information — these are already on the store theme.

CONVERSION COPYWRITING FRAMEWORK:
1. HOOK: hero claim as H2. 2. PROBLEM: empathy paragraph. 3. SOLUTION: product intro with wa-grid-2 (image + text side by side).
4. PROOF: claims as wa-highlight badges in a wa-grid-2 or wa-grid-3 layout. 5. DETAILS: key features as wa-benefit blocks (use wa-grid-2 for 2-column layout on desktop).
6. INGREDIENTS: if visible, use wa-ingredients tags. 7. HOW-TO: wa-steps with numbered steps.
8. TRANSFORMATION: visible results as timeline in wa-card. 9. TESTIMONIALS: 2-3 editorial in wa-grid-2 on desktop.
10. VALUE: whats included as wa-grid-3 cards. 11. FAQ: real objections as wa-faq details.
12. CTA: wa-cta with action text. 13. INTERLINKS: keyword anchors in body + footer wa-collections-grid.

LAYOUT RULES:
- Use wa-grid-2 and wa-grid-3 for side-by-side layouts (they collapse to 1 column on mobile)
- Benefits section: 2 columns on desktop (wa-grid-2 wrapping wa-benefit blocks)
- Image + text: place image and description side by side using wa-grid-2
- Specs table: full width
- FAQ: full width accordion
- Never put everything in a single narrow column — use the full width

LANGUAGE: {LANGUAGE}

{_tax_block}"""

            if USE_MODULAR_HTML:
                # === Modular path: Opus 4.7 picks layout/palette/text → assemble_page ===
                _designer_model = MODEL_DESIGNER
                _module_lines = []
                for _mid, _spec in MODULE_CATALOG.items():
                    _module_lines.append(f"  {_mid} ({_spec['kind']}): {_spec['description']}")
                    _module_lines.append(f"    slots: {_spec['slots']}")
                _modules_block = "\n".join(_module_lines)
                _palette_hint = stage1.get("palette") or {
                    "brand_1": stage1.get("accent","#C4B09A"),
                    "brand_2": stage1.get("accent_soft","#F5F0EA"),
                    "brand_3": stage1.get("accent","#C4B09A"),
                    "brand_soft": stage1.get("accent_soft","#F5F0EA"),
                    "brand_deep": stage1.get("accent","#C4B09A"),
                }
                _designer_system = (
                    "You are a senior product page designer. You assemble long-form product "
                    "presentation pages from a fixed catalog of pre-built modules.\n\n"
                    "RULES:\n"
                    "1. Pick 4-8 modules from the catalog. Build a coherent narrative: hero → "
                    "intro → story (1-2 chapters) → features → facts → reviews. You can repeat "
                    "modules (e.g. m21 + m22 for two-chapter story).\n"
                    "2. For each picked module, fill ALL its slots. Use <em>...</em> sparingly "
                    "for typographic emphasis on key words. Russian or English — match LANGUAGE.\n"
                    "3. Pick 5 hex colors for the palette (brand_1/2/3/soft/deep). Use the "
                    "PALETTE_HINT from vision as a starting point but you can refine based on "
                    "product mood (warm/cool/luxury/clinical/playful).\n"
                    "4. Output ONLY a single JSON object. No prose, no markdown code fences.\n\n"
                    "OUTPUT SHAPE:\n"
                    "{\n"
                    '  "design": {\n'
                    '    "layout": ["hero1","m21","m32","m41","m51"],\n'
                    '    "palette": {"brand_1":"#hex","brand_2":"#hex","brand_3":"#hex","brand_soft":"#hex","brand_deep":"#hex"},\n'
                    '    "slots": [ {slot map for module 0}, {for module 1}, ... ]\n'
                    "  },\n"
                    '  "meta": {\n'
                    '    "short_description": "2-3 sentences for product card",\n'
                    '    "seo_meta": {"title":"55-60 chars","description":"140-155 chars","handle":"url-slug"},\n'
                    '    "product_tags": ["cluster:xxx","persona:yyy","intent:zzz","demo:female","demo:young-adult"]\n'
                    "  }\n"
                    "}\n\n"
                    "TAG RULES (same as before): pick from AVAILABLE CLUSTERS / PERSONAS / INTENTS / "
                    "DEMOS lists in user message. 2-4 cluster + 1-2 persona + 1-2 intent + EXACTLY 1 "
                    "demo:gender + 1-2 demo:age. Total 6-9 tags. Never invent tags.\n\n"
                    "FOR BADGES_HTML slots: emit pre-rendered HTML, e.g. "
                    '\'<span class="badge solid">630nm</span><span class="badge">USB-C</span>\'.\n'
                    "FOR <em> tags inside text slots: keep them; they style typographic emphasis."
                    + HTML_CLEAN_OUTPUT_RULES
                )
                _designer_user = prompt + (
                    "\n\n=== MODULE CATALOG ===\n" + _modules_block +
                    "\n\n=== PALETTE HINT (from vision) ===\n" + json.dumps(_palette_hint)
                )
                r2 = client.messages.create(
                    model=_designer_model, max_tokens=16000,
                    system=[{"type":"text","text":_designer_system,"cache_control":{"type":"ephemeral"}}],
                    messages=[{"role":"user","content":_designer_user}])
                raw2 = r2.content[0].text.strip()
                # Opus pricing approximation: $15/$75 per 1M
                cost2 = r2.usage.input_tokens*15/1e6 + r2.usage.output_tokens*75/1e6
                # Parse JSON (with code-fence fallback)
                if raw2.startswith("```"):
                    raw2 = raw2.split("\n",1)[-1].rsplit("\n",1)[0]
                try:
                    designer_resp = json.loads(raw2)
                except json.JSONDecodeError:
                    _jm = re.search(r'\{.*\}', raw2, re.DOTALL)
                    designer_resp = json.loads(_jm.group(0)) if _jm else {}
                _design = designer_resp.get("design") or {}
                _layout = _design.get("layout") or ["hero1"]
                _slots  = _design.get("slots")  or [{}]
                _palette = _design.get("palette") or _palette_hint
                # Drop unknown module ids (fail-soft)
                _layout_filtered = [m for m in _layout if m in MODULE_CATALOG]
                if not _layout_filtered:
                    _layout_filtered = ["hero1"]; _slots = [{}]
                # Align slots length
                _slots = (_slots + [{}] * len(_layout_filtered))[:len(_layout_filtered)]
                final_html = assemble_page(_layout_filtered, _slots, _palette, output_format="shopify_fragment")
                meta = designer_resp.get("meta") or {}
                # raw_html alias for downstream (validate_html, image fixup)
                raw_html = final_html
            else:
                # === Legacy path: Sonnet writes HTML inline with ===HTML_START===/===META_START=== ===
                _writer_model = MODEL_WRITER
                r2 = client.messages.create(
                    model=_writer_model, max_tokens=16000,
                    system=[{"type":"text","text":WRITER_SYSTEM_PROMPT,"cache_control":{"type":"ephemeral"}}],
                    messages=[{"role":"user","content":prompt}])
                raw2 = r2.content[0].text.strip()
                cost2 = r2.usage.input_tokens*3/1e6 + r2.usage.output_tokens*15/1e6

                hm = re.search(r'===HTML_START===\s*(.*?)\s*===HTML_END===', raw2, re.DOTALL)
                raw_html = hm.group(1).strip() if hm else raw2.split('===META_START===')[0].strip()

                if CSS_MODE == "inline":
                    if '<style>' in raw_html:
                        final_html = raw_html.replace('<style>', '<style>\n' + WA_CSS_FRAMEWORK + '\n', 1)
                    else:
                        final_html = raw_html.replace('<div class="rte wa-page">', '<div class="rte wa-page">\n<style>\n' + WA_CSS_FRAMEWORK + '\n</style>', 1)
                    if WA_OBSERVER_SCRIPT not in final_html:
                        last_div = final_html.rfind('</div>')
                        if last_div >= 0:
                            final_html = final_html[:last_div] + '\n' + WA_OBSERVER_SCRIPT + '\n' + final_html[last_div:]
                else:
                    final_html = raw_html
                meta = None  # parsed downstream from raw2 markers

            final_html = sanitize_html(final_html)

            # Shipping/trust badges: NOT injected — already on store theme

            if meta is None:
                # Legacy path didn't extract meta yet — pull from === markers
                mm = re.search(r'===META_START===\s*(.*?)\s*===META_END===', raw2, re.DOTALL)
                meta = {}
                if mm:
                    try: meta = json.loads(mm.group(1).strip())
                    except: pass
            seo_m = meta.get('seo_meta', {})

            # QA validation
            _provided_imgs = [img['url'] for img in img_data.get('desc',[]) + img_data.get('top',[]) if img.get('url')]
            _provided_colls = set(lnk['handle'] for lnk in interlinks) if interlinks else set()
            _primary_kws = [kd['keyword'] for kd in t1_primary[:3]] if t1_primary else []
            _html_ok, _html_issues = validate_html(final_html, _provided_imgs, _provided_colls, _primary_kws)
            _meta_ok, _meta_issues = validate_meta(meta)

            if _html_issues:
                for issue in _html_issues:
                    if 'image not in provided list' in issue:
                        bad_url = issue.split(': ')[1] if ': ' in issue else ''
                        if bad_url:
                            final_html = final_html.replace(bad_url, '')
                            print(f'    Removed hallucinated image: {bad_url[:50]}')
                    elif 'collection not found' in issue:
                        bad_handle = issue.split(': ')[1] if ': ' in issue else ''
                        if bad_handle:
                            final_html = re.sub(f'<a[^>]*href="/collections/{re.escape(bad_handle)}"[^>]*>.*?</a>', '', final_html)
                            print(f'    Removed bad collection: {bad_handle}')
                remaining = [i for i in _html_issues if 'image not in' not in i and 'collection not found' not in i]
                if remaining:
                    print(f'    HTML issues: {"; ".join(remaining)}')
            if _meta_issues:
                print(f'    Meta issues: {"; ".join(_meta_issues)}')

            # Schema.org
            cost_p = product.get('cost_price', 0)
            sell_price = calc_price(cost_p) if cost_p > 0 else 0
            compare_p = calc_compare_price(cost_p) if cost_p > 0 else 0
            gallery_urls = [img['url'] for img in img_data.get('top',[]) if img.get('url','').startswith('http')]
            schema_json = build_schema_jsonld(
                title=safe_title, description=meta.get('short_description', safe_desc[:200]),
                handle=seo_m.get('handle', ''), price=sell_price, compare_price=compare_p,
                images=gallery_urls[:5], category=stage1.get('category',''))
            if schema_json:
                for sb in schema_json.split('\n'):
                    if sb.strip():
                        final_html += '\n<script type="application/ld+json">' + sb + '</script>'
            faq_schema = build_faq_schema(final_html)
            if faq_schema:
                final_html += '\n<script type="application/ld+json">' + faq_schema + '</script>'
            howto_schema = build_howto_schema(final_html, safe_title)
            if howto_schema:
                final_html += '\n<script type="application/ld+json">' + howto_schema + '</script>'
            if product.get('video_urls'):
                hero_img = next((img['url'] for img in img_data.get('top',[]) if img.get('url','')), '')
                vid_schema = build_video_schema(product['video_urls'][0], safe_title, hero_img, stage1.get('product_summary',''))
                if vid_schema:
                    final_html += '\n<script type="application/ld+json">' + vid_schema + '</script>'

            print(f'    HTML: {len(final_html)} chars | ${cost2:.4f} | QA:{"PASS" if _html_ok else "WARN"}')

            # ═══ Validate tags against taxonomy ═══
            _raw_tags = meta.get('product_tags', [])
            _validated_tags, _dropped_tags = validate_tags(_raw_tags)
            if _dropped_tags:
                print(f'    ⚠ Dropped {len(_dropped_tags)} hallucinated tags: {_dropped_tags[:3]}{"..." if len(_dropped_tags)>3 else ""}')
            if len(_validated_tags) < TAGS_PER_PRODUCT_MIN:
                print(f'    ⚠ Only {len(_validated_tags)} valid tags (min {TAGS_PER_PRODUCT_MIN}) — Claude underused taxonomy')
            print(f'    Tags: {len(_validated_tags)} valid')

            cur_cost = db.execute('SELECT cost_usd FROM products WHERE id=?',(pid,)).fetchone()[0] or 0
            db_update_status(pid, 'html_ready', final_html=final_html,
                stage2_json=json.dumps(meta, ensure_ascii=False),
                seo_title=seo_m.get('title',''), seo_description=seo_m.get('description',''),
                url_handle=seo_m.get('handle',''), product_tags=json.dumps(_validated_tags),
                cost_usd=cur_cost+cost2)
            fname = re.sub(r'[^a-zA-Z0-9]','_', product['title'][:50]).strip('_').lower() or 'product'
            with open(f'{OUTPUT_DIR}/{fname}.html', 'w', encoding='utf-8') as f: f.write(final_html)
            pstatus = 'html_ready'

        # ═══ STEP 5: Shopify (v9: + variants) ═══
        if pstatus == 'html_ready':
            row = db.execute('SELECT * FROM products WHERE id=?',(pid,)).fetchone()
            product = json.loads(row['scrape_json'])
            stage1 = json.loads(row['stage1_json'] or '{}')
            full_html = row['final_html']
            meta = json.loads(row['stage2_json'] or '{}')
            seo_t = row['seo_title'] or product['title'][:60]
            seo_d = row['seo_description'] or ''
            handle = row['url_handle'] or ''
            tags = json.loads(row['product_tags'] or '[]')
            img_data = json.loads(row['image_urls_json'])
            variants = json.loads(row['variants_json'] or '[]')

            short_desc = meta.get('short_description', product.get('description','')[:200])
            body_parts = [f'<p>{short_desc}</p>']
            hero = stage1.get('conversion',{}).get('hero_claim','')
            if hero: body_parts.append(f'<p><strong>{hero}</strong></p>')
            pain_pts = stage1.get('conversion',{}).get('pain_points_solved',[])
            if pain_pts: body_parts.append('<ul>' + ''.join(f'<li>{p}</li>' for p in pain_pts[:4]) + '</ul>')
            body_html = '\n'.join(body_parts)

            print('  → Shopify...')
            existing_id = await shopify_find_product(product['title'])
            if not existing_id and handle:
                _hq = f'handle:{handle}'
                r_handle = await shopify_gql('query($q:String!){products(first:1,query:$q){nodes{id}}}', {"q": _hq})
                nodes = r_handle.get('data',{}).get('products',{}).get('nodes',[]) if r_handle else []
                if nodes: existing_id = nodes[0]['id']
            if existing_id:
                shop_pid = await shopify_update_product(existing_id, body_html, seo_t, seo_d, handle)
                print(f'    Updated: {existing_id}')
            else:
                p_type = stage1.get('category', '')
                shop_pid = await shopify_create_product(product['title'], body_html, handle, seo_t, seo_d, tags,
                    product_type=p_type, vendor=STORE_VENDOR)
                print(f'    Created: {shop_pid}')
            if shop_pid:
                if full_html:
                    ok = await shopify_set_metafield(shop_pid, 'custom', 'html_description', full_html)
                    print(f'    Metafield: {"OK" if ok else "FAIL"} ({len(full_html)} chars)')
                og_img = ''
                gallery = [img['url'] for img in img_data.get('top',[]) if img.get('url','').startswith('http')]
                if gallery:
                    og_img = gallery[0]
                    vision_alts = {}
                    for img_info in stage1.get('images', []):
                        idx = img_info.get('index')
                        desc = img_info.get('desc', '')
                        if idx and desc: vision_alts[idx] = desc
                    n = await shopify_attach_media(shop_pid, gallery, alt_map=vision_alts)
                    print(f'    Gallery: {n}/{len(gallery)}')
                if og_img:
                    await shopify_set_metafield(shop_pid, 'global', 'og_image', og_img, mtype='url')
                weight_g = product.get('weight_g', 0)
                if weight_g > 0:
                    ok = await shopify_set_weight(shop_pid, weight_g)
                    print(f'    Weight: {weight_g:.0f}g {"OK" if ok else "FAIL"}')
                inv_count = await shopify_set_inventory(shop_pid, DEFAULT_INVENTORY)
                if inv_count: print(f'    Inventory: {inv_count} variants')
                g_cat = stage1.get('category', '')
                if g_cat:
                    ok = await shopify_set_google_shopping(shop_pid, g_cat)
                    if ok: print(f'    Google Shopping: {g_cat}')
                n_merchant = await shopify_set_merchant_extended(shop_pid, stage1, variants)
                if n_merchant: print(f'    Merchant: {n_merchant} fields')
                if variants:
                    if product.get('variant_prices') and variants:
                        variants[0]['_variant_prices'] = product['variant_prices']
                    nv = await shopify_create_variants(shop_pid, variants, product.get('cost_price', 0))
                    print(f'    Variants: {nv} created')
                elif product.get('cost_price', 0) > 0:
                    cost = product['cost_price']
                    bundle_n = detect_bundle(product.get('title',''))
                    if bundle_n > 1:
                        p = round(calc_price(cost) * min(bundle_n * 0.8, 3.5), 2)
                        p = round(round(p) - 0.10, 2)
                        cp = round(calc_compare_price(cost) * bundle_n)
                        print(f'    Bundle: {bundle_n} pcs ${p:.2f}')
                    else:
                        p, cp = calc_price(cost), calc_compare_price(cost)
                    ok = await shopify_set_price(shop_pid, p, cp)
                    print(f'    Price: ${p:.2f} (was ${cp:.0f}) {"OK" if ok else "FAIL"}')
                # Link to collections (primary + any from junction table)
                coll_ids = set()
                # Method 1: From junction table
                for c in get_product_collections(pid):
                    if len(c) > 2 and c[2]: coll_ids.add(c[2])
                # Method 2: Direct from primary_collection field (more reliable)
                pc_row = db.execute('SELECT primary_collection FROM products WHERE id=?', (pid,)).fetchone()
                if pc_row and pc_row['primary_collection']:
                    pc_title = pc_row['primary_collection']
                    # Find Shopify collection ID by seo_title
                    cr = db.execute('SELECT shopify_collection_id FROM collections WHERE seo_title=?', (pc_title,)).fetchone()
                    if cr and cr['shopify_collection_id']:
                        coll_ids.add(cr['shopify_collection_id'])
                    else:
                        # Fallback: search by handle in ALL_SHOP_COLLECTIONS
                        cr2 = db.execute('SELECT seo_handle FROM collections WHERE seo_title=?', (pc_title,)).fetchone()
                        if cr2:
                            for sc in ALL_SHOP_COLLECTIONS:
                                if sc.get('handle') == cr2['seo_handle']:
                                    coll_ids.add(sc['id'])
                                    break
                if coll_ids:
                    await shopify_add_to_collections(shop_pid, list(coll_ids))
                    print(f'    Collections: {len(coll_ids)}')
                else:
                    print(f'    ⚠ No collections found for this product')
                _cost = product.get('cost_price', 0)
                _sell = calc_price(_cost) if _cost > 0 else 0
                if _sell > 0 and _cost > 0:
                    _fees = _sell * 0.029 + 0.30 + _sell * 0.02
                    _net = _sell - _cost - _fees
                    _margin_pct = (_net / _sell) * 100
                    if _margin_pct < 40:
                        print(f'    MARGIN: ${_sell:.2f} - ${_cost:.2f} - ${_fees:.2f} = ${_net:.2f} ({_margin_pct:.0f}%)')
                db_update_status(pid, 'done', shopify_product_id=shop_pid)
            else:
                db_update_status(pid, 'done')
            print(f'  DONE!')
            shutil.copy(DB_LOCAL, DB_PATH)

    except Exception as e:
        import traceback; traceback.print_exc()
        db_update_status(pid, 'error', error_msg=str(e)[:500])
        print(f'  ERROR: {e}')

print(f'\n{"="*60}')
d = db.execute("SELECT COUNT(*) FROM products WHERE status='done'").fetchone()[0]
e = db.execute("SELECT COUNT(*) FROM products WHERE status='error'").fetchone()[0]
cost = db.execute("SELECT SUM(cost_usd) FROM products").fetchone()[0] or 0
s = db.execute("SELECT COUNT(*) FROM products WHERE shopify_product_id IS NOT NULL").fetchone()[0]
print(f'  Done: {d}/{total} | Shopify: {s} | Errors: {e} | Cost: ${cost:.4f}')
print(f'{"="*60}')
shutil.copy(DB_LOCAL, DB_PATH)
if d > 0:
    ok = ping_google_sitemap(STORE_DOMAIN)
    print(f'  Google sitemap ping: {"OK" if ok else "FAIL"}')


Shopify collections for interlinking: 100

  26 total | 0 done | 26 remaining

────────────────────────────────────────────────────────────
[1/26] Transparent Crystal Grape Soap Gift Box Set, Handmade Grape 
  Status: pending
  → Scraping...
  Scrape: 10 top + 6 desc | $1.81 | variants: none | 7 specs
    Title: Transparent Crystal Grape Soap Gift Box Set, Handmade Grape 
  → Images...
    CDN: 16/16
  → Vision...
    OK $0.0158 | decorative | 5 imgs | claims:5 ingr:9
    Hero: Handmade decorative soap that's beautifu
  → Generating HTML + SEO...
    Removed bad collection: natural-skincare-essentials
    Removed bad collection: decorative-soap-sets
    Removed bad collection: natural-skincare-essentials
    Removed bad collection: luxury-bath-gift-sets
    Removed bad collection: luxury-bath-gift-sets
    Removed bad collection: decorative-soap-sets
    Removed bad collection: natural-skincare-essentials
    HTML: 27885 chars | $0.0988 | QA:WARN
  → Shopify...
    Created: gid://shopi

Traceback (most recent call last):
  File "/tmp/ipykernel_2552/2687877196.py", line 633, in <cell line: 1>
    pc_row = db.execute('SELECT primary_collection FROM products WHERE id=?', (pid,)).fetchone()
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
sqlite3.OperationalError: no such column: primary_collection


  Scrape: 5 top + 3 desc | $1.81 | variants: none | 5 specs, size chart
    Title: Style Solid Color Water-Absorbing Hair Drying Cap Thickened 
  → Images...
    CDN: 8/8
  → Vision...
    OK $0.0088 | simple | 5 imgs | claims:4 ingr:0
    Hero: Super-absorbent microfiber dries hair in
  → Generating HTML + SEO...
    Removed bad collection: hair-care-accessories
    Removed bad collection: bath-towels
    Removed bad collection: bathroom-essentials
    HTML issues: no inline body links found (all links in footer only)
    Meta issues: meta description too long: 173 chars
    HTML: 24138 chars | $0.0819 | QA:WARN
  → Shopify...
    Created: gid://shopify/Product/8790649897138
    Metafield: OK (24138 chars)
    Gallery: 5/5
    Google Shopping: Bath & Towels / Hair Care Accessories
    Merchant: 5 fields
    Price: $8.90 (was $15) OK
  ERROR: no such column: primary_collection

────────────────────────────────────────────────────────────
[3/26] Baby Silicone Shampoo Scalp Massage Body 

Traceback (most recent call last):
  File "/tmp/ipykernel_2552/2687877196.py", line 633, in <cell line: 1>
    pc_row = db.execute('SELECT primary_collection FROM products WHERE id=?', (pid,)).fetchone()
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
sqlite3.OperationalError: no such column: primary_collection


  Scrape: 6 top + 0 desc | $1.81 | variants: none | 6 specs
    Title: Baby Silicone Shampoo Scalp Massage Body Clean Scrub Feel Ba
  → Images...


## Cell 7 — Status

In [ ]:
for s in ['done','html_ready','content_ready','vision_done','images_uploaded','scraped','pending','error']:
    n = db.execute("SELECT COUNT(*) FROM products WHERE status=?", (s,)).fetchone()[0]
    if n > 0: print(f'  {s:>18}: {n}')
cost = db.execute("SELECT SUM(cost_usd) FROM products").fetchone()[0] or 0
s_ok = db.execute("SELECT COUNT(*) FROM products WHERE shopify_product_id IS NOT NULL").fetchone()[0]
print(f'  Shopify: {s_ok} | Cost: ${cost:.4f}')
# ── Smart Retry (resume from last successful step) ──
# db.execute("UPDATE products SET status='scraped' WHERE status='error' AND scrape_json IS NOT NULL AND image_urls_json IS NULL"); db.commit()
# db.execute("UPDATE products SET status='images_uploaded' WHERE status='error' AND image_urls_json IS NOT NULL AND stage1_json IS NULL"); db.commit()
# db.execute("UPDATE products SET status='vision_done' WHERE status='error' AND stage1_json IS NOT NULL AND final_html IS NULL"); db.commit()
# db.execute("UPDATE products SET status='html_ready' WHERE status='error' AND final_html IS NOT NULL AND shopify_product_id IS NULL"); db.commit()
# db.execute("UPDATE products SET status='pending' WHERE status='error' AND scrape_json IS NULL"); db.commit()  # Full re-scrape only if no data

# ── Batch Publish (after reviewing DRAFT products) ──
# Uncomment to activate all DRAFT products at once:
# import httpx, asyncio
# async def batch_publish():
#     rows = db.execute("SELECT shopify_product_id FROM products WHERE shopify_product_id IS NOT NULL").fetchall()
#     for r in rows:
#         pid = r['shopify_product_id']
#         await shopify_gql("mutation($p:ProductUpdateInput!){productUpdate(product:$p){product{id}userErrors{field message}}}",
#             {"p":{"id":pid,"status":"ACTIVE"}})
#         await asyncio.sleep(0.3)
#     print(f"Published {len(rows)} products")
# await batch_publish()
